# COR2_enrich_infocuria_document_layer_v16

This notebook enriches the InfoCuria case skeleton with procedural/case metadata, document metadata, relevance flags, compact document context, and a streamlined download manifest.

v16 changes (cumulative from v7):

- Writes outputs under `output/court_infocuria/excel`, `output/court_infocuria/jsonl`, and `output/court_infocuria/parquet`.
- Defaults to `SCRAPE_MODE = "rebuild_from_cache"` so rerunning normally reconstructs outputs from saved raw metadata instead of scraping again.
- Uses `affId` as the case-level document/relevance/download rollup key because one case can have multiple `procedureId` values.
- Does not create a temporary enrich folder; transient stream JSONL files are written under the JSONL output folder and then deleted.
- Removes oversized download-manifest columns `candidate_download_records` and `all_candidate_download_urls`.
- Keeps English metadata only in final manifests.
- Adds document-level download status to the document metadata manifest.
- Adds `manual_joined_case_search` to the case metadata manifest by rendering the InfoCuria affair page for cases flagged with `joinExist == 1`.
- v16 patch: replaces fragile JavaScript `page.evaluate(...join('\n')...)` with Playwright `locator('body').inner_text(...)` to avoid SyntaxError from newline escaping.
- Adds case-level download/relevance rollups to the case metadata manifest, including whether all relevant documents were downloaded/present.
- Normalizes generated document/case `internal_key` values to `YYYY/NNNN/PREFIX/SUFFIX/`, preserving an empty suffix as `//`.


v9 patch:

- Replaces the fragile rendered-page/manual joined-case lookup with a deterministic InfoCuria `affairId/procedures` lookup.
- For `joinExist == 1` cases, it now extracts joined cases only from `content.idPilot` and `content.joinAffairs`, taking the stable case number after `#`, e.g. `...#C-7/56`.
- Avoids broad full-text scans, doctrine-note false positives, duplicate `<body>` errors, and truncated UI previews such as `..., C-7/57`.


v12 patch:

- Keeps the v9 InfoCuria procedures-endpoint joined-case lookup approach.
- Runs joined-case lookup for every case with a usable case identifier and affId, not only rows where `joinExist == 1`.
- Adds `manual_joined_case_collection`: a propagated `|`-separated collection of the full joined-case group for every member row, including the row's own case number.
- The collection is built by taking the row case plus `manual_joined_case_search`, adding all pairwise links, and resolving connected components so asymmetric pilot/join data is carried back to all related rows.


v16 joined-case cache patch:

- Adds a resumable `infocuria_joined_case_api_cache.jsonl` for the joined-case repair step.
- Adds `JOINED_CASE_JSONL_MODE` with `rebuild_jsonl`, `update_failed`, and `csv_from_jsonl`.
- Records 403/429/other failures in JSONL instead of printing one warning per row in tqdm.
- Adds periodic cooldown after every configured number of joined-case requests.
- Keeps the rest of the enrichment/download/document-manifest notebook intact; this is an additive cache layer around the joined-case repair.


## v16 keying change

`procedureId` is now the canonical unique key wherever the data supports it.

The notebook now uses `procedureId` first for:

- flattened case-metadata deduplication;
- procedural metadata joins;
- case-level document aggregation;
- document/download joins where available;
- case-level download rollups.

`affId` remains only as a fallback and for genuinely affair-based API operations, such as joined-affair lookups.


## v16 procedure-level request architecture

The upstream InfoCuria case manifest already contains `procedureId`.

This version therefore:

- builds one fetch row per distinct `procedureId`;
- uses `procedureId` as the JSONL cache key;
- writes to a new procedure-level raw cache:
  `infocuria_procedural_metadata_by_procedure_raw.jsonl`;
- still routes requests through the required `affId` endpoint;
- passes the procedure-specific `publishedId` in `searchTerm`;
- selects the exact response hit matching the requested `procedureId`;
- extracts documents only from that matched procedure hit;
- refuses to silently fall back to the first unrelated hit;
- uses `procedureId` first in document deduplication, joins, aggregation, and rollups.

Because the old JSONL contains affair-oriented cache records, v16 defaults to:

```python
SCRAPE_MODE = "fresh_scrape"
```

A full new procedure-level JSONL must be built once. Later runs can use
`update_scrape` or `rebuild_from_cache`.


In [9]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------
import json
import os
import re
import time
import asyncio
import logging
import random
from pathlib import Path
from datetime import datetime
from collections import Counter
from typing import Any
from urllib.parse import urlparse, quote

import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("enrich_infocuria_document_layer")


In [10]:

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

# Set this manually if automatic discovery chooses the wrong folder.
PROJECT_ROOT_OVERRIDE = None

# Input manifests.
# The upstream InfoCuria case/web manifest is the master spine. It should already
# contain `internal_key` in the format YYYY/NNNN/PREFIX/SUFFIX/.
INFOCURIA_CASE_MANIFEST_OVERRIDE = None
CURIA_LEGACY_MANIFEST_OVERRIDE = None

# Procedural metadata scraper/cache mode.
#   fresh_scrape        = ignore/delete old raw JSONL and scrape from zero
#   update_scrape       = load old raw JSONL, scrape missing/current rows, compact keeping latest per key
#   rebuild_from_cache  = do not scrape; rebuild Excel/JSONL/Parquet outputs from saved raw JSONL
SCRAPE_MODE = "update_scrape"
VALID_SCRAPE_MODES = {"fresh_scrape", "update_scrape", "rebuild_from_cache"}
if SCRAPE_MODE not in VALID_SCRAPE_MODES:
    raise ValueError(f"SCRAPE_MODE must be one of {sorted(VALID_SCRAPE_MODES)}, got {SCRAPE_MODE!r}")

# v16 uses one cache record per procedureId. The old affair-level cache
# is deliberately not reused because it cannot guarantee one request/response
# record for every procedure row in the upstream manifest.
# Raw InfoCuria metadata cache.
RAW_METADATA_JSONL_OVERRIDE = None
RAW_METADATA_JSONL_FILENAME = "infocuria_procedural_metadata_by_procedure_raw.jsonl"

# Fetch controls.
# LIMIT_METADATA_FETCH_CASES is useful for testing. None means all cases.
LIMIT_METADATA_FETCH_CASES = None
UPDATE_EXISTING_CACHE_RECORDS = False  # if True in update_scrape, refetches even records already in the cache

# InfoCuria metadata endpoint.
PROCEDURES_URL = "https://infocuriaws.curia.europa.eu/elastic-connector/affairId/procedures"
LANGUAGE = "EN"
METADATA_REQUEST_TIMEOUT_SECONDS = 60
METADATA_MAX_RETRIES = 5
METADATA_SLEEP_SECONDS = 0.25

# Relevance rule requested by Eddy.
# v7: ORD_COMM is explicitly relevant and therefore targeted for download.
# Codes may also appear in combinations such as "ORD_COMM|REQ_COMM";
# relevance is evaluated token-wise downstream.
RELEVANT_DOC_TYPE_CODES = {
    "ARRET",
    "ARRET_NP",
    "ARRET_DR",
    "ARRET_SOM",
    "ORD",
    "ORD_NP",
    "ORD_DR",
    "ORD_SOM",
    "ORD_COMM",
    "RECT",
    "DECISION",
}

RELEVANCE_REASON = {
    "ARRET": "relevant_final_judgment",
    "ARRET_NP": "relevant_final_judgment_not_published",
    "ARRET_DR": "relevant_judgment_dr_variant",
    "ARRET_SOM": "relevant_summary_judgment_variant",
    "ORD": "relevant_final_order",
    "ORD_NP": "relevant_final_order_not_published",
    "ORD_DR": "relevant_order_dr_variant",
    "ORD_SOM": "relevant_summary_order_variant",
    "ORD_COMM": "relevant_order_communication_variant",
    "RECT": "relevant_rectification_corrigendum",
    "DECISION": "relevant_decision",
}

# Download choices.
# Safe default: build manifests only. Set True only when you want to download files.
RUN_DOWNLOADS = True
OVERWRITE_EXISTING_FILES = False
DOWNLOAD_LIMIT = None
DOWNLOAD_SAVE_EVERY = 100

# Language and format priority.
PREFERRED_LANGUAGES = ["EN", "DE", "FR"]
PREFERRED_FORMATS = ["HTML", "PDF"]

REQUEST_TIMEOUT_SECONDS = 90
DOWNLOAD_SLEEP_SECONDS = 1.25
DOWNLOAD_SLEEP_JITTER_SECONDS = 0.75
RETRIES_PER_URL = 3
RETRY_BACKOFF_SECONDS = 4.0
SESSION_WARMUP = True
SESSION_WARMUP_SLEEP_SECONDS = 0.75
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml,application/pdf,application/json;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9,de;q=0.8,fr;q=0.7",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Origin": "https://infocuria.curia.europa.eu",
    "Referer": "https://infocuria.curia.europa.eu/",
}

# Streaming settings.
STREAM_JSONL_CHUNK_SIZE = 5000
KEEP_RAW_CONTENT_JSON = False

# Output stems.
CASE_METADATA_STEM = "infocuria_case_metadata_manifest"
DOCUMENT_METADATA_STEM = "infocuria_document_metadata_manifest"
DOCUMENT_CONTEXT_STEM = "infocuria_document_context_manifest"
DOWNLOAD_MANIFEST_STEM = "infocuria_document_download_manifest"
LEGACY_JOIN_FAILURES_STEM = "infocuria_legacy_join_failures"
METADATA_FETCH_LOG_STEM = "infocuria_metadata_fetch_log"
METADATA_FETCH_FAILURES_STEM = "infocuria_metadata_fetch_failures"
FLATTEN_FAILED_STEM = "infocuria_metadata_flatten_failures"

# Final metadata should be English-only.
DROP_NON_ENGLISH_METADATA_COLUMNS = [
    "case_name_de", "case_name_fr",
    "case_object_de", "case_object_fr",
    "subject_matter_de", "subject_matter_fr",
    "procedure_result_de", "procedure_result_fr",
]

# Oversized download columns are deliberately not exported in v5.
DROP_HUGE_DOWNLOAD_COLUMNS = [
    "candidate_download_records",
    "all_candidate_download_urls",
]


# Joined-case lookup repair.
# v9 uses the InfoCuria affairId/procedures endpoint and extracts ONLY the stable
# joined-case references in content.idPilot and content.joinAffairs. This avoids
# the old rendered-page scraper, which was fragile because InfoCuria may create
# duplicate <body> elements, empty Angular shells, and truncated UI previews.
RUN_MANUAL_JOINED_CASE_SEARCH = True
MANUAL_JOINED_CASE_SEARCH_LIMIT = None  # useful for testing; None means all cases with affId + case identifier
MANUAL_JOINED_CASE_SLEEP_SECONDS = 0.15
MANUAL_JOINED_CASE_RENDER_WAIT_MS = 3000  # retained for compatibility; not used by the v9 API lookup
MANUAL_JOINED_CASE_BROWSER_TIMEOUT_MS = 60000  # retained for compatibility; not used by the v9 API lookup

# v14 joined-case API cache controls.
#   rebuild_jsonl   = overwrite the joined-case JSONL and request every eligible row again
#   update_failed   = request missing rows plus rows whose latest JSONL status is failed/403/429/etc.
#   csv_from_jsonl  = do not call InfoCuria; rebuild joined-case columns from the current JSONL/cache/local fields
JOINED_CASE_JSONL_MODE = "update_failed"
VALID_JOINED_CASE_JSONL_MODES = {"rebuild_jsonl", "update_failed", "csv_from_jsonl"}
if JOINED_CASE_JSONL_MODE not in VALID_JOINED_CASE_JSONL_MODES:
    raise ValueError(f"JOINED_CASE_JSONL_MODE must be one of {sorted(VALID_JOINED_CASE_JSONL_MODES)}, got {JOINED_CASE_JSONL_MODE!r}")

JOINED_CASE_JSONL_FILENAME = "infocuria_joined_case_api_cache.jsonl"
JOINED_CASE_RETRY_HTTP_STATUSES = {403, 429, 500, 502, 503, 504}
JOINED_CASE_BATCH_PAUSE_EVERY = 1000
JOINED_CASE_BATCH_PAUSE_SECONDS = 60
JOINED_CASE_COMPACT_JSONL_AFTER_RUN = True
JOINED_CASE_RECORD_RESPONSE_SUMMARY = True  # stores a light response summary, not the whole API payload

RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print("Configured at:", RUN_TIMESTAMP)
print("SCRAPE_MODE:", SCRAPE_MODE)
print("Default v7 behavior: rebuild from cache unless you explicitly choose fresh_scrape/update_scrape.")


Configured at: 2026-07-16 18:13:08
SCRAPE_MODE: update_scrape
Default v7 behavior: rebuild from cache unless you explicitly choose fresh_scrape/update_scrape.


## v2 patch: expanded relevance document types

This version adds the following InfoCuria document-type codes to the relevant set: `ARRET_DR`, `ARRET_SOM`, `ORD_SOM`, `ORD_DR`, and `DECISION`.

The practical implication is that relevance is still assigned at document level first, but cases containing any of these additional document types now become relevant at case level as well. The document metadata manifest stores the resulting `relevance_flag`, `relevance_category`, and `relevance_reason`, while the case metadata manifest aggregates the relevant document counts and relevant document-type list back onto the full case skeleton.

In [11]:

# ---------------------------------------------------------------------
# Project-root, input-path, and output-path discovery
# ---------------------------------------------------------------------

def find_project_root(start: Path | None = None) -> Path:
    """Find a likely project root by walking upward from cwd."""
    if PROJECT_ROOT_OVERRIDE:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    start = (start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "output").exists() or (p / "code").exists() or (p / "data").exists():
            return p
    return start

def first_existing(candidates, required=True, label="path"):
    for p in candidates:
        if p is None:
            continue
        p = Path(p).expanduser()
        if p.exists():
            return p.resolve()
    if required:
        raise FileNotFoundError(
            f"Could not find required {label}. Checked:\n" + "\n".join(str(x) for x in candidates if x is not None)
        )
    return None

PROJECT_ROOT = find_project_root()

# v5 output layout: everything goes under output/court_infocuria.
OUTPUT_DIR = PROJECT_ROOT / "output" / "court_infocuria"
EXCEL_DIR = OUTPUT_DIR / "excel"
JSONL_DIR = OUTPUT_DIR / "jsonl"
PARQUET_DIR = OUTPUT_DIR / "parquet"

for p in [OUTPUT_DIR, EXCEL_DIR, JSONL_DIR, PARQUET_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Downloads stay under data/raw/court_infocuria, matching the existing downloaded files.
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
COURT_INFOCURIA_RAW_DIR = RAW_DATA_DIR / "court_infocuria"
COURT_INFOCURIA_HTML_DIR = COURT_INFOCURIA_RAW_DIR / "html"
COURT_INFOCURIA_PDFS_DIR = COURT_INFOCURIA_RAW_DIR / "pdfs"
COURT_INFOCURIA_HTML_EXISTING_DIRS = [COURT_INFOCURIA_HTML_DIR, COURT_INFOCURIA_RAW_DIR / "htmls"]
COURT_INFOCURIA_PDF_EXISTING_DIRS = [COURT_INFOCURIA_PDFS_DIR, COURT_INFOCURIA_RAW_DIR / "pdf"]

for p in [COURT_INFOCURIA_RAW_DIR, COURT_INFOCURIA_HTML_DIR, COURT_INFOCURIA_PDFS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Output root:", OUTPUT_DIR)
print("Excel outputs:", EXCEL_DIR)
print("JSONL outputs:", JSONL_DIR)
print("Parquet outputs:", PARQUET_DIR)
print("Raw download root:", COURT_INFOCURIA_RAW_DIR)

INFOCURIA_CASE_MANIFEST_CANDIDATES = [
    INFOCURIA_CASE_MANIFEST_OVERRIDE,
    # Current case skeleton locations under output/court_infocuria.
    PROJECT_ROOT / "output" / "court_infocuria" / "excel" / "infocuria_case_scrape_manifest.xlsx",
    PROJECT_ROOT / "output" / "court_infocuria" / "parquet" / "infocuria_case_scrape_manifest.parquet",
    PROJECT_ROOT / "output" / "court_infocuria" / "jsonl" / "infocuria_case_scrape_manifest.jsonl",
    PROJECT_ROOT / "output" / "court_infocuria" / "infocuria_case_scrape_manifest.csv",
    # Previous temporary infocuria locations.
    PROJECT_ROOT / "output" / "infocuria" / "excel" / "infocuria_case_scrape_manifest.xlsx",
    PROJECT_ROOT / "output" / "infocuria" / "parquet" / "infocuria_case_scrape_manifest.parquet",
    PROJECT_ROOT / "output" / "infocuria" / "jsonl" / "infocuria_case_scrape_manifest.jsonl",
    PROJECT_ROOT / "output" / "infocuria" / "infocuria_case_scrape_manifest.csv",
    PROJECT_ROOT / "output" / "court_infocuria" / "infocuria_case_scrape_manifest.csv",
    PROJECT_ROOT / "output" / "court_infocuria" / "web_scrape_manifest_infocuria.csv",
    PROJECT_ROOT / "infocuria_case_scrape_manifest.csv",
    Path.cwd() / "infocuria_case_scrape_manifest.csv",
]

CURIA_LEGACY_MANIFEST_CANDIDATES = [
    CURIA_LEGACY_MANIFEST_OVERRIDE,
    PROJECT_ROOT / "output" / "court_curia" / "web_scrape_manifest.csv",
    PROJECT_ROOT / "output" / "court_infocuria" / "web_scrape_manifest.csv",
    PROJECT_ROOT / "output" / "web_scrape_manifest.csv",
    PROJECT_ROOT / "web_scrape_manifest.csv",
    Path.cwd() / "web_scrape_manifest.csv",
    Path("/mnt/data/web_scrape_manifest (6).csv"),
]

RAW_METADATA_JSONL = (
    Path(RAW_METADATA_JSONL_OVERRIDE).expanduser().resolve()
    if RAW_METADATA_JSONL_OVERRIDE
    else (JSONL_DIR / RAW_METADATA_JSONL_FILENAME)
)

JOINED_CASE_JSONL = JSONL_DIR / JOINED_CASE_JSONL_FILENAME

INFOCURIA_CASE_MANIFEST = first_existing(INFOCURIA_CASE_MANIFEST_CANDIDATES, required=True, label="InfoCuria case scrape manifest")
CURIA_LEGACY_MANIFEST = first_existing(CURIA_LEGACY_MANIFEST_CANDIDATES, required=False, label="legacy CURIA manifest")

# Final output paths.
CASE_METADATA_XLSX = EXCEL_DIR / f"{CASE_METADATA_STEM}.xlsx"
DOCUMENT_METADATA_XLSX = EXCEL_DIR / f"{DOCUMENT_METADATA_STEM}.xlsx"
DOCUMENT_CONTEXT_XLSX = EXCEL_DIR / f"{DOCUMENT_CONTEXT_STEM}.xlsx"
DOWNLOAD_MANIFEST_XLSX = EXCEL_DIR / f"{DOWNLOAD_MANIFEST_STEM}.xlsx"
LEGACY_JOIN_FAILURES_XLSX = EXCEL_DIR / f"{LEGACY_JOIN_FAILURES_STEM}.xlsx"
METADATA_FETCH_LOG_XLSX = EXCEL_DIR / f"{METADATA_FETCH_LOG_STEM}.xlsx"
METADATA_FETCH_FAILURES_XLSX = EXCEL_DIR / f"{METADATA_FETCH_FAILURES_STEM}.xlsx"
FLATTEN_FAILED_XLSX = EXCEL_DIR / f"{FLATTEN_FAILED_STEM}.xlsx"

CASE_METADATA_JSONL = JSONL_DIR / f"{CASE_METADATA_STEM}.jsonl"
DOCUMENT_METADATA_JSONL = JSONL_DIR / f"{DOCUMENT_METADATA_STEM}.jsonl"
DOCUMENT_CONTEXT_JSONL = JSONL_DIR / f"{DOCUMENT_CONTEXT_STEM}.jsonl"
DOWNLOAD_MANIFEST_JSONL = JSONL_DIR / f"{DOWNLOAD_MANIFEST_STEM}.jsonl"
LEGACY_JOIN_FAILURES_JSONL = JSONL_DIR / f"{LEGACY_JOIN_FAILURES_STEM}.jsonl"
METADATA_FETCH_LOG_JSONL = JSONL_DIR / f"{METADATA_FETCH_LOG_STEM}.jsonl"
METADATA_FETCH_FAILURES_JSONL = JSONL_DIR / f"{METADATA_FETCH_FAILURES_STEM}.jsonl"
FLATTEN_FAILED_JSONL = JSONL_DIR / f"{FLATTEN_FAILED_STEM}.jsonl"

CASE_METADATA_PARQUET = PARQUET_DIR / f"{CASE_METADATA_STEM}.parquet"
DOCUMENT_METADATA_PARQUET = PARQUET_DIR / f"{DOCUMENT_METADATA_STEM}.parquet"
DOCUMENT_CONTEXT_PARQUET = PARQUET_DIR / f"{DOCUMENT_CONTEXT_STEM}.parquet"
DOWNLOAD_MANIFEST_PARQUET = PARQUET_DIR / f"{DOWNLOAD_MANIFEST_STEM}.parquet"
LEGACY_JOIN_FAILURES_PARQUET = PARQUET_DIR / f"{LEGACY_JOIN_FAILURES_STEM}.parquet"
METADATA_FETCH_LOG_PARQUET = PARQUET_DIR / f"{METADATA_FETCH_LOG_STEM}.parquet"
METADATA_FETCH_FAILURES_PARQUET = PARQUET_DIR / f"{METADATA_FETCH_FAILURES_STEM}.parquet"
FLATTEN_FAILED_PARQUET = PARQUET_DIR / f"{FLATTEN_FAILED_STEM}.parquet"

# Backward-compatible names used by some older cells/functions.
DOWNLOAD_MANIFEST_CSV = DOWNLOAD_MANIFEST_JSONL
METADATA_FETCH_LOG_CSV = METADATA_FETCH_LOG_JSONL
METADATA_FETCH_FAILURES_CSV = METADATA_FETCH_FAILURES_JSONL
FLATTEN_FAILED_CSV = FLATTEN_FAILED_JSONL

print("INFOCURIA_CASE_MANIFEST:", INFOCURIA_CASE_MANIFEST)
print("CURIA_LEGACY_MANIFEST:", CURIA_LEGACY_MANIFEST)
print("RAW_METADATA_JSONL:", RAW_METADATA_JSONL)

# Readers/writers.
def read_manifest(path: Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, dtype=str)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".jsonl":
        return pd.read_json(path, orient="records", lines=True, dtype=False)
    return pd.read_csv(path, dtype=str, low_memory=False)

def save_jsonl(df: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(path, orient="records", lines=True, force_ascii=False)
    print(f"Saved JSONL {len(df):,} rows -> {path}")

def save_parquet(df: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
        print(f"Saved parquet {len(df):,} rows -> {path}")
    except Exception as e:
        print(f"Skipped parquet {path.name}: {e}")

def save_excel(df: pd.DataFrame, path: Path, sheet_name="manifest"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_excel(path, index=False, sheet_name=sheet_name[:31])
        print(f"Saved Excel {len(df):,} rows -> {path}")
    except Exception as e:
        print(f"Skipped Excel {path.name}: {e}")

def save_outputs(df: pd.DataFrame, stem: str, excel_path: Path, jsonl_path: Path, parquet_path: Path):
    save_jsonl(df, jsonl_path)
    save_parquet(df, parquet_path)
    save_excel(df, excel_path, sheet_name=stem)


PROJECT_ROOT: /home/edik/projects/eccjeu
Output root: /home/edik/projects/eccjeu/output/court_infocuria
Excel outputs: /home/edik/projects/eccjeu/output/court_infocuria/excel
JSONL outputs: /home/edik/projects/eccjeu/output/court_infocuria/jsonl
Parquet outputs: /home/edik/projects/eccjeu/output/court_infocuria/parquet
Raw download root: /home/edik/projects/eccjeu/data/raw/court_infocuria
INFOCURIA_CASE_MANIFEST: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_case_scrape_manifest.xlsx
CURIA_LEGACY_MANIFEST: /home/edik/projects/eccjeu/output/court_curia/web_scrape_manifest.csv
RAW_METADATA_JSONL: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_procedural_metadata_by_procedure_raw.jsonl


In [12]:

# ---------------------------------------------------------------------
# Input validation
# ---------------------------------------------------------------------

def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 * 1024)

def count_jsonl_lines(path: Path, max_lines=None) -> int:
    n = 0
    if not path.exists():
        return 0
    with open(path, "rb") as f:
        for n, _ in enumerate(f, start=1):
            if max_lines is not None and n >= max_lines:
                break
    return n

print("InfoCuria case manifest exists:", INFOCURIA_CASE_MANIFEST.exists(), "| size MB:", round(file_size_mb(INFOCURIA_CASE_MANIFEST), 2))

if RAW_METADATA_JSONL.exists():
    print("Raw procedural metadata JSONL exists:", RAW_METADATA_JSONL)
    print("Raw JSONL size MB:", round(file_size_mb(RAW_METADATA_JSONL), 2))
    print("Raw JSONL line count:", f"{count_jsonl_lines(RAW_METADATA_JSONL):,}")
elif SCRAPE_MODE == "rebuild_from_cache":
    raise FileNotFoundError(
        f"No raw InfoCuria procedural metadata JSONL cache found at {RAW_METADATA_JSONL}. "
        "Use SCRAPE_MODE='fresh_scrape' first, or 'update_scrape' if you want to build/update the cache."
    )
else:
    print("Raw JSONL does not exist yet and will be created:", RAW_METADATA_JSONL)

if CURIA_LEGACY_MANIFEST is not None:
    print("Legacy CURIA manifest exists:", CURIA_LEGACY_MANIFEST.exists(), "| size MB:", round(file_size_mb(CURIA_LEGACY_MANIFEST), 2))
else:
    print("No legacy CURIA manifest found. Legacy enrichment and join-failure diagnostics will be skipped.")


InfoCuria case manifest exists: True | size MB: 5.37
Raw procedural metadata JSONL exists: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_procedural_metadata_by_procedure_raw.jsonl
Raw JSONL size MB: 33363.09
Raw JSONL line count: 36,383
Legacy CURIA manifest exists: True | size MB: 48.66


In [13]:

# ---------------------------------------------------------------------
# Load optional input manifests
# ---------------------------------------------------------------------

df_cases_in = read_manifest(INFOCURIA_CASE_MANIFEST) if INFOCURIA_CASE_MANIFEST is not None else pd.DataFrame()
df_curia = read_manifest(CURIA_LEGACY_MANIFEST) if CURIA_LEGACY_MANIFEST is not None else pd.DataFrame()

print("InfoCuria input case manifest:", df_cases_in.shape)
if not df_cases_in.empty:
    display(df_cases_in.head(3))
    print(df_cases_in.columns.tolist())

print("Legacy CURIA manifest:", df_curia.shape)
if not df_curia.empty:
    display(df_curia.head(3))
    print(df_curia.columns.tolist())


InfoCuria input case manifest: (56828, 17)


,case_number_raw,case_number_clean,case_prefix,case_suffix,case_number_int,case_number_padded,case_year_raw,case_year_full,internal_key,affId,procedureId,case_name_en,introductionDate,closeDate,affairStateCode,joinExist,joinAffairs
0,C-1/00,C-1/00,C,NaN,1,0001,00,2000,2000/0001/C//,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,Commission v France,2000-01-04,2001-12-13,CLOTPUB,0,[]
1,C-1/00 SA,C-1/00 SA,C,SA,1,0001,00,2000,2000/0001/C/SA/,C/0001/00/00000000SA/01,C/0001/00/00000000SA/01/P/01,Cotecna Inspection v Commission,2000-12-14,2001-05-29,CLOTPUB,0,[]
2,C-1/01 P,C-1/01 P,C,P,1,0001,01,2001,2001/0001/C/P/,C/0001/01/00000000PV/01,C/0001/01/00000000PV/01/P/01,Asia Motor France and Others v Commission,2001-01-03,2001-09-20,CLOTPUB,0,[]


['case_number_raw', 'case_number_clean', 'case_prefix', 'case_suffix', 'case_number_int', 'case_number_padded', 'case_year_raw', 'case_year_full', 'internal_key', 'affId', 'procedureId', 'case_name_en', 'introductionDate', 'closeDate', 'affairStateCode', 'joinExist', 'joinAffairs']
Legacy CURIA manifest: (50503, 34)


,source,source_url,source_row_index,case_number_raw,case_number_clean,case_prefix,case_first_number,case_year,case_year_full,case_suffix,case_prefix_original,case_prefix_inferred_from_source,case_number_padded,case_year_full_padded,procedure_key,case_number_link,case_description,case_description_link,date,date_match_text,document_type,ecli,celex_from_case_number_link,celex_from_description_link,see_case_number_raw,see_case_number_clean,see_procedure_key,status,case_number_links_all_json,case_description_links_all_json,raw_left_cell_html,raw_right_cell_html,downloadable_from_curia_manifest,celex_download_manifest_row
0,c1_juris,https://curia.europa.eu/en/content/juris/c1_juris.htm,1,1/53,1/53,C,1,53,1953,NaN,NaN,True,0001,1953,1953/0001/C/,NaN,"Removed from the register on 7 May 1954, Verband Deutscher Reeder / ECSC High Authority (1/53) ECLI:EU:C:1954:1",NaN,07.05.1954,on 7 May 1954,removed_from_register,ECLI:EU:C:1954:1,NaN,NaN,NaN,NaN,NaN,removed,[],[],"<td ""top""="""" valign=""""><a name=""1/53""></a><b>1/53</b></td>","<td ""top""="""" valign=""""><i> Removed from the register on 7 May 1954, Verband Deutscher Reeder / ECSC High Authority (1/53) ECLI:EU:C:1954:1</i></td>",False,False
1,c1_juris,https://curia.europa.eu/en/content/juris/c1_juris.htm,2,2/53,2/53,C,2,53,1953,NaN,NaN,True,0002,1953,1953/0002/C/,NaN,"Removed from the register on 7 May 1954, Bunkerfirmen-Vereinigung / ECSC High Authority (2/53) ECLI:EU:C:1954:2",NaN,07.05.1954,on 7 May 1954,removed_from_register,ECLI:EU:C:1954:2,NaN,NaN,NaN,NaN,NaN,removed,[],[],"<td ""top""="""" valign=""""><a name=""2/53""></a><b>2/53</b></td>","<td ""top""="""" valign=""""><i> Removed from the register on 7 May 1954, Bunkerfirmen-Vereinigung / ECSC High Authority (2/53) ECLI:EU:C:1954:2</i></td>",False,False
2,c1_juris,https://curia.europa.eu/en/content/juris/c1_juris.htm,3,3/53,3/53,C,3,53,1953,NaN,NaN,True,0003,1953,1953/0003/C/,NaN,"Removed from the register on 5 November 1953, France / ECSC High Authority (3/53) ECLI:EU:C:1953:2",NaN,05.11.1953,on 5 November 1953,removed_from_register,ECLI:EU:C:1953:2,NaN,NaN,NaN,NaN,NaN,removed,[],[],"<td ""top""="""" valign=""""><a name=""3/53""></a><b>3/53</b></td>","<td ""top""="""" valign=""""><i> Removed from the register on 5 November 1953, France / ECSC High Authority (3/53) ECLI:EU:C:1953:2</i></td>",False,False


['source', 'source_url', 'source_row_index', 'case_number_raw', 'case_number_clean', 'case_prefix', 'case_first_number', 'case_year', 'case_year_full', 'case_suffix', 'case_prefix_original', 'case_prefix_inferred_from_source', 'case_number_padded', 'case_year_full_padded', 'procedure_key', 'case_number_link', 'case_description', 'case_description_link', 'date', 'date_match_text', 'document_type', 'ecli', 'celex_from_case_number_link', 'celex_from_description_link', 'see_case_number_raw', 'see_case_number_clean', 'see_procedure_key', 'status', 'case_number_links_all_json', 'case_description_links_all_json', 'raw_left_cell_html', 'raw_right_cell_html', 'downloadable_from_curia_manifest', 'celex_download_manifest_row']


In [14]:
# ---------------------------------------------------------------------
# Generic helpers
# ---------------------------------------------------------------------

def clean_str(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()



def first_nonempty(*values):
    """Return the first value that is not None/NA/empty after clean_str().

    This avoids pandas pd.NA boolean errors from expressions like
    `row.get("a") or row.get("b")`.
    """
    for value in values:
        s = clean_str(value)
        if s and s.lower() not in {"nan", "none", "<na>"}:
            return s
    return ""

def path_exists_safe(*values):
    """NA-safe check for the first non-empty path-like value."""
    s = first_nonempty(*values)
    if not s:
        return False
    try:
        return Path(s).exists()
    except Exception:
        return False

def normalize_whitespace(x):
    return re.sub(r"\s+", " ", clean_str(x)).strip()

def as_json_string(x):
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return "[]"

def unique_join(values, sep="|", max_items=None):
    vals = []
    for v in values:
        s = clean_str(v)
        if not s or s.lower() in {"nan", "none", "<na>"}:
            continue
        vals.append(s)
    vals = sorted(set(vals))
    if max_items is not None:
        vals = vals[:max_items]
    return sep.join(vals)

def ml_value(obj, lang="en", fallback=True):
    """Extract multilingual value from InfoCuria ML structures."""
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except Exception:
            return obj if fallback else ""
    lang = str(lang).lower()
    if isinstance(obj, list):
        found = []
        for item in obj:
            if isinstance(item, dict):
                for k, v in item.items():
                    if str(k).lower() == lang and clean_str(v):
                        return clean_str(v)
                    if clean_str(v):
                        found.append(clean_str(v))
        return found[0] if fallback and found else ""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).lower() == lang and clean_str(v):
                return clean_str(v)
        vals = [clean_str(v) for v in obj.values() if clean_str(v)]
        return vals[0] if fallback and vals else ""
    return ""

def label_text(obj, lang="en"):
    """Extract label text from ML label structures."""
    if isinstance(obj, str):
        try:
            obj = json.loads(obj)
        except Exception:
            return obj
    if isinstance(obj, list):
        vals = []
        for item in obj:
            if isinstance(item, dict):
                label = item.get("label", item)
                val = ml_value(label, lang, fallback=True)
                if val:
                    vals.append(val)
        return "|".join(sorted(set(vals)))
    return ml_value(obj, lang, fallback=True)

def parse_affair_ref_list(values):
    """Pull published case IDs from InfoCuria affair reference fields."""
    if values is None:
        return []
    if isinstance(values, str):
        try:
            values = json.loads(values)
        except Exception:
            values = [values]
    if not isinstance(values, list):
        values = [values]
    out = []
    for v in values:
        if isinstance(v, dict):
            for key in ["publishedId", "publishedAffId", "affaire", "label", "value", "id"]:
                if clean_str(v.get(key)):
                    out.append(clean_str(v.get(key)))
                    break
        else:
            s = clean_str(v)
            # Often fields are like "C-123/20".
            m = re.search(r"\b[CT]\s*-?\s*\d{1,5}\s*/\s*\d{2,4}(?:\s+[A-Z][A-Z0-9-]*)?\b", s, flags=re.I)
            if m:
                out.append(normalize_whitespace(m.group(0)))
    return sorted(set(out))

def normalize_doc_type(value):
    return normalize_whitespace(value).upper().replace(" ", "_").replace("-", "_")

def split_doc_type_codes(value):
    """Return normalized document-type tokens.

    InfoCuria sometimes returns combined codes such as ORD_COMM|REQ_COMM.
    v7 evaluates relevance token-wise, so ORD_COMM is still recognised as
    relevant inside a combined code.
    """
    s = normalize_doc_type(value)
    if not s:
        return []
    return [p for p in re.split(r"[|,;/]+", s) if p]

def relevant_doc_type_tokens(code):
    return [t for t in split_doc_type_codes(code) if t in RELEVANT_DOC_TYPE_CODES]

def is_relevant_doc_type(code):
    return bool(relevant_doc_type_tokens(code))

def relevance_reason(code):
    hits = relevant_doc_type_tokens(code)
    if hits:
        return "|".join(RELEVANCE_REASON.get(h, f"relevant_{h.lower()}") for h in hits)
    return "irrelevant_non_final_or_duplicate_or_administrative_document"


In [15]:

# ---------------------------------------------------------------------
# Case-key helpers
# ---------------------------------------------------------------------

# v4 policy:
# - The upstream web/case manifest already contains `internal_key`.
# - Use that as the procedural key whenever present.
# - Only parse/generate keys as a fallback for older manifests or legacy CURIA rows.

def normalize_case_suffix(suffix):
    suffix = normalize_whitespace(suffix).upper()
    suffix = suffix.replace("–", "-").replace("—", "-")
    suffix = re.sub(r"[^A-Z0-9 -]+", "", suffix)
    suffix = re.sub(r"\s+", "-", suffix).strip("-")
    return suffix

def normalize_internal_key(value):
    """Normalize procedural key to YYYY/NNNN/PREFIX/SUFFIX/.

    Empty suffix is preserved as an empty fourth segment, e.g. 1966/0001/C//.
    This matches the upstream case skeleton internal_key format and prevents
    case/document join mismatches caused by 1966/0001/C/ vs 1966/0001/C//.
    """
    s = clean_str(value).upper().replace("\\", "/")
    if not s:
        return ""
    parts = s.split("/")
    # Preserve an explicitly empty suffix if present; remove only surrounding empty parts.
    if len(parts) >= 3 and parts[0] == "":
        parts = parts[1:]
    if len(parts) >= 4:
        year, num, prefix = parts[0], parts[1], parts[2]
        suffix = parts[3]
    else:
        compact = re.sub(r"/+", "/", s).strip("/")
        parts = compact.split("/") if compact else []
        if len(parts) >= 3:
            year, num, prefix = parts[0], parts[1], parts[2]
            suffix = parts[3] if len(parts) >= 4 else ""
        else:
            return re.sub(r"/+", "/", s if s.endswith("/") else s + "/")
    year = f"{int(year):04d}" if str(year).isdigit() else str(year)
    num = f"{int(num):04d}" if str(num).isdigit() else str(num)
    prefix = clean_str(prefix).upper()
    suffix = normalize_case_suffix(suffix)
    return f"{year}/{num}/{prefix}/{suffix}/"


def internal_key_base(value):
    s = normalize_internal_key(value)
    parts = [p for p in s.split("/") if p]
    if len(parts) >= 3:
        return f"{parts[0]}/{parts[1]}/{parts[2]}/"
    return ""

def make_slash_procedure_keys(prefix, num, year_full, suffix=""):
    prefix = clean_str(prefix).upper()
    suffix = normalize_case_suffix(suffix)
    if not prefix or num is None or not clean_str(year_full):
        return "", ""
    base = f"{int(year_full):04d}/{int(num):04d}/{prefix}/"
    full = f"{int(year_full):04d}/{int(num):04d}/{prefix}/{suffix}/"
    return full, base

def parse_case_number(raw: Any, clean: Any = None, prefix_hint: Any = None, year_hint: Any = None):
    raw_s = normalize_whitespace(raw).upper().replace("–", "-").replace("—", "-")
    clean_s = normalize_whitespace(clean).upper().replace("–", "-").replace("—", "-")
    prefix_hint = clean_str(prefix_hint).upper()

    slash_re = r"\b(\d{4})\s*/\s*(\d{1,5})\s*/\s*([CT])\s*/?\s*([A-Z0-9 -]*)\b"
    m0 = re.search(slash_re, raw_s) or (re.search(slash_re, clean_s) if clean_s else None)
    if m0:
        year_full = int(m0.group(1))
        num = int(m0.group(2))
        prefix = m0.group(3)
        suffix = normalize_case_suffix(m0.group(4))
        procedure_key, procedure_key_base = make_slash_procedure_keys(prefix, num, year_full, suffix)
        return {
            "case_prefix": prefix,
            "case_number_int": num,
            "case_number_int_full": f"{num:04d}",
            "case_number_padded": f"{num:04d}",
            "case_year_raw": str(year_full),
            "case_year_full": str(year_full),
            "case_suffix": suffix,
            "procedure_key": procedure_key,
            "procedure_key_base": procedure_key_base,
        }

    m = re.search(r"\b([CT])\s*-?\s*(\d{1,5})\s*/\s*(\d{2,4})\b\s*(.*)$", raw_s)
    if not m and clean_s:
        m = re.search(r"\b([CT])\s*-?\s*(\d{1,5})\s*/\s*(\d{2,4})\b\s*(.*)$", clean_s)

    if m:
        prefix = m.group(1)
        num = int(m.group(2))
        year_raw = m.group(3)
        suffix = normalize_case_suffix(m.group(4))
    else:
        m2 = re.search(r"\b(\d{1,5})\s*/\s*(\d{2,4})\b\s*(.*)$", raw_s or clean_s)
        if not m2:
            return {
                "case_prefix": prefix_hint or "",
                "case_number_int": None,
                "case_number_int_full": "",
                "case_number_padded": "",
                "case_year_raw": clean_str(year_hint),
                "case_year_full": "",
                "case_suffix": "",
                "procedure_key": "",
                "procedure_key_base": "",
            }
        prefix = prefix_hint or "C"
        num = int(m2.group(1))
        year_raw = m2.group(2)
        suffix = normalize_case_suffix(m2.group(3))

    year_int = int(year_raw)
    year_full = 1900 + year_int if year_int < 100 and year_int >= 50 else 2000 + year_int if year_int < 100 else year_int
    procedure_key, procedure_key_base = make_slash_procedure_keys(prefix, num, year_full, suffix)
    return {
        "case_prefix": prefix,
        "case_number_int": num,
        "case_number_int_full": f"{num:04d}",
        "case_number_padded": f"{num:04d}",
        "case_year_raw": year_raw,
        "case_year_full": str(year_full),
        "case_suffix": suffix,
        "procedure_key": procedure_key,
        "procedure_key_base": procedure_key_base,
    }

def add_or_preserve_procedure_key_columns(df, raw_col=None, clean_col=None, prefix_col=None, year_col=None, prefer_internal_key=True):
    if df.empty:
        return df.copy()
    out = df.copy()

    internal_col = next((c for c in ["internal_key", "procedure_key"] if c in out.columns), None) if prefer_internal_key else None
    has_internal = internal_col is not None and out[internal_col].fillna("").astype(str).str.strip().ne("")

    parsed_rows = []
    if raw_col is None:
        raw_col = next((c for c in ["case_number_raw", "publishedId", "case_number", "case", "affaire", "procedure_key"] if c in out.columns), None)
    if clean_col is None:
        clean_col = next((c for c in ["case_number_clean", "case_number", "procedure_key_base"] if c in out.columns), None)
    if prefix_col is None:
        prefix_col = next((c for c in ["case_prefix", "prefix"] if c in out.columns), None)
    if year_col is None:
        year_col = next((c for c in ["case_year_full", "year"] if c in out.columns), None)

    for _, row in out.iterrows():
        parsed_rows.append(parse_case_number(
            row.get(raw_col) if raw_col else "",
            row.get(clean_col) if clean_col else "",
            row.get(prefix_col) if prefix_col else "",
            row.get(year_col) if year_col else "",
        ))
    parsed_df = pd.DataFrame(parsed_rows, index=out.index)

    # Fill missing key component columns, but do not overwrite an upstream internal_key.
    for col in ["case_prefix", "case_number_int", "case_number_int_full", "case_number_padded", "case_year_raw", "case_year_full", "case_suffix"]:
        if col not in out.columns:
            out[col] = parsed_df.get(col)
        else:
            out[col] = out[col].where(out[col].fillna("").astype(str).str.strip().ne(""), parsed_df.get(col))

    if internal_col and prefer_internal_key:
        out["procedure_key"] = out[internal_col].map(normalize_internal_key)
        out["procedure_key_base"] = out["procedure_key"].map(internal_key_base)
    else:
        out["procedure_key"] = parsed_df["procedure_key"]
        out["procedure_key_base"] = parsed_df["procedure_key_base"]

    # Keep `internal_key` explicit in outputs.
    if "internal_key" not in out.columns:
        out["internal_key"] = out["procedure_key"]

    return out

df_cases = add_or_preserve_procedure_key_columns(
    df_cases_in,
    raw_col="case_number_raw" if "case_number_raw" in df_cases_in.columns else None,
    prefer_internal_key=True,
)
# Legacy CURIA may not have an internal_key, so parse it as before.
df_curia_keyed = add_or_preserve_procedure_key_columns(
    df_curia,
    raw_col="procedure_key" if "procedure_key" in df_curia.columns else ("case_number_raw" if "case_number_raw" in df_curia.columns else None),
    prefer_internal_key=False,
) if not df_curia.empty else pd.DataFrame()

print("InfoCuria cases keyed:", df_cases.shape)
if not df_cases.empty and "procedure_key" in df_cases.columns:
    display(df_cases[[c for c in ["internal_key", "case_number_raw", "case_number_clean", "publishedId", "affId", "procedureId", "procedure_key", "procedure_key_base"] if c in df_cases.columns]].head(10))

print("CURIA cases keyed:", df_curia_keyed.shape)
if not df_curia_keyed.empty and "procedure_key" in df_curia_keyed.columns:
    display(df_curia_keyed[[c for c in ["case_number_raw", "case_number_clean", "document_type", "date", "status", "procedure_key", "procedure_key_base"] if c in df_curia_keyed.columns]].head(10))


InfoCuria cases keyed: (56828, 20)


,internal_key,case_number_raw,case_number_clean,affId,procedureId,procedure_key,procedure_key_base
0,2000/0001/C//,C-1/00,C-1/00,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,2000/0001/C//,2000/0001/C/
1,2000/0001/C/SA/,C-1/00 SA,C-1/00 SA,C/0001/00/00000000SA/01,C/0001/00/00000000SA/01/P/01,2000/0001/C/SA/,2000/0001/C/
2,2001/0001/C/P/,C-1/01 P,C-1/01 P,C/0001/01/00000000PV/01,C/0001/01/00000000PV/01/P/01,2001/0001/C/P/,2001/0001/C/
3,2002/0001/C//,C-1/02,C-1/02,C/0001/02/00000000RP/01,C/0001/02/00000000RP/01/P/01,2002/0001/C//,2002/0001/C/
4,2002/0001/C/SA/,C-1/02 SA,C-1/02 SA,C/0001/02/00000000SA/01,C/0001/02/00000000SA/01/P/01,2002/0001/C/SA/,2002/0001/C/
5,2003/0001/C//,C-1/03,C-1/03,C/0001/03/00000000RP/01,C/0001/03/00000000RP/01/P/01,2003/0001/C//,2003/0001/C/
6,2003/0001/C/SA/,C-1/03 SA,C-1/03 SA,C/0001/03/00000000SA/01,C/0001/03/00000000SA/01/P/01,2003/0001/C/SA/,2003/0001/C/
7,2004/0001/C//,C-1/04,C-1/04,C/0001/04/00000000RP/01,C/0001/04/00000000RP/01/P/01,2004/0001/C//,2004/0001/C/
8,2004/0001/C/SA/,C-1/04 SA,C-1/04 SA,C/0001/04/00000000SA/01,C/0001/04/00000000SA/01/P/01,2004/0001/C/SA/,2004/0001/C/
9,2005/0001/C//,C-1/05,C-1/05,C/0001/05/00000000RP/01,C/0001/05/00000000RP/01/P/01,2005/0001/C//,2005/0001/C/


CURIA cases keyed: (50503, 39)


,case_number_raw,case_number_clean,document_type,date,status,procedure_key,procedure_key_base
0,1/53,1/53,removed_from_register,07.05.1954,removed,1953/0001/C//,1953/0001/C/
1,2/53,2/53,removed_from_register,07.05.1954,removed,1953/0002/C//,1953/0002/C/
2,3/53,3/53,removed_from_register,05.11.1953,removed,1953/0003/C//,1953/0003/C/
3,4/53,4/53,removed_from_register,30.10.1953,removed,1953/0004/C//,1953/0004/C/
4,1/54,1/54,judgment,21.12.1954,downloadable,1954/0001/C//,1954/0001/C/
5,2/54,2/54,judgment,21.12.1954,downloadable,1954/0002/C//,1954/0002/C/
6,3/54,3/54,judgment,11.02.1955,downloadable,1954/0003/C//,1954/0003/C/
7,4/54,4/54,judgment,11.02.1955,downloadable,1954/0004/C//,1954/0004/C/
8,5/54,5/54,removed_from_register,26.05.1954,removed,1954/0005/C//,1954/0005/C/
9,6/54,6/54,judgment,21.03.1955,downloadable,1954/0006/C//,1954/0006/C/


# Build or reuse the raw InfoCuria metadata JSONL

This section reintroduces the v2 metadata-fetching mechanism. The metadata endpoint is queried by `affId`, with the published case number passed as a quoted search term. Each response is appended as one JSON line.

For normal reruns, keep `REBUILD_METADATA_JSONL = False` so the notebook uses the existing cache. For a new full scrape, set it to `True`; with `RESUME_METADATA_JSONL = True`, already cached cases are skipped and only missing cases are fetched.


In [16]:

# ---------------------------------------------------------------------
# Optional affairId/procedures metadata fetch/cache construction
# ---------------------------------------------------------------------

def metadata_post_json(url, payload, max_retries=METADATA_MAX_RETRIES):
    last_error = None
    metadata_headers = {
        **HEADERS,
        "accept": "application/json",
        "content-type": "application/json; charset=UTF-8",
    }
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                url,
                headers=metadata_headers,
                json=payload,
                timeout=METADATA_REQUEST_TIMEOUT_SECONDS,
            )
            if response.status_code >= 400:
                last_error = f"HTTP {response.status_code}: {response.text[:500]}"
                raise RuntimeError(last_error)
            return response.json()
        except Exception as e:
            last_error = repr(e)
            wait = min(60, 2 ** attempt) + random.uniform(0, 1.0)
            logger.warning("Metadata request failed attempt %s/%s: %s | sleeping %.1fs", attempt, max_retries, last_error, wait)
            time.sleep(wait)
    raise RuntimeError(f"Metadata request failed after {max_retries} retries: {last_error}")



def fetch_procedure_record(
    aff_id: str,
    published_id: str = "",
    language: str = LANGUAGE,
):
    """
    Query the affairId/procedures endpoint for one procedure-manifest row.

    The endpoint is routed by affId, while the exact procedure label is passed
    in searchTerm. The returned response is later filtered to the requested
    procedureId before case metadata and documents are flattened.
    """
    payload = {
        "affId": aff_id,
        "searchTerm": f'"{published_id}"' if published_id else "",
        "tabName": "affair",
        "language": language,
        "advancedFiltersValue": [],
    }
    return metadata_post_json(PROCEDURES_URL, payload)


def append_raw_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", newline="\n") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def iter_raw_jsonl(path: Path):
    if not path.exists():
        return
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception as e:
                logger.warning("Skipping bad JSONL line %s: %r", line_no, e)


def raw_jsonl_key(record: dict) -> str:
    """
    procedureId is the canonical cache key.

    Fallbacks are retained only for malformed or legacy rows where procedureId
    is absent.
    """
    return first_nonempty(
        record.get("procedureId"),
        record.get("internal_key"),
        record.get("procedure_key"),
        record.get("publishedId"),
        record.get("affId"),
    )


def compact_raw_jsonl_keep_last(path: Path):
    """Rewrite the JSONL so each procedure key appears once, keeping the last record."""
    if not path.exists():
        return

    latest = {}
    order = []

    for rec in iter_raw_jsonl(path):
        key = raw_jsonl_key(rec)
        if not key:
            key = f"__no_key__{len(order)}"

        if key not in latest:
            order.append(key)

        latest[key] = rec

    tmp = path.with_suffix(path.suffix + ".tmp")

    with open(tmp, "w", encoding="utf-8", newline="\n") as f:
        for key in order:
            f.write(json.dumps(latest[key], ensure_ascii=False) + "\n")

    tmp.replace(path)
    print(
        f"Compacted raw JSONL to {len(latest):,} unique procedure records "
        f"-> {path}"
    )


def load_raw_jsonl_keys(path: Path) -> set:
    keys = set()

    if not path.exists():
        return keys

    for obj in tqdm(
        iter_raw_jsonl(path),
        desc="Scanning procedure-level raw JSONL keys",
    ):
        key = raw_jsonl_key(obj)
        if key:
            keys.add(key)

    return keys


def prepare_procedure_fetch_spine(
    df_cases_source: pd.DataFrame,
) -> pd.DataFrame:
    """
    Produce one input row per procedureId.

    Rows with a valid procedureId are deduplicated only by procedureId.
    A fallback path is retained for rows missing procedureId so they can be
    logged and inspected rather than silently lost.
    """
    df_fetch = df_cases_source.copy()

    if "procedureId" not in df_fetch.columns:
        raise KeyError(
            "The InfoCuria case manifest must contain a procedureId column."
        )

    df_fetch["procedureId"] = (
        df_fetch["procedureId"]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    has_procedure_id = df_fetch["procedureId"].notna()

    with_procedure_id = (
        df_fetch.loc[has_procedure_id]
        .drop_duplicates(subset=["procedureId"], keep="last")
    )

    without_procedure_id = df_fetch.loc[~has_procedure_id].copy()

    fallback_cols = [
        c
        for c in ["affId", "internal_key", "publishedId"]
        if c in without_procedure_id.columns
    ]

    if fallback_cols and not without_procedure_id.empty:
        without_procedure_id = without_procedure_id.drop_duplicates(
            subset=fallback_cols,
            keep="last",
        )

    df_fetch = pd.concat(
        [with_procedure_id, without_procedure_id],
        ignore_index=True,
    ).reset_index(drop=True)

    print(
        "Procedure-level fetch spine:",
        f"{len(df_fetch):,} rows |",
        f"{df_fetch['procedureId'].nunique(dropna=True):,} distinct procedureIds |",
        f"{df_fetch['procedureId'].isna().sum():,} rows missing procedureId",
    )

    return df_fetch


def build_or_update_metadata_jsonl(
    df_cases_source: pd.DataFrame,
):
    mode = SCRAPE_MODE

    if mode == "rebuild_from_cache":
        print(
            "SCRAPE_MODE='rebuild_from_cache': using existing "
            "procedure-level raw JSONL cache only:",
            RAW_METADATA_JSONL,
        )
        return

    if mode == "fresh_scrape" and RAW_METADATA_JSONL.exists():
        RAW_METADATA_JSONL.unlink()
        print(
            "Deleted old procedure-level raw JSONL for fresh scrape:",
            RAW_METADATA_JSONL,
        )

    df_fetch = prepare_procedure_fetch_spine(df_cases_source)

    if LIMIT_METADATA_FETCH_CASES is not None:
        df_fetch = df_fetch.head(
            int(LIMIT_METADATA_FETCH_CASES)
        ).copy()

    raw_cache_keys = (
        load_raw_jsonl_keys(RAW_METADATA_JSONL)
        if RAW_METADATA_JSONL.exists()
        else set()
    )

    run_log_rows = []
    failed_rows = []

    for _, row in tqdm(
        df_fetch.iterrows(),
        total=len(df_fetch),
        desc=(
            f"{mode}: fetching InfoCuria procedure-level "
            "affairId/procedures metadata"
        ),
    ):
        procedure_id = clean_str(row.get("procedureId"))
        aff_id = clean_str(row.get("affId"))
        internal_key = clean_str(
            row.get("internal_key")
            or row.get("procedure_key")
        )
        published_id = clean_str(
            row.get("publishedId")
            or row.get("case_number_raw")
            or row.get("case_number_clean")
        )

        cache_key = first_nonempty(
            procedure_id,
            internal_key,
            published_id,
            aff_id,
        )

        if not procedure_id:
            failed_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "error": "missing_procedureId",
            })
            continue

        if not aff_id:
            failed_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "error": "missing_affId",
            })
            continue

        if (
            mode == "update_scrape"
            and cache_key in raw_cache_keys
            and not UPDATE_EXISTING_CACHE_RECORDS
        ):
            run_log_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "status": "cached_skipped",
            })
            continue

        try:
            data = fetch_procedure_record(
                aff_id=aff_id,
                published_id=published_id,
                language=LANGUAGE,
            )

            rec = {
                "fetched_at": (
                    datetime.utcnow()
                    .isoformat(timespec="seconds")
                    + "Z"
                ),
                "cache_key": cache_key,
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "response": data,
            }

            append_raw_jsonl(
                RAW_METADATA_JSONL,
                rec,
            )

            raw_cache_keys.add(cache_key)

            run_log_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "status": "ok",
                "totalHits": data.get("totalHits"),
            })

        except Exception as e:
            failed_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "error": repr(e),
            })

            run_log_rows.append({
                "affId": aff_id,
                "procedureId": procedure_id,
                "internal_key": internal_key,
                "publishedId": published_id,
                "procedure_key": row.get("procedure_key"),
                "status": "failed",
                "error": repr(e),
            })

        time.sleep(float(METADATA_SLEEP_SECONDS))

    df_log = pd.DataFrame(run_log_rows)
    df_fail = pd.DataFrame(failed_rows)

    save_jsonl(
        df_log,
        METADATA_FETCH_LOG_JSONL,
    )
    save_parquet(
        df_log,
        METADATA_FETCH_LOG_PARQUET,
    )
    save_excel(
        df_log,
        METADATA_FETCH_LOG_XLSX,
        sheet_name="fetch_log",
    )

    save_jsonl(
        df_fail,
        METADATA_FETCH_FAILURES_JSONL,
    )
    save_parquet(
        df_fail,
        METADATA_FETCH_FAILURES_PARQUET,
    )
    save_excel(
        df_fail,
        METADATA_FETCH_FAILURES_XLSX,
        sheet_name="fetch_failures",
    )

    compact_raw_jsonl_keep_last(
        RAW_METADATA_JSONL
    )

    print("Procedure-level metadata fetch/update pass complete.")
    print("Fetch log rows:", f"{len(run_log_rows):,}")
    print("Failures:", f"{len(failed_rows):,}")


build_or_update_metadata_jsonl(df_cases)

if not RAW_METADATA_JSONL.exists():
    raise FileNotFoundError(f"Raw metadata JSONL still does not exist: {RAW_METADATA_JSONL}")

print("Raw JSONL ready:", RAW_METADATA_JSONL)
print("Raw JSONL size MB:", round(file_size_mb(RAW_METADATA_JSONL), 2))
print("Raw JSONL line count:", f"{count_jsonl_lines(RAW_METADATA_JSONL):,}")


Procedure-level fetch spine: 56,828 rows | 56,828 distinct procedureIds | 0 rows missing procedureId


Scanning procedure-level raw JSONL keys: 0it [00:00, ?it/s]

update_scrape: fetching InfoCuria procedure-level affairId/procedures metadata:   0%|          | 0/56828 [00:0…

/tmp/ipykernel_50701/2730840949.py:308: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow()


Saved JSONL 56,828 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_metadata_fetch_log.jsonl
Saved parquet 56,828 rows -> /home/edik/projects/eccjeu/output/court_infocuria/parquet/infocuria_metadata_fetch_log.parquet
Saved Excel 56,828 rows -> /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_metadata_fetch_log.xlsx
Saved JSONL 0 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_metadata_fetch_failures.jsonl
Saved parquet 0 rows -> /home/edik/projects/eccjeu/output/court_infocuria/parquet/infocuria_metadata_fetch_failures.parquet
Saved Excel 0 rows -> /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_metadata_fetch_failures.xlsx
Compacted raw JSONL to 56,828 unique procedure records -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_procedural_metadata_by_procedure_raw.jsonl
Procedure-level metadata fetch/update pass complete.
Fetch log rows: 56,828
Failures: 0
Raw JSONL ready: /home/edik

In [17]:
# ---------------------------------------------------------------------
# Raw JSONL iterator and InfoCuria flattening helpers
# ---------------------------------------------------------------------

def iter_raw_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception as e:
                logger.warning("Skipping bad JSONL line %s: %r", line_no, e)

def select_requested_procedure_hit(raw_rec):
    """
    Select the response hit matching the requested procedureId.

    Matching order:
      1. exact procedureId
      2. exact publishedId
      3. exact affId only when the response contains one hit
      4. no match -> return an empty result rather than silently using
         the wrong procedure
    """
    response = raw_rec.get("response") or {}
    hits = response.get("searchHits") or []

    if not hits:
        return None, {}

    requested_procedure_id = clean_str(
        raw_rec.get("procedureId")
    )

    if requested_procedure_id:
        for hit in hits:
            content = hit.get("content") or {}

            if clean_str(
                content.get("procedureId")
            ) == requested_procedure_id:
                return hit, content

    requested_published_id = clean_str(
        raw_rec.get("publishedId")
    )

    if requested_published_id:
        for hit in hits:
            content = hit.get("content") or {}

            if clean_str(
                content.get("publishedId")
            ) == requested_published_id:
                return hit, content

    if len(hits) == 1:
        hit = hits[0]
        content = hit.get("content") or {}

        requested_aff_id = clean_str(
            raw_rec.get("affId")
        )

        if (
            not requested_aff_id
            or clean_str(content.get("affId")) == requested_aff_id
        ):
            return hit, content

    return None, {}

def extract_inner_document_hits(hit):
    if not isinstance(hit, dict):
        return []
    inner = hit.get("innerHits") or {}
    doc_block = inner.get("document") or {}
    return doc_block.get("searchHits") or []

def extract_available_langs_from_content_ml(content_ml):
    if isinstance(content_ml, str):
        try:
            content_ml = json.loads(content_ml)
        except Exception:
            return []
    langs = []
    if isinstance(content_ml, list):
        for item in content_ml:
            if isinstance(item, dict):
                langs.extend([str(k).upper() for k in item.keys()])
    return sorted(set(langs))

def normalize_doc_formats(value):
    if value is None:
        return []
    if isinstance(value, list):
        vals = value
    elif isinstance(value, str):
        try:
            parsed = json.loads(value)
            vals = parsed if isinstance(parsed, list) else [value]
        except Exception:
            vals = re.split(r"[|,; ]+", value)
    else:
        vals = [value]
    return sorted(set([str(x).upper().strip() for x in vals if str(x).strip()]))

# Build input lookup once. This enriches raw JSON rows with fields from the original input manifest.
INPUT_BY_AFFID = {}
INPUT_BY_PROCEDURE_ID = {}
INPUT_BY_PUBLISHED_ID = {}

if not df_cases.empty:
    for _, _row in df_cases.iterrows():
        d = _row.to_dict()
        for col, target in [
            ("affId", INPUT_BY_AFFID),
            ("procedureId", INPUT_BY_PROCEDURE_ID),
            ("publishedId", INPUT_BY_PUBLISHED_ID),
            ("case_number_raw", INPUT_BY_PUBLISHED_ID),
        ]:
            key = clean_str(_row.get(col)) if col in df_cases.columns else ""
            if key and key not in target:
                target[key] = d

def lookup_input_row(raw_rec, content):
    """
    Resolve the upstream manifest row using procedureId first.
    """
    for key, mapping in [
        (
            clean_str(
                raw_rec.get("procedureId")
                or content.get("procedureId")
            ),
            INPUT_BY_PROCEDURE_ID,
        ),
        (
            clean_str(
                raw_rec.get("publishedId")
                or content.get("publishedId")
            ),
            INPUT_BY_PUBLISHED_ID,
        ),
        (
            clean_str(
                raw_rec.get("affId")
                or content.get("affId")
            ),
            INPUT_BY_AFFID,
        ),
    ]:
        if key and key in mapping:
            return mapping[key]

    return {}

def flatten_case_record(raw_rec):
    response = raw_rec.get("response") or {}
    hit, c = select_requested_procedure_hit(raw_rec)

    if not c:
        raise ValueError(
            "No response hit matched requested procedureId="
            f"{raw_rec.get('procedureId')!r}, "
            f"publishedId={raw_rec.get('publishedId')!r}, "
            f"affId={raw_rec.get('affId')!r}"
        )

    input_row = lookup_input_row(raw_rec, c)

    published = c.get("publishedId") or input_row.get("publishedId") or input_row.get("case_number_raw") or raw_rec.get("publishedId")
    parsed = parse_case_number(published, input_row.get("case_number_clean"), input_row.get("case_prefix"))

    calendars = c.get("calendars") or []
    delivery_dates = sorted({x.get("seanceDate") for x in calendars if isinstance(x, dict) and x.get("seanceDate")})
    delivery_types = sorted({x.get("seanceType") for x in calendars if isinstance(x, dict) and x.get("seanceType")})

    # Keep the existing notebook-level procedure_key for compatibility, but also expose
    # an explicit InfoCuria slash key in the requested format:
    #   infocuria_slash_procedure_key_base = YYYY/NNNN/PREFIX/
    #   infocuria_slash_procedure_key      = YYYY/NNNN/PREFIX/ or YYYY/NNNN/PREFIX/SUFFIX
    # This makes the matching key auditable even if upstream InfoCuria metadata already
    # contains another field called procedure_key.
    infocuria_slash_procedure_key = parsed.get("procedure_key") or ""
    infocuria_slash_procedure_key_base = parsed.get("procedure_key_base") or ""

    row = {
        "internal_key": normalize_internal_key(input_row.get("internal_key") or raw_rec.get("internal_key") or raw_rec.get("procedure_key") or input_row.get("procedure_key") or infocuria_slash_procedure_key),
        "procedure_key": normalize_internal_key(input_row.get("internal_key") or raw_rec.get("internal_key") or raw_rec.get("procedure_key") or input_row.get("procedure_key") or infocuria_slash_procedure_key),
        "procedure_key_base": internal_key_base(input_row.get("internal_key") or raw_rec.get("internal_key") or raw_rec.get("procedure_key") or input_row.get("procedure_key") or infocuria_slash_procedure_key) or infocuria_slash_procedure_key_base or input_row.get("procedure_key_base"),
        "infocuria_slash_procedure_key": infocuria_slash_procedure_key,
        "infocuria_slash_procedure_key_base": infocuria_slash_procedure_key_base,
        "case_number_raw": published,
        "case_number_clean": input_row.get("case_number_clean"),
        "case_prefix": parsed.get("case_prefix"),
        "case_number_int": parsed.get("case_number_int"),
        "case_number_int_full": parsed.get("case_number_int_full"),
        "case_year_raw": parsed.get("case_year_raw"),
        "case_year_full": parsed.get("case_year_full"),
        "case_suffix": parsed.get("case_suffix"),
        "affId": c.get("affId") or raw_rec.get("affId") or input_row.get("affId"),
        "publishedId": c.get("publishedId") or published,
        "publishedAffId": c.get("publishedAffId"),
        "procedureId": c.get("procedureId") or raw_rec.get("procedureId") or input_row.get("procedureId"),
        "jurisdiction": c.get("jurisdiction"),
        "jurisdictionCode": c.get("jurisdictionCode"),
        "nature": c.get("nature"),
        "natureCode": c.get("natureCode"),
        "procType": c.get("procType"),
        "procNumber": c.get("procNumber"),
        "suiteNumber": c.get("suiteNumber"),
        "case_name_en": ml_value(c.get("usualNameML"), "en"),

        "case_object_en": ml_value(c.get("affObjectML"), "en"),

        "introductionDate": c.get("introductionDate"),
        "procIntroDate": c.get("procIntroDate"),
        "introductionYear": c.get("introductionYear"),
        "closeDate": c.get("closeDate"),
        "procClosDate": c.get("procClosDate"),
        "closeYear": c.get("closeYear"),
        "affairState": c.get("affairState"),
        "affairStateCode": c.get("affairStateCode"),
        "procState": c.get("procState"),
        "procLang": c.get("procLang"),
        "joinExist": c.get("joinExist"),
        "pilot": c.get("pilot"),
        "idPilot": c.get("idPilot"),
        "pilot_case_publishedId": parse_affair_ref_list([c.get("idPilot")])[0] if c.get("idPilot") and parse_affair_ref_list([c.get("idPilot")]) else "",
        "joinAffairs": as_json_string(c.get("joinAffairs") or []),
        "joinAffairs_publishedIds": as_json_string(parse_affair_ref_list(c.get("joinAffairs") or [])),
        "pourvoiAffIds": as_json_string(c.get("pourvoiAffIds") or []),
        "pourvoi_publishedIds": as_json_string(parse_affair_ref_list(c.get("pourvoiAffIds") or [])),
        "originPourvoiIds": as_json_string(c.get("originPourvoiIds") or []),
        "origin_pourvoi_publishedIds": as_json_string(parse_affair_ref_list(c.get("originPourvoiIds") or [])),
        "reExamenAffIds": as_json_string(c.get("reExamenAffIds") or []),
        "originReExamanIds": as_json_string(c.get("originReExamanIds") or []),
        "transfertAffIds": as_json_string(c.get("transfertAffIds") or []),
        "originTransfertIds": as_json_string(c.get("originTransfertIds") or []),
        "matCode": as_json_string(c.get("matCode") or []),
        "subject_matter_en": label_text(c.get("matCodeML"), "en"),

        "reportingJudge": c.get("reportingJudge"),
        "formation": c.get("formation"),
        "advocate_general_code": c.get("avg"),
        "resultTypes": as_json_string(c.get("resultTypes") or []),
        "procedure_result_en": label_text(c.get("procedureResultTypeML"), "en"),

        "parties": c.get("parties"),
        "conclsDate": as_json_string(c.get("conclsDate") or []),
        "conclLang": c.get("conclLang"),
        "delivery_dates": as_json_string(delivery_dates),
        "delivery_types": as_json_string(delivery_types),
        "numDocs": c.get("numDocs"),
        "doctrineNotes_count": len(c.get("doctrineNotes") or []),
        "lastModifiedDate": c.get("lastModifiedDate"),
        "totalHits": response.get("totalHits"),
        "inner_document_totalHits": (((hit or {}).get("innerHits") or {}).get("document") or {}).get("totalHits"),
        "raw_case_content_json": json.dumps(c, ensure_ascii=False) if KEEP_RAW_CONTENT_JSON else "",
    }
    return row

def flatten_document_records(raw_rec):
    response = raw_rec.get("response") or {}
    hit, c = select_requested_procedure_hit(raw_rec)

    if not c:
        raise ValueError(
            "No document-bearing response hit matched requested procedureId="
            f"{raw_rec.get('procedureId')!r}, "
            f"publishedId={raw_rec.get('publishedId')!r}, "
            f"affId={raw_rec.get('affId')!r}"
        )

    case_row = flatten_case_record(raw_rec)
    docs = extract_inner_document_hits(hit)
    rows = []

    for pos, dh in enumerate(docs):
        dc = dh.get("content") or {}
        content_ml = dc.get("contentML") or []
        available_langs = extract_available_langs_from_content_ml(content_ml)
        doc_formats = normalize_doc_formats(dc.get("docFormats") or ([dc.get("docFormat")] if dc.get("docFormat") else []))
        code = normalize_doc_type(dc.get("docTypeCode") or dc.get("docType"))

        row = {
            "internal_key": case_row.get("internal_key"),
            "procedure_key": case_row.get("procedure_key"),
            "procedure_key_base": case_row.get("procedure_key_base"),
            "infocuria_slash_procedure_key": case_row.get("infocuria_slash_procedure_key"),
            "infocuria_slash_procedure_key_base": case_row.get("infocuria_slash_procedure_key_base"),
            "case_number_raw": case_row.get("case_number_raw"),
            "case_number_clean": case_row.get("case_number_clean"),
            "case_prefix": case_row.get("case_prefix"),
            "case_number_int": case_row.get("case_number_int"),
            "case_number_int_full": case_row.get("case_number_int_full"),
            "case_year_full": case_row.get("case_year_full"),
            "case_suffix": case_row.get("case_suffix"),
            "affId": case_row.get("affId"),
            "publishedId": case_row.get("publishedId"),
            "procedureId": case_row.get("procedureId"),
            "procLang": case_row.get("procLang"),
            "case_name_en": case_row.get("case_name_en"),
            "introductionDate": case_row.get("introductionDate"),
            "closeDate": case_row.get("closeDate"),
            "procIntroDate": case_row.get("procIntroDate"),
            "procClosDate": case_row.get("procClosDate"),
            "doc_position": pos,
            "inner_hit_id": dh.get("id"),
            "docId": dc.get("docId"),
            "logicDocId": dc.get("logicDocId"),
            "jpLogicDocId": dc.get("jpLogicDocId"),
            "idProcedure": dc.get("idProcedure"),
            "idPublished": dc.get("idPublished"),
            "docType": dc.get("docType"),
            "docTypeCode": dc.get("docTypeCode"),
            "normalized_docTypeCode": code,
            "relevance_flag": is_relevant_doc_type(code),
            "relevance_category": "relevant" if is_relevant_doc_type(code) else "irrelevant",
            "relevance_reason": relevance_reason(code),
            "lnsDec": dc.get("lnsDec"),
            "numDoc": dc.get("numDoc"),
            "docDate": dc.get("docDate"),
            "docFormat": dc.get("docFormat"),
            "docFormats": as_json_string(doc_formats),
            "docLang": str(dc.get("docLang") or "").upper(),
            "available_languages": as_json_string(available_langs),
            "docUrl": dc.get("docUrl"),
            "contentLength": dc.get("contentLength"),
            # Keep only English context text here. It is split into a separate compact context manifest
            # before writing final outputs, so the metadata manifest remains manageable.
            "context_text_en": ml_value(content_ml, "en", fallback=False),
            "raw_document_content_json": json.dumps(dc, ensure_ascii=False) if KEEP_RAW_CONTENT_JSON else "",
        }

        row["document_dedup_key"] = "|".join([
            clean_str(row.get("procedureId"))
            or clean_str(row.get("procedure_key")),
            clean_str(row.get("logicDocId"))
            or clean_str(row.get("docId"))
            or clean_str(row.get("inner_hit_id")),
            clean_str(row.get("normalized_docTypeCode")),
            clean_str(row.get("docDate")),
        ])
        rows.append(row)

    return rows


In [18]:

# ---------------------------------------------------------------------
# Stream raw JSONL into flattened case/document tables
# ---------------------------------------------------------------------

# Stream files are written directly under JSONL_DIR and deleted after loading.
# No temporary enrich folder is created in v5.
CASE_METADATA_STREAM_JSONL = JSONL_DIR / "_case_metadata_stream.jsonl"
DOCUMENT_METADATA_STREAM_JSONL = JSONL_DIR / "_document_metadata_stream.jsonl"

for p in [CASE_METADATA_STREAM_JSONL, DOCUMENT_METADATA_STREAM_JSONL, FLATTEN_FAILED_JSONL]:
    if p.exists():
        p.unlink()

def append_rows_jsonl(rows, path: Path):
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", newline="\n") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

case_buffer = []
doc_buffer = []
failed_buffer = []
seen_cases = 0
seen_docs = 0

for rec in tqdm(iter_raw_jsonl(RAW_METADATA_JSONL), desc="Streaming flatten raw JSONL"):
    try:
        case_buffer.append(flatten_case_record(rec))
        doc_rows = flatten_document_records(rec)
        doc_buffer.extend(doc_rows)
        seen_cases += 1
        seen_docs += len(doc_rows)
    except Exception as e:
        failed_buffer.append({
            "affId": rec.get("affId"),
            "publishedId": rec.get("publishedId"),
            "procedureId": rec.get("procedureId"),
            "internal_key": rec.get("internal_key"),
            "procedure_key": rec.get("procedure_key"),
            "error": "flatten_failed: " + repr(e),
        })

    if len(case_buffer) >= STREAM_JSONL_CHUNK_SIZE:
        append_rows_jsonl(case_buffer, CASE_METADATA_STREAM_JSONL)
        case_buffer = []
    if len(doc_buffer) >= STREAM_JSONL_CHUNK_SIZE:
        append_rows_jsonl(doc_buffer, DOCUMENT_METADATA_STREAM_JSONL)
        doc_buffer = []
    if len(failed_buffer) >= STREAM_JSONL_CHUNK_SIZE:
        append_rows_jsonl(failed_buffer, FLATTEN_FAILED_JSONL)
        failed_buffer = []

append_rows_jsonl(case_buffer, CASE_METADATA_STREAM_JSONL)
append_rows_jsonl(doc_buffer, DOCUMENT_METADATA_STREAM_JSONL)
append_rows_jsonl(failed_buffer, FLATTEN_FAILED_JSONL)

print("Streamed case records:", f"{seen_cases:,}")
print("Streamed document records:", f"{seen_docs:,}")
print("Flatten failures:", FLATTEN_FAILED_JSONL if FLATTEN_FAILED_JSONL.exists() else "none")

df_case_flat = read_manifest(CASE_METADATA_STREAM_JSONL) if CASE_METADATA_STREAM_JSONL.exists() else pd.DataFrame()
df_doc_meta = read_manifest(DOCUMENT_METADATA_STREAM_JSONL) if DOCUMENT_METADATA_STREAM_JSONL.exists() else pd.DataFrame()

# Deduplicate internally.
#
# procedureId is the canonical case/procedure key. Rows with a populated
# procedureId are deduplicated only by procedureId. Fallback identifiers
# are used only for rows where procedureId is missing.
if not df_case_flat.empty:
    if "procedureId" in df_case_flat.columns:
        has_procedure_id = (
            df_case_flat["procedureId"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
        )

        with_procedure_id = (
            df_case_flat.loc[has_procedure_id]
            .drop_duplicates(subset=["procedureId"], keep="last")
        )

        without_procedure_id = df_case_flat.loc[~has_procedure_id].copy()

        fallback_cols = [
            c
            for c in ["affId", "procedure_key", "internal_key"]
            if c in without_procedure_id.columns
        ]

        if fallback_cols:
            without_procedure_id = without_procedure_id.drop_duplicates(
                subset=fallback_cols,
                keep="last",
            )

        df_case_flat = pd.concat(
            [with_procedure_id, without_procedure_id],
            ignore_index=True,
        ).reset_index(drop=True)

    else:
        fallback_cols = [
            c
            for c in ["affId", "procedure_key", "internal_key"]
            if c in df_case_flat.columns
        ]

        if fallback_cols:
            df_case_flat = df_case_flat.drop_duplicates(
                subset=fallback_cols,
                keep="last",
            ).reset_index(drop=True)

if not df_doc_meta.empty and "document_dedup_key" in df_doc_meta.columns:
    df_doc_meta = df_doc_meta.drop_duplicates(
        subset=["document_dedup_key"],
        keep="last",
    ).reset_index(drop=True)

# Split English context text into a separate compact context manifest.
context_id_cols = [
    "internal_key", "document_dedup_key", "procedure_key", "procedure_key_base", "infocuria_slash_procedure_key",
    "case_number_raw", "case_number_clean", "affId", "procedureId", "publishedId",
    "docId", "logicDocId", "jpLogicDocId", "idProcedure", "idPublished",
    "docTypeCode", "normalized_docTypeCode", "docDate",
]
if not df_doc_meta.empty:
    if "context_text_en" not in df_doc_meta.columns:
        df_doc_meta["context_text_en"] = ""
    df_document_context_manifest = df_doc_meta[[c for c in context_id_cols if c in df_doc_meta.columns] + ["context_text_en"]].copy()
else:
    df_document_context_manifest = pd.DataFrame(columns=["logicDocId", "context_text_en"])

# Remove large context/raw fields and non-English fields from document metadata.
large_doc_cols = [
    "context_text_en", "content_text_en", "content_text_de", "content_text_fr", "content_text_best", "raw_document_content_json"
]
df_doc_meta = df_doc_meta.drop(columns=[c for c in large_doc_cols + DROP_NON_ENGLISH_METADATA_COLUMNS if c in df_doc_meta.columns], errors="ignore")
df_case_flat = df_case_flat.drop(columns=[c for c in DROP_NON_ENGLISH_METADATA_COLUMNS if c in df_case_flat.columns], errors="ignore")

# Remove temporary stream files. Final manifests are written later.
for p in [CASE_METADATA_STREAM_JSONL, DOCUMENT_METADATA_STREAM_JSONL]:
    try:
        if p.exists():
            p.unlink()
    except Exception as e:
        print(f"Could not delete temporary stream file {p}: {e}")

print("Case metadata rows after dedupe:", df_case_flat.shape)
print("Document metadata rows after dedupe, without context text:", df_doc_meta.shape)
print("Document context rows:", df_document_context_manifest.shape)
display(df_case_flat.head(3))
display(df_doc_meta.head(3))
display(df_document_context_manifest.head(3))


Streaming flatten raw JSONL: 0it [00:00, ?it/s]

Streamed case records: 56,828
Streamed document records: 152,915
Flatten failures: none
Case metadata rows after dedupe: (56828, 68)
Document metadata rows after dedupe, without context text: (152915, 44)
Document context rows: (152915, 19)


,internal_key,procedure_key,procedure_key_base,infocuria_slash_procedure_key,infocuria_slash_procedure_key_base,case_number_raw,case_number_clean,case_prefix,case_number_int,case_number_int_full,case_year_raw,case_year_full,case_suffix,affId,publishedId,publishedAffId,procedureId,jurisdiction,jurisdictionCode,nature,natureCode,procType,procNumber,suiteNumber,case_name_en,case_object_en,introductionDate,procIntroDate,introductionYear,closeDate,procClosDate,closeYear,affairState,affairStateCode,procState,procLang,joinExist,pilot,idPilot,pilot_case_publishedId,joinAffairs,joinAffairs_publishedIds,pourvoiAffIds,pourvoi_publishedIds,originPourvoiIds,origin_pourvoi_publishedIds,reExamenAffIds,originReExamanIds,transfertAffIds,originTransfertIds,matCode,subject_matter_en,reportingJudge,formation,advocate_general_code,resultTypes,procedure_result_en,parties,conclsDate,conclLang,delivery_dates,delivery_types,numDocs,doctrineNotes_count,lastModifiedDate,totalHits,inner_document_totalHits,raw_case_content_json
0,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,00,2000,,C/0001/00/00000000RD/01,C-1/00,C-1/00,C/0001/00/00000000RD/01/P/01,Cour de justice,C,Recours direct,RD,P,1,1,Commission v France,"Failure by a State to comply with its obligations ─ Refusal to adopt the necessary measures to comply with Council Decision 98/256/EC (emergency measures to protect against bovine spongiform encephalopathy) and Commission Decision 1999/514/EC (setting the date for resumption of shipments of ""DBE...",2000-01-04,NaN,2000,2001-12-13,2001-12-13,2001.0,Affaire clôturée,CLOTPUB,CLOTPUB,FR,0,0,NaN,,[],[],[],[],[],[],[],[],[],[],"[""AGRI"", ""AGRI.VETE"", ""AGRI.OCMA""]",Agriculture|Common organisation of the markets|Veterinary legislation,SEV,CP_C,MIS,"[""CONS=OB"", ""CONS=RF""]",Action for a declaration of failure to fulfil obligations: application granted|Action for a declaration of failure to fulfil obligations: dismissal on substantive grounds,Commission / France,"[""2001-09-20""]",FR,"[""2001-06-19"", ""2001-12-13""]","[""ARRET"", ""PO""]",C2000/0001/J,7,2026-07-14T08:27:55.43,1,5,
1,2000/0001/C/SA/,2000/0001/C/SA/,2000/0001/C/,2000/0001/C/SA/,2000/0001/C/,C-1/00 SA,C-1/00 SA,C,1,0001,00,2000,SA,C/0001/00/00000000SA/01,C-1/00 SA,C-1/00 SA,C/0001/00/00000000SA/01/P/01,Cour de justice,C,Saisie-arrêt,SA,P,1,1,Cotecna Inspection v Commission,"Application for leave pursuant to Article 1 of the Protocol on Privileges and Immunities of the European Communities to attach assets held by the Commission, by way of enforcement of an arbitration award made against the State of Djibouti which has been rendered enforceable by order of a Belgian...",2000-12-14,2000-12-14,2000,2001-05-29,2001-05-29,2001.0,Affaire clôturée,CLOTPUB,NaN,FR,0,0,NaN,,[],[],[],[],[],[],[],[],[],[],"[""PRIV""]",Privileges and immunities,VBO,CP_C,RUI,"[""IMMU=RF""]",Request concerning immunities: dismissal on substantive grounds,Cotecna Inspection / Commission,[],,[],[],C2000/0001/S,0,2025-10-25T15:04:50.623,1,2,
2,2001/0001/C/P/,2001/0001/C/P/,2001/0001/C/,2001/0001/C/P/,2001/0001/C/,C-1/01 P,C-1/01 P,C,1,0001,01,2001,P,C/0001/01/00000000PV/01,C-1/01 P,C-1/01 P,C/0001/01/00000000PV/01/P/01,Cour de justice,C,Pourvoi,PV,P,1,1,Asia Motor France and Others v Commission,Case C-01/01 P\r\n\r\nAppeal against the judgment in Case T-154/98 by which the Court of First Instance dismissed an application for annulment of a decision rejecting the appellants' complaints regarding an alleged agreement between importers into France of five makes of Japanese motor-car.,2001-01-03,2001-01-03,2001,2001-09-20,2001-09-20,2001.0,Affaire clôturée,CLOTPUB,CLOTPUB,FR,0,0,NaN,,[],[],"[""C/0001/01/00000000PV/01/P/01#C-1/01 P""]","[""C-1/01 P""]","[""T/0154/98/00000000RD/01/P/01#T-154/98""]","[""T-154/98""]",[],[],[],[],"[""CONC"", ""CONC.ENTE""]","Agreements, decisions and concerted practices|Competition",STG,II_C,JAC,"[""ANNU"", ""PVOI=RI"", ""PVOI=RF""]",Actions for annu

,internal_key,procedure_key,procedure_key_base,infocuria_slash_procedure_key,infocuria_slash_procedure_key_base,case_number_raw,case_number_clean,case_prefix,case_number_int,case_number_int_full,case_year_full,case_suffix,affId,publishedId,procedureId,procLang,case_name_en,introductionDate,closeDate,procIntroDate,procClosDate,doc_position,inner_hit_id,docId,logicDocId,jpLogicDocId,idProcedure,idPublished,docType,docTypeCode,normalized_docTypeCode,relevance_flag,relevance_category,relevance_reason,lnsDec,numDoc,docDate,docFormat,docFormats,docLang,available_languages,docUrl,contentLength,document_dedup_key
0,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,0,C/0001/00/00000000RD/01/P/01-623390,623390,id_53538,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Avis (JO),AVI_COMM,AVI_COMM,False,irrelevant,irrelevant_non_final_or_duplicate_or_administrative_document,,,2002-06-14,HTML,"[""HTML""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_53538|AVI_COMM|2002-06-14
1,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,1,C/0001/00/00000000RD/01/P/01-1216021,1216021,id_85906,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Arrêt (Sommaire),ARRET_SOM,ARRET_SOM,True,relevant,relevant_summary_judgment_variant,,C2000/0001/J,2001-12-13,PDF,"[""PDF""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_85906|ARRET_SOM|2001-12-13
2,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,2,C/0001/00/00000000RD/01/P/01-616977,616977,id_46950,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Arrêt,ARRET,ARRET,True,relevant,relevant_final_judgment,217582,C2000/0001/J,2001-12-13,HTML,"[""HTML"", ""PDF""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_46950|ARRET|2001-12-13


,internal_key,document_dedup_key,procedure_key,procedure_key_base,infocuria_slash_procedure_key,case_number_raw,case_number_clean,affId,procedureId,publishedId,docId,logicDocId,jpLogicDocId,idProcedure,idPublished,docTypeCode,normalized_docTypeCode,docDate,context_text_en
0,2000/0001/C//,C/0001/00/00000000RD/01/P/01|id_53538|AVI_COMM|2002-06-14,2000/0001/C//,2000/0001/C/,2000/0001/C//,C-1/00,C-1/00,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,C-1/00,623390,id_53538,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,AVI_COMM,AVI_COMM,2002-06-14,"Notice for the OJ Opinion 1/00 of the Court of 18 April 2002 (Opinion pursuant to Article 300(6) EC ( Proposed agreement between the European Community and non-Member States on the establishment of a European Common Aviation Area) The Court of Justice has received a request for an opinion, lodge..."
1,2000/0001/C//,C/0001/00/00000000RD/01/P/01|id_85906|ARRET_SOM|2001-12-13,2000/0001/C//,2000/0001/C/,2000/0001/C//,C-1/00,C-1/00,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,C-1/00,1216021,id_85906,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,ARRET_SOM,ARRET_SOM,2001-12-13,"Case C-1/00 \nCommission of the European Communities \nv \nFrench Republic \n(Failure by a Member State to fulfil its obligations — \nRefusal to end the ban on British beef and veal) \nOpinion of Advocate General Mischo delivered on 20 September 2001 . . I - 9993 \nJudgment of the Court, 13 Dece..."
2,2000/0001/C//,C/0001/00/00000000RD/01/P/01|id_46950|ARRET|2001-12-13,2000/0001/C//,2000/0001/C/,2000/0001/C//,C-1/00,C-1/00,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,C-1/00,616977,id_46950,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,ARRET,ARRET,2001-12-13,"JUDGMENT OF THE COURT 13 December 2001 (1) (Failure of a Member State to fulfil its obligations - Refusal to end the ban on British beef and veal) In Case C-1/00, Commission of the European Communities, represented by D. Booss and G. Berscheid, acting as Agents, with an address for service in Lu..."


In [19]:
# ---------------------------------------------------------------------
# Document metadata manifest base
# ---------------------------------------------------------------------

# The final document metadata manifest is df_doc_meta after relevance columns are assigned.
# It intentionally excludes context text and raw document JSON.
print("Document metadata base rows:", len(df_doc_meta))
if not df_doc_meta.empty:
    display(df_doc_meta.head(10))


Document metadata base rows: 152915


,internal_key,procedure_key,procedure_key_base,infocuria_slash_procedure_key,infocuria_slash_procedure_key_base,case_number_raw,case_number_clean,case_prefix,case_number_int,case_number_int_full,case_year_full,case_suffix,affId,publishedId,procedureId,procLang,case_name_en,introductionDate,closeDate,procIntroDate,procClosDate,doc_position,inner_hit_id,docId,logicDocId,jpLogicDocId,idProcedure,idPublished,docType,docTypeCode,normalized_docTypeCode,relevance_flag,relevance_category,relevance_reason,lnsDec,numDoc,docDate,docFormat,docFormats,docLang,available_languages,docUrl,contentLength,document_dedup_key
0,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,0,C/0001/00/00000000RD/01/P/01-623390,623390,id_53538,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Avis (JO),AVI_COMM,AVI_COMM,False,irrelevant,irrelevant_non_final_or_duplicate_or_administrative_document,,,2002-06-14,HTML,"[""HTML""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_53538|AVI_COMM|2002-06-14
1,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,1,C/0001/00/00000000RD/01/P/01-1216021,1216021,id_85906,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Arrêt (Sommaire),ARRET_SOM,ARRET_SOM,True,relevant,relevant_summary_judgment_variant,,C2000/0001/J,2001-12-13,PDF,"[""PDF""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_85906|ARRET_SOM|2001-12-13
2,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,2,C/0001/00/00000000RD/01/P/01-616977,616977,id_46950,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Arrêt,ARRET,ARRET,True,relevant,relevant_final_judgment,217582,C2000/0001/J,2001-12-13,HTML,"[""HTML"", ""PDF""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_46950|ARRET|2001-12-13
3,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,3,C/0001/00/00000000RD/01/P/01-616653,616653,id_46615,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Conclusions,CONCL,CONCL,False,irrelevant,irrelevant_non_final_or_duplicate_or_administrative_document,,C2000/0001/J,2001-09-20,HTML,"[""HTML"", ""PDF""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_46615|CONCL|2001-09-20
4,2000/0001/C//,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,C-1/00,C,1,0001,2000,,C/0001/00/00000000RD/01,C-1/00,C/0001/00/00000000RD/01/P/01,FR,Commission v France,2000-01-04,2001-12-13,NaN,2001-12-13,4,C/0001/00/00000000RD/01/P/01-623384,623384,id_53532,id_null,C/0001/00/00000000RD/01/P/01,C-1/00,Arrêt (JO),ARR_COMM,ARR_COMM,False,irrelevant,irrelevant_non_final_or_duplicate_or_administrative_document,,,2002-02-16,HTML,"[""HTML""]",FR,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]",,0,C/0001/00/00000000RD/01/P/01|id_53532|ARR_COMM|2002-02-16
5,2000/0001/C/SA/,2000/0001/C/SA/,2000/0001/C/,2000/0001/C/SA/,2000/0001/C/,C-1/00 SA,C-1/00 SA,C,1,0001,2000,SA,C/0001/00/00000000SA/01,C-1/00 SA,C/0001/00/00000000SA/01/P/01,FR,Cotecna Inspection v Commission,2000-12-14,2001-05-29,2000-12-14,2001-05-29,0,C/0001/00/00000000SA/01/P/01-616472,616472,id_46421,NaN,C/0001/00/00000000SA/01/P/01,C-1/00 SA,Ordonnanc

In [20]:
# ---------------------------------------------------------------------
# 2) Filtered relevant document metadata manifest
# ---------------------------------------------------------------------

if df_doc_meta.empty:
    df_relevant_docs = df_doc_meta.copy()
else:
    df_doc_meta["normalized_docTypeCode"] = df_doc_meta["normalized_docTypeCode"].map(normalize_doc_type)
    df_doc_meta["relevance_flag"] = df_doc_meta["normalized_docTypeCode"].map(is_relevant_doc_type)
    df_doc_meta["relevance_category"] = df_doc_meta["relevance_flag"].map({True: "relevant", False: "irrelevant"})
    df_doc_meta["relevance_reason"] = df_doc_meta["normalized_docTypeCode"].map(relevance_reason)
    df_doc_meta["relevant_doc_type_tokens"] = df_doc_meta["normalized_docTypeCode"].map(lambda x: "|".join(relevant_doc_type_tokens(x)))

    df_relevant_docs = df_doc_meta[df_doc_meta["relevance_flag"]].copy().reset_index(drop=True)

print("Relevant document rows:", len(df_relevant_docs))
if not df_relevant_docs.empty:
    display(df_relevant_docs[[
        c for c in ["procedure_key", "procedure_key_base", "infocuria_slash_procedure_key", "infocuria_slash_procedure_key_base", "publishedId", "logicDocId", "docType", "docTypeCode", "docDate", "available_languages", "docFormats", "relevance_reason"]
        if c in df_relevant_docs.columns
    ]].head(20))


Relevant document rows: 77085


,procedure_key,procedure_key_base,infocuria_slash_procedure_key,infocuria_slash_procedure_key_base,publishedId,logicDocId,docType,docTypeCode,docDate,available_languages,docFormats,relevance_reason
0,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,id_85906,Arrêt (Sommaire),ARRET_SOM,2001-12-13,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""PDF""]",relevant_summary_judgment_variant
1,2000/0001/C//,2000/0001/C/,2000/0001/C//,2000/0001/C/,C-1/00,id_46950,Arrêt,ARRET,2001-12-13,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML"", ""PDF""]",relevant_final_judgment
2,2000/0001/C/SA/,2000/0001/C/,2000/0001/C/SA/,2000/0001/C/,C-1/00 SA,id_46421,Ordonnance,ORD,2001-05-29,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML"", ""PDF""]",relevant_final_order
3,2000/0001/C/SA/,2000/0001/C/,2000/0001/C/SA/,2000/0001/C/,C-1/00 SA,id_85947,Ordonnance (Sommaire),ORD_SOM,2001-05-29,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""PDF""]",relevant_summary_order_variant
4,2001/0001/C/P/,2001/0001/C/,2001/0001/C/P/,2001/0001/C/,C-1/01 P,id_85572,Ordonnance (Sommaire),ORD_SOM,2001-09-20,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""PDF""]",relevant_summary_order_variant
5,2001/0001/C/P/,2001/0001/C/,2001/0001/C/P/,2001/0001/C/,C-1/01 P,id_46652,Ordonnance,ORD,2001-09-20,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML"", ""PDF""]",relevant_final_order
6,2002/0001/C//,2002/0001/C/,2002/0001/C//,2002/0001/C/,C-1/02,id_49061,Arrêt,ARRET,2004-04-01,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML"", ""PDF""]",relevant_final_judgment
7,2002/0001/C//,2002/0001/C/,2002/0001/C//,2002/0001/C/,C-1/02,id_84842,Ordonnance (JO),ORD_COMM,2003-08-22,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML""]",relevant_order_communication_variant
8,2002/0001/C//,2002/0001/C/,2002/0001/C//,2002/0001/C/,C-1/02,id_54478,Ordonnance (JO),ORD_COMM,2003-09-06,"[""FR""]","[""HTML""]",relevant_order_communication_variant
9,2002/0001/C//,2002/0001/C/,2002/0001/C//,2002/0001/C/,C-1/02,id_55247,Arrêt (Sommaire),ARRET_SOM,2004-04-01,"[""DA"", ""DE"", ""EL"", ""EN"", ""ES"", ""FI"", ""FR"", ""IT"", ""NL"", ""PT"", ""SV""]","[""HTML"", ""PDF""]",relevant_summary_judgment_variant


In [21]:

# ---------------------------------------------------------------------
# 3) Case-level document availability aggregation and case metadata join
# ---------------------------------------------------------------------

def bool_series(s):
    return s.astype(str).str.lower().isin(["true", "1", "yes"])

def normalized_date_join(values, sep="|"):
    vals = []
    for v in values:
        s = clean_str(v)
        if not s or s.lower() in {"nan", "none", "<na>"}:
            continue
        try:
            dt = pd.to_datetime(s, errors="coerce", dayfirst=False)
            vals.append(dt.strftime("%Y-%m-%d") if pd.notna(dt) else s)
        except Exception:
            vals.append(s)
    return sep.join(sorted(set(vals)))

def best_join_key_cols(left, right):
    """
    Return the best shared identifier for joining two tables.

    procedureId is the canonical key. affId is only a fallback when
    procedureId is unavailable in either table.
    """
    for cols in [
        ["procedureId"],
        ["affId"],
        ["internal_key"],
        ["procedure_key"],
    ]:
        if all(c in left.columns and c in right.columns for c in cols):
            left_nonempty = left[cols[0]].fillna("").astype(str).str.strip().ne("").any()
            right_nonempty = right[cols[0]].fillna("").astype(str).str.strip().ne("").any()
            if left_nonempty and right_nonempty:
                return cols
    return []

if df_doc_meta.empty:
    doc_case_agg = pd.DataFrame(columns=[
        "procedureId", "affId", "internal_key", "procedure_key",
        "available_document_types", "relevant_document_types",
        "available_document_dates", "relevant_document_dates",
        "total_document_count", "relevant_document_count", "irrelevant_document_count",
        "case_relevance_flag",
    ])
else:
    df_doc_meta["relevance_flag"] = bool_series(df_doc_meta["relevance_flag"]) if "relevance_flag" in df_doc_meta.columns else False
    if (
        "procedureId" in df_doc_meta.columns
        and df_doc_meta["procedureId"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .any()
    ):
        group_col = "procedureId"
    elif (
        "affId" in df_doc_meta.columns
        and df_doc_meta["affId"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .any()
    ):
        group_col = "affId"
        print(
            "Warning: procedureId unavailable for document aggregation; "
            "falling back to affId."
        )
    else:
        group_col = next(
            (
                c
                for c in ["internal_key", "procedure_key"]
                if c in df_doc_meta.columns
            ),
            "procedure_key",
        )
    count_col = "document_dedup_key" if "document_dedup_key" in df_doc_meta.columns else "logicDocId"

    agg_kwargs = {
        "available_document_types": ("normalized_docTypeCode", unique_join),
        "total_document_count": (count_col, "count"),
        "relevant_document_count": ("relevance_flag", "sum"),
    }
    if "docDate" in df_doc_meta.columns:
        agg_kwargs["available_document_dates"] = ("docDate", normalized_date_join)

    doc_case_agg = df_doc_meta.groupby(group_col, dropna=False).agg(**agg_kwargs).reset_index()

    # Add representative identifiers for easier downstream joins/inspection.
    id_cols = [c for c in ["procedureId", "affId", "internal_key", "procedure_key", "procedure_key_base"] if c in df_doc_meta.columns and c != group_col]
    if id_cols:
        id_rep = df_doc_meta.groupby(group_col, dropna=False)[id_cols].agg(unique_join).reset_index()
        doc_case_agg = doc_case_agg.merge(id_rep, on=group_col, how="left")

    rel_doc = df_doc_meta[df_doc_meta["relevance_flag"]].copy()
    rel_types = rel_doc.groupby(group_col, dropna=False)["normalized_docTypeCode"].apply(unique_join).reset_index(name="relevant_document_types")
    doc_case_agg = doc_case_agg.merge(rel_types, on=group_col, how="left")

    if "docDate" in df_doc_meta.columns:
        rel_dates = rel_doc.groupby(group_col, dropna=False)["docDate"].apply(normalized_date_join).reset_index(name="relevant_document_dates")
        doc_case_agg = doc_case_agg.merge(rel_dates, on=group_col, how="left")

    for col in ["relevant_document_types", "available_document_dates", "relevant_document_dates"]:
        if col not in doc_case_agg.columns:
            doc_case_agg[col] = ""
        doc_case_agg[col] = doc_case_agg[col].fillna("")

    doc_case_agg["irrelevant_document_count"] = doc_case_agg["total_document_count"] - doc_case_agg["relevant_document_count"]
    doc_case_agg["case_relevance_flag"] = doc_case_agg["relevant_document_count"].gt(0)

# Start case manifest with full InfoCuria case skeleton, then left-join procedural metadata and document availability.
df_case_metadata_manifest = df_cases.copy()

# Join flattened procedural metadata by stable InfoCuria identifiers, not only by reconstructed keys.
metadata_join_keys = best_join_key_cols(df_case_metadata_manifest, df_case_flat)
if not df_case_flat.empty and metadata_join_keys:
    metadata_cols = [c for c in df_case_flat.columns if c not in df_case_metadata_manifest.columns or c in metadata_join_keys]
    df_case_metadata_manifest = df_case_metadata_manifest.merge(
        df_case_flat[metadata_cols].drop_duplicates(metadata_join_keys, keep="last"),
        on=metadata_join_keys,
        how="left",
        suffixes=("", "__metadata"),
    )
else:
    df_case_metadata_manifest["metadata_join_status"] = "no_raw_metadata_jsonl_records_or_no_join_key"

if "metadata_join_status" not in df_case_metadata_manifest.columns:
    joined_indicator_col = next((c for c in ["totalHits", "publishedAffId", "jurisdictionCode", "numDocs"] if c in df_case_metadata_manifest.columns), None)
    metadata_joined = df_case_metadata_manifest[joined_indicator_col].notna() if joined_indicator_col else pd.Series(False, index=df_case_metadata_manifest.index)
    df_case_metadata_manifest["metadata_join_status"] = metadata_joined.map({True: "metadata_joined", False: "no_metadata_join"})

# Join document aggregation by procedureId first. affId is only a fallback.
doc_join_keys = best_join_key_cols(df_case_metadata_manifest, doc_case_agg)
if not doc_case_agg.empty and doc_join_keys:
    doc_agg_cols = [c for c in doc_case_agg.columns if c not in df_case_metadata_manifest.columns or c in doc_join_keys]
    df_case_metadata_manifest = df_case_metadata_manifest.merge(doc_case_agg[doc_agg_cols], on=doc_join_keys, how="left")
else:
    print("Warning: no suitable key found for document aggregation join.")

for col in ["available_document_types", "relevant_document_types", "available_document_dates", "relevant_document_dates"]:
    if col not in df_case_metadata_manifest.columns:
        df_case_metadata_manifest[col] = ""
    df_case_metadata_manifest[col] = df_case_metadata_manifest[col].fillna("")

for col in ["total_document_count", "relevant_document_count", "irrelevant_document_count"]:
    if col not in df_case_metadata_manifest.columns:
        df_case_metadata_manifest[col] = 0
    df_case_metadata_manifest[col] = pd.to_numeric(df_case_metadata_manifest[col], errors="coerce").fillna(0).astype(int)

df_case_metadata_manifest["case_relevance_flag"] = df_case_metadata_manifest["relevant_document_count"].gt(0)
df_case_metadata_manifest = df_case_metadata_manifest.drop(columns=[c for c in DROP_NON_ENGLISH_METADATA_COLUMNS if c in df_case_metadata_manifest.columns], errors="ignore")

print("Case metadata manifest rows before legacy CURIA join:", len(df_case_metadata_manifest))
print("Metadata join keys used:", metadata_join_keys)
print("Document aggregation join keys used:", doc_join_keys)
display(df_case_metadata_manifest[[
    c for c in ["internal_key", "procedure_key", "procedure_key_base", "publishedId", "affId", "procedureId", "available_document_types", "relevant_document_types", "available_document_dates", "relevant_document_dates", "total_document_count", "relevant_document_count", "irrelevant_document_count", "case_relevance_flag"]
    if c in df_case_metadata_manifest.columns
]].head(20))


Case metadata manifest rows before legacy CURIA join: 56828
Metadata join keys used: ['procedureId']
Document aggregation join keys used: ['procedureId']


,internal_key,procedure_key,procedure_key_base,publishedId,affId,procedureId,available_document_types,relevant_document_types,available_document_dates,relevant_document_dates,total_document_count,relevant_document_count,irrelevant_document_count,case_relevance_flag
0,2000/0001/C//,2000/0001/C//,2000/0001/C/,C-1/00,C/0001/00/00000000RD/01,C/0001/00/00000000RD/01/P/01,ARRET|ARRET_SOM|ARR_COMM|AVI_COMM|CONCL,ARRET|ARRET_SOM,2001-09-20|2001-12-13|2002-02-16|2002-06-14,2001-12-13,5,2,3,True
1,2000/0001/C/SA/,2000/0001/C/SA/,2000/0001/C/,C-1/00 SA,C/0001/00/00000000SA/01,C/0001/00/00000000SA/01/P/01,ORD|ORD_SOM,ORD|ORD_SOM,2001-05-29,2001-05-29,2,2,0,True
2,2001/0001/C/P/,2001/0001/C/P/,2001/0001/C/,C-1/01 P,C/0001/01/00000000PV/01,C/0001/01/00000000PV/01/P/01,ORD|ORD_SOM,ORD|ORD_SOM,2001-09-20,2001-09-20,2,2,0,True
3,2002/0001/C//,2002/0001/C//,2002/0001/C/,C-1/02,C/0001/02/00000000RP/01,C/0001/02/00000000RP/01/P/01,ARRET|ARRET_SOM|ARR_COMM|CONCL|DDP_COMM|ORD_COMM,ARRET|ARRET_SOM|ORD_COMM,2002-03-02|2003-07-03|2003-08-22|2003-09-06|2004-04-01|2004-04-30,2003-08-22|2003-09-06|2004-04-01,7,4,3,True
4,2002/0001/C/SA/,2002/0001/C/SA/,2002/0001/C/,C-1/02 SA,C/0001/02/00000000SA/01,C/0001/02/00000000SA/01/P/01,ORD|ORD_SOM,ORD|ORD_SOM,2003-03-27,2003-03-27,2,2,0,True
5,2003/0001/C//,2003/0001/C//,2003/0001/C/,C-1/03,C/0001/03/00000000RP/01,C/0001/03/00000000RP/01/P/01,ARRET|ARRET_SOM|ARR_COMM|CONCL|DDP_COMM|RAD_COMM|REQ_COMM,ARRET|ARRET_SOM,2003-02-07|2003-04-04|2004-01-29|2004-04-16|2004-09-07|2004-10-23,2004-09-07,7,2,5,True
6,2003/0001/C/SA/,2003/0001/C/SA/,2003/0001/C/,C-1/03 SA,C/0001/03/00000000SA/01,C/0001/03/00000000SA/01/P/01,,,,,0,0,0,False
7,2004/0001/C//,2004/0001/C//,2004/0001/C/,C-1/04,C/0001/04/00000000RP/01,C/0001/04/00000000RP/01/P/01,ARRET|ARRET_SOM|AVI_COMM|CONCL|DDP_COMM|ORD_COMM|RAD_COMM|REQ_COMM,ARRET|ARRET_SOM|ORD_COMM,2004-03-05|2004-04-26|2005-03-04|2005-03-19|2005-09-06|2006-01-17,2005-03-04|2005-03-19|2006-01-17,9,4,5,True
8,2004/0001/C/SA/,2004/0001/C/SA/,2004/0001/C/,C-1/04 SA,C/0001/04/00000000SA/01,C/0001/04/00000000SA/01/P/01,ORD|ORD_SOM,ORD|ORD_SOM,2004-12-14,2004-12-14,2,2,0,True
9,2005/0001/C//,2005/0001/C//,2005/0001/C/,C-1/05,C/0001/05/00000000RP/01,C/0001/05/00000000RP/01/P/01,ARRET|ARRET_SOM|CONCL|DDP_COMM,ARRET|ARRET_SOM,2005-03-05|2006-04-27|2007-01-09,2007-01-09,4,2,2,True


In [22]:
# ---------------------------------------------------------------------
# 4) Join legacy CURIA metadata onto the case manifest
# ---------------------------------------------------------------------

def normalize_date_value(x):
    """Return YYYY-MM-DD for common CURIA/InfoCuria date strings, or ''.

    Important:
    - CURIA legacy dates are commonly DD.MM.YYYY, e.g. 04.02.1959 = 1959-02-04.
    - InfoCuria closeDate/procClosDate are commonly ISO YYYY-MM-DD.
    The parser therefore handles obvious formats explicitly before falling back to pandas.
    """
    s = clean_str(x)
    if not s or s.lower() in {"nan", "none", "<na>", "nat"}:
        return ""

    s = s.strip()

    # ISO-like: YYYY-MM-DD, YYYY/MM/DD, YYYY.MM.DD
    m_iso = re.match(r"^(\d{4})[-/.](\d{1,2})[-/.](\d{1,2})$", s)
    if m_iso:
        y, m, d = map(int, m_iso.groups())
        try:
            return pd.Timestamp(year=y, month=m, day=d).strftime("%Y-%m-%d")
        except Exception:
            return ""

    # European legacy CURIA style: DD.MM.YYYY or DD/MM/YYYY or DD-MM-YYYY
    m_eu = re.match(r"^(\d{1,2})[-/.](\d{1,2})[-/.](\d{4})$", s)
    if m_eu:
        d, m, y = map(int, m_eu.groups())
        try:
            return pd.Timestamp(year=y, month=m, day=d).strftime("%Y-%m-%d")
        except Exception:
            return ""

    # Textual or unusual date strings. Try day-first first because CURIA is European.
    for dayfirst in [True, False]:
        try:
            dt = pd.to_datetime(s, errors="coerce", dayfirst=dayfirst)
            if pd.notna(dt):
                return dt.strftime("%Y-%m-%d")
        except Exception:
            pass
    return ""

def split_joined_values(x):
    s = clean_str(x)
    if not s:
        return []
    # Handles pipe cells and JSON-list-looking cells.
    if s.startswith("["):
        try:
            obj = json.loads(s)
            if isinstance(obj, list):
                return [clean_str(v) for v in obj if clean_str(v)]
        except Exception:
            pass
    return [clean_str(v) for v in re.split(r"[|;]", s) if clean_str(v)]

def normalized_date_set_from_values(values):
    out = set()
    if isinstance(values, (str, int, float)) or values is None:
        values = [values]
    for v in values:
        for part in split_joined_values(v):
            nd = normalize_date_value(part)
            if nd:
                out.add(nd)
    return out

def row_date_set(row, candidate_cols):
    vals = []
    for col in candidate_cols:
        if col in row.index:
            vals.extend(split_joined_values(row.get(col)))
    return normalized_date_set_from_values(vals)

if not df_curia_keyed.empty:
    df_curia_keyed["curia_row_count"] = 1

    if "document_type" in df_curia_keyed.columns:
        df_curia_keyed["curia_normalized_document_type_row"] = df_curia_keyed["document_type"].map(normalize_doc_type)
    else:
        df_curia_keyed["curia_normalized_document_type_row"] = ""

    # Matching rule:
    #   CURIA has only the base key: YYYY/NNNN/PREFIX/
    #   InfoCuria may have full slash keys: YYYY/NNNN/PREFIX/ and YYYY/NNNN/PREFIX/SUFFIX
    #   - if one InfoCuria candidate exists for the base, match it
    #   - if multiple candidates exist, match CURIA date to InfoCuria closeDate/procClosDate
    #   - if no date is available on either side, match all candidates for that base
    #   - otherwise leave the CURIA base as failed/no-date-match diagnostic
    info_match_cols = [
        "procedure_key", "procedure_key_base",
        "infocuria_slash_procedure_key", "infocuria_slash_procedure_key_base",
        "publishedId", "case_number_raw", "case_number_clean",
        "closeDate", "procClosDate",
    ]
    df_info_candidates = df_case_metadata_manifest[[c for c in info_match_cols if c in df_case_metadata_manifest.columns]].copy()

    # Use the explicit InfoCuria slash key columns for matching when present.
    # Fall back to procedure_key/procedure_key_base for backwards compatibility.
    if "infocuria_slash_procedure_key" not in df_info_candidates.columns:
        df_info_candidates["infocuria_slash_procedure_key"] = df_info_candidates.get("procedure_key", "")
    if "infocuria_slash_procedure_key_base" not in df_info_candidates.columns:
        df_info_candidates["infocuria_slash_procedure_key_base"] = df_info_candidates.get("procedure_key_base", "")

    df_info_candidates["match_procedure_key"] = df_info_candidates["infocuria_slash_procedure_key"].fillna("").astype(str)
    df_info_candidates["match_procedure_key_base"] = df_info_candidates["infocuria_slash_procedure_key_base"].fillna("").astype(str)
    df_info_candidates.loc[df_info_candidates["match_procedure_key"].eq(""), "match_procedure_key"] = df_info_candidates.get("procedure_key", "")
    df_info_candidates.loc[df_info_candidates["match_procedure_key_base"].eq(""), "match_procedure_key_base"] = df_info_candidates.get("procedure_key_base", "")

    df_info_candidates = df_info_candidates[df_info_candidates["match_procedure_key_base"].fillna("").astype(str).ne("")].copy()

    # Date disambiguation is deliberately only CURIA date vs InfoCuria closeDate/procClosDate.
    info_date_cols = ["closeDate", "procClosDate"]
    df_info_candidates["infocuria_match_dates"] = df_info_candidates.apply(lambda r: "|".join(sorted(row_date_set(r, info_date_cols))), axis=1)

    # Aggregate legacy CURIA rows to base-key level.
    agg_spec = {
        "curia_row_count": ("curia_row_count", "sum"),
        "curia_document_types": ("document_type", unique_join) if "document_type" in df_curia_keyed.columns else ("curia_normalized_document_type_row", unique_join),
        "curia_normalized_document_types": ("curia_normalized_document_type_row", unique_join),
    }

    optional_curia_cols = [
        "source", "status", "date", "case_description", "see_case_number", "see_case_number_clean",
        "contains_joined_cases", "force_string", "document_url", "source_url", "celex_from_case_number_link",
        "celex_from_description_link", "ecli",
    ]

    for col in optional_curia_cols:
        if col in df_curia_keyed.columns:
            agg_spec[f"curia_{col}s"] = (col, unique_join)

    curia_base_agg = df_curia_keyed.groupby("procedure_key_base", dropna=False).agg(**agg_spec).reset_index()
    curia_base_agg = curia_base_agg[curia_base_agg["procedure_key_base"].fillna("").astype(str).ne("")].copy()
    curia_base_agg["curia_match_dates"] = curia_base_agg["curia_dates"].map(lambda x: "|".join(sorted(normalized_date_set_from_values(x)))) if "curia_dates" in curia_base_agg.columns else ""

    info_groups = {
        base: grp.copy()
        for base, grp in df_info_candidates.groupby("match_procedure_key_base", dropna=False)
    }

    expanded_rows = []
    failed_rows = []

    for _, curia_row in curia_base_agg.iterrows():
        base = clean_str(curia_row.get("procedure_key_base"))
        candidates = info_groups.get(base, pd.DataFrame())
        candidate_count = len(candidates)
        candidate_keys = candidates["match_procedure_key"].dropna().astype(str).tolist() if not candidates.empty else []
        candidate_date_map = ""
        if not candidates.empty:
            candidate_date_map = "|".join(
                sorted(
                    set(
                        f"{clean_str(r.get('match_procedure_key'))}:{clean_str(r.get('infocuria_match_dates'))}"
                        for _, r in candidates.iterrows()
                    )
                )
            )
        curia_dates = set(split_joined_values(curia_row.get("curia_match_dates")))

        matched_candidates = pd.DataFrame()
        method = ""

        if candidate_count == 0:
            method = "no_infocuria_base_candidate"
        elif candidate_count == 1:
            matched_candidates = candidates
            method = "base_unique"
        else:
            candidate_dates_any = False
            exact_mask = []
            for _, cand in candidates.iterrows():
                cand_dates = set(split_joined_values(cand.get("infocuria_match_dates")))
                candidate_dates_any = candidate_dates_any or bool(cand_dates)
                exact_mask.append(bool(curia_dates and cand_dates and curia_dates.intersection(cand_dates)))

            if any(exact_mask):
                matched_candidates = candidates.loc[exact_mask].copy()
                method = "base_multiple_date_exact"
            elif not curia_dates or not candidate_dates_any:
                matched_candidates = candidates
                method = "base_multiple_no_date_match_all"
            else:
                method = "base_multiple_no_exact_date_match"

        if not matched_candidates.empty:
            for _, cand in matched_candidates.iterrows():
                row = curia_row.to_dict()
                row["procedure_key"] = cand.get("procedure_key")
                row["matched_infocuria_slash_procedure_key"] = first_nonempty(cand.get("infocuria_slash_procedure_key"), cand.get("match_procedure_key"))
                row["matched_infocuria_slash_procedure_key_base"] = first_nonempty(cand.get("infocuria_slash_procedure_key_base"), cand.get("match_procedure_key_base"))
                row["curia_join_status"] = "matched"
                row["curia_match_method"] = method
                row["curia_candidate_count_for_base"] = candidate_count
                row["curia_candidate_procedure_keys"] = "|".join(sorted(set(candidate_keys)))
                row["curia_candidate_date_map"] = candidate_date_map
                row["infocuria_match_dates"] = cand.get("infocuria_match_dates", "")
                expanded_rows.append(row)
        else:
            row = curia_row.to_dict()
            row["procedure_key"] = ""
            row["matched_infocuria_slash_procedure_key"] = ""
            row["matched_infocuria_slash_procedure_key_base"] = ""
            row["curia_join_status"] = method
            row["curia_match_method"] = method
            row["curia_candidate_count_for_base"] = candidate_count
            row["curia_candidate_procedure_keys"] = "|".join(sorted(set(candidate_keys)))
            row["curia_candidate_date_map"] = candidate_date_map
            row["infocuria_match_dates"] = ""
            failed_rows.append(row)

    curia_proc_expanded = pd.DataFrame(expanded_rows)

    if not curia_proc_expanded.empty:
        # If several CURIA base rows ended up on the same InfoCuria procedure, collapse them again.
        passthrough_cols = [
            c for c in curia_proc_expanded.columns
            if c not in {"procedure_key", "curia_row_count", "curia_candidate_count_for_base"}
        ]
        collapse_spec = {
            "curia_row_count": ("curia_row_count", "sum"),
            "curia_candidate_count_for_base": ("curia_candidate_count_for_base", "max"),
        }
        for col in passthrough_cols:
            collapse_spec[col] = (col, unique_join)

        curia_proc_agg = curia_proc_expanded.groupby("procedure_key", dropna=False).agg(**collapse_spec).reset_index()
        df_case_metadata_manifest = df_case_metadata_manifest.merge(curia_proc_agg, on="procedure_key", how="left")
    else:
        df_case_metadata_manifest["curia_join_status"] = pd.NA
        df_case_metadata_manifest["curia_row_count"] = 0

    df_case_metadata_manifest["curia_join_status"] = df_case_metadata_manifest["curia_join_status"].fillna("no_curia_match")
    if "curia_row_count" not in df_case_metadata_manifest.columns:
        df_case_metadata_manifest["curia_row_count"] = 0
    df_case_metadata_manifest["curia_row_count"] = pd.to_numeric(df_case_metadata_manifest["curia_row_count"], errors="coerce").fillna(0).astype(int)

    # Failed join diagnostic: CURIA base keys that were not matched to any InfoCuria procedure under the new logic.
    df_curia_failed_join = pd.DataFrame(failed_rows)
    if not df_curia_failed_join.empty:
        # Attach representative raw CURIA rows for debugging.
        debug_cols = [
            c for c in [
                "procedure_key", "procedure_key_base", "case_number_raw", "case_number_clean",
                "case_prefix", "case_number_int_full", "case_year_full", "case_suffix",
                "document_type", "date", "status", "source", "source_url", "document_url",
                "see_case_number", "see_case_number_clean", "contains_joined_cases", "force_string",
                "case_description", "celex_from_case_number_link", "celex_from_description_link", "ecli"
            ] if c in df_curia_keyed.columns
        ]
        raw_debug = df_curia_keyed[debug_cols].drop_duplicates() if debug_cols else pd.DataFrame()
        if not raw_debug.empty:
            raw_debug = raw_debug.rename(columns={"procedure_key": "curia_original_procedure_key"})
            df_curia_failed_join = df_curia_failed_join.merge(raw_debug, on="procedure_key_base", how="left")

        failed_keep_cols = [
            c for c in [
                "procedure_key_base", "curia_join_status", "curia_match_method",
                "matched_infocuria_slash_procedure_key", "matched_infocuria_slash_procedure_key_base",
                "curia_candidate_count_for_base", "curia_candidate_procedure_keys", "curia_candidate_date_map",
                "curia_match_dates", "infocuria_match_dates",
                "curia_original_procedure_key", "case_number_raw", "case_number_clean",
                "case_prefix", "case_number_int_full", "case_year_full", "case_suffix",
                "document_type", "date", "status", "source", "source_url", "document_url",
                "see_case_number", "see_case_number_clean", "contains_joined_cases", "force_string",
                "case_description", "celex_from_case_number_link", "celex_from_description_link", "ecli",
                "curia_document_types", "curia_normalized_document_types", "curia_dates", "curia_sources",
                "curia_statuss", "curia_source_urls", "curia_document_urls", "curia_eclis",
            ] if c in df_curia_failed_join.columns
        ]
        df_curia_failed_join = df_curia_failed_join[failed_keep_cols].drop_duplicates().reset_index(drop=True)
    else:
        df_curia_failed_join = pd.DataFrame(columns=[
            "procedure_key_base", "curia_join_status", "curia_match_method",
            "matched_infocuria_slash_procedure_key", "matched_infocuria_slash_procedure_key_base",
            "curia_candidate_count_for_base", "curia_candidate_procedure_keys", "curia_candidate_date_map",
            "curia_match_dates", "infocuria_match_dates",
        ])

else:
    df_case_metadata_manifest["curia_join_status"] = "curia_manifest_not_provided"
    df_curia_failed_join = pd.DataFrame()

print("CURIA join status counts:")
display(df_case_metadata_manifest["curia_join_status"].value_counts(dropna=False).to_frame("count"))

if "curia_match_method" in df_case_metadata_manifest.columns:
    print("CURIA match method counts:")
    display(df_case_metadata_manifest["curia_match_method"].fillna("no_curia_match").value_counts(dropna=False).to_frame("count"))

print("CURIA failed join rows:", len(df_curia_failed_join))
if not df_curia_failed_join.empty:
    display(df_curia_failed_join.head(20))

CURIA join status counts:


,count
curia_join_status,
matched,53482
no_curia_match,3346


CURIA match method counts:


,count
curia_match_method,
base_unique,41625
base_multiple_date_exact,10457
no_curia_match,3346
base_multiple_no_date_match_all,1400


CURIA failed join rows: 135


,procedure_key_base,curia_join_status,curia_match_method,matched_infocuria_slash_procedure_key,matched_infocuria_slash_procedure_key_base,curia_candidate_count_for_base,curia_candidate_procedure_keys,curia_candidate_date_map,curia_match_dates,infocuria_match_dates,curia_original_procedure_key,case_number_raw,case_number_clean,case_prefix,case_number_int_full,case_year_full,case_suffix,document_type,date,status,source,source_url,see_case_number_clean,case_description,celex_from_case_number_link,celex_from_description_link,ecli,curia_document_types,curia_normalized_document_types,curia_dates,curia_sources,curia_statuss,curia_source_urls,curia_eclis
0,1955/0003/C/,no_infocuria_base_candidate,no_infocuria_base_candidate,,,0,,,1955-03-15,,1955/0003/C//,3/55,3/55,C,0003,1955,,removed_from_register,15.03.1955,removed,c1_juris,https://curia.europa.eu/en/content/juris/c1_juris.htm,NaN,"Removed from the register on 15 March 1955, Luxembourg / ECSC High Authority (3/55) ECLI:EU:C:1955:4",NaN,NaN,ECLI:EU:C:1955:4,removed_from_register,REMOVED_FROM_REGISTER,15.03.1955,c1_juris,removed,https://curia.europa.eu/en/content/juris/c1_juris.htm,ECLI:EU:C:1955:4
1,1955/0007/C/,no_infocuria_base_candidate,no_infocuria_base_candidate,,,0,,,,,1955/0007/C//,7/55,7/55,C,0007,1955,,removed_from_register,NaN,removed,c1_juris,https://curia.europa.eu/en/content/juris/c1_juris.htm,NaN,"Removed from the register , Luxembourg / ECSC High Authority (7/55)",NaN,NaN,NaN,removed_from_register,REMOVED_FROM_REGISTER,,c1_juris,removed,https://curia.europa.eu/en/content/juris/c1_juris.htm,
2,1959/0002/C/,no_infocuria_base_candidate,no_infocuria_base_candidate,,,0,,,1959-01-30,,1959/0002/C//,2/59,2/59,C,0002,1959,,order,30.01.1959,downloadable,c1_juris,https://curia.europa.eu/en/content/juris/c1_juris.htm,NaN,"Order of 30 January 1959, Mannesmann / ECSC High Authority (2-59 and 4-59 to 13-59, ECR 1960 p. 345) (FR1960/00345 NL1960/00353 DE1960/00351 IT1960/00335 EN1960/00162) ECLI:EU:C:1959:2",61959CO0002,NaN,ECLI:EU:C:1959:2,order,ORDER,30.01.1959,c1_juris,downloadable,https://curia.europa.eu/en/content/juris/c1_juris.htm,ECLI:EU:C:1959:2
3,1959/0019/C/,no_infocuria_base_candidate,no_infocuria_base_candidate,,,0,,,1959-05-12,,1959/0019/C//,19/59,19/59,C,0019,1959,,order,12.05.1959,downloadable,c1_juris,https://curia.europa.eu/en/content/juris/c1_juris.htm,NaN,"Order of 12 May 1959, Geitling Ruhrkohlen-Verkaufsgesellschaft and others / ECSC High Authority (16-59, 17-59, 18-59 and 19-59, ECR 1960 p. 85) (FR1960/00085 NL1960/00085 DE1960/00087 IT1960/00081 EN1960/00034 DA1954-1964/00165 EL1954-1964/00369 PT1954-1961/00319) ECLI:EU:C:1959:8",61959CO0019,NaN,ECLI:EU:C:1959:8,order,ORDER,12.05.1959,c1_juris,downloadable,https://curia.europa.eu/en/content/juris/c1_juris.htm,ECLI:EU:C:1959:8
4,1959/0021/C/,no_infocuria_base_candidate,no_infocuria_base_candidate,,,0,,,1959-05-20,,1959/0021/C//,21/59,21/59,C,0021,1959,,order,20.05.1959,downloadable,c1_juris,https://curia.europa.eu/en/content/juris/c1_juris.htm,NaN,"Order of 20 May 1959, Italy / ECSC High Authority (21/59 R, ECR 1960 p. 351) (FR1960/00717 NL1960/00739 DE1960/00737 IT1960/00691 EN1960/00351) ECLI:EU:C:1959:9",61959CO0021,NaN,ECLI:EU:C:1959:9,order,ORDER,20.05.1959,c1_juris,downloadable,https://curia.europa.eu/en/content/juris/c1_juris.htm,ECLI:EU:C:1959:9
5,1987/0150/C/,base_multiple_no_exact_date_match,base_multiple_no_exact_date_match,,,2,1987/0150/C//,1987/0150/C//:1989-02-01|1990-03-14|1987/0150/C//:1990-03-14,1987-11-11,,1987/0150/C//,150/87,150/87,C,0150,1987,,order,11.11.1987,downloadable,c1_juris,https://curia.europa.eu/en/content/juris/c1_juris.htm,NaN,"Order of 11 November 1987, Nashua Corporation / Council and Commission (150/87, ECR 1987 p. 4421) ECLI:EU:C:1987:484",61987CO0150,NaN,ECLI:EU:C:1987:484,order|see_case_reference,ORDER|SEE_CASE_REFERENCE,11.11.1987,c1_juris,downloadable|joined_case,https://curia.europa.eu/en/content/juris/c1_juris.htm,ECLI:EU:C:1987:484
6,1987/0150/C/,base_multiple_no_exac

In [23]:

# ---------------------------------------------------------------------
# 5) Download URL and target-path builders
# ---------------------------------------------------------------------

# v4 keeps the download manifest compact. Candidate URLs are generated in memory
# when downloading, but the final manifest stores only attempted URLs, the selected
# first URL, and the final working URL/status/path.

def parse_json_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            obj = json.loads(x)
            return obj if isinstance(obj, list) else []
        except Exception:
            return []
    return []


def ordered_unique(values):
    out = []
    seen = set()
    for v in values:
        s = clean_str(v)
        if s and s not in seen:
            seen.add(s)
            out.append(s)
    return out


def available_languages_for_row(row):
    langs = [str(x).upper() for x in parse_json_list(row.get("available_languages")) if x]
    proc_lang = clean_str(row.get("procLang")).upper()
    ordered = []
    for lang in PREFERRED_LANGUAGES:
        if lang.upper() in langs:
            ordered.append(lang.upper())
    if proc_lang and proc_lang in langs:
        ordered.append(proc_lang)
    ordered.extend(langs)
    return ordered_unique(ordered)


def available_formats_for_row(row):
    fmts = [str(x).upper() for x in parse_json_list(row.get("docFormats")) if x]
    if not fmts and clean_str(row.get("docFormat")):
        fmts = normalize_doc_formats(row.get("docFormat"))
    ordered = []
    for fmt in PREFERRED_FORMATS:
        if fmt.upper() in fmts:
            ordered.append(fmt.upper())
    ordered.extend(fmts)
    return ordered_unique(ordered)


def language_format_pairs_for_row(row):
    langs = available_languages_for_row(row) or ["EN"]
    fmts = available_formats_for_row(row) or ["HTML", "PDF"]
    pairs = []
    for lang in langs:
        for fmt in fmts:
            pairs.append((lang, fmt))
    return pairs


def year_from_date_value(value):
    s = clean_str(value)
    if not s:
        return ""
    m = re.search(r"(19|20)\d{2}", s)
    if m:
        return m.group(0)
    for dayfirst in [False, True]:
        try:
            dt = pd.to_datetime(s, errors="coerce", dayfirst=dayfirst)
            if pd.notna(dt):
                return f"{int(dt.year):04d}"
        except Exception:
            pass
    return ""


def parse_routing_parts(routing):
    parts = str(routing or "").strip().split("/")
    if len(parts) < 3:
        return None
    court, number, two_digit_year, *rest = parts
    if not court or not number or not two_digit_year:
        return None
    return court, number, two_digit_year, rest


def full_year_from_two_digit(two_digit_year):
    try:
        y = int(str(two_digit_year))
        return "19" + f"{y:02d}" if y >= 50 else "20" + f"{y:02d}"
    except Exception:
        return ""


def routing_case_year(routing):
    parsed = parse_routing_parts(routing)
    return full_year_from_two_digit(parsed[2]) if parsed else ""


def candidate_years_for_row(row):
    routing = first_nonempty(row.get("idProcedure"), row.get("procedureId"))
    base_years = []
    for y in [
        routing_case_year(routing),
        year_from_date_value(row.get("docDate")),
        year_from_date_value(row.get("closeDate")),
        year_from_date_value(row.get("procClosDate")),
        year_from_date_value(row.get("introductionDate")),
        year_from_date_value(row.get("procIntroDate")),
        clean_str(row.get("case_year_full")),
    ]:
        if re.fullmatch(r"(19|20)\d{2}", clean_str(y)):
            base_years.append(int(y))
    if not base_years:
        return []
    lo, hi = min(base_years), max(base_years)
    years = list(range(lo - 2, hi + 3))
    ordered = [int(y) for y in base_years] + years
    return [str(y) for y in ordered_unique([str(y) for y in ordered if 1950 <= int(y) <= 2035])]


def routing_to_blob_path(routing, override_year=None):
    parsed = parse_routing_parts(routing)
    if not parsed:
        return None
    court, number, two_digit_year, rest = parsed
    full_year = clean_str(override_year) or full_year_from_two_digit(two_digit_year)
    if not full_year:
        return None
    filename_part = f"{court}_{number}_{two_digit_year}_" + "_".join(rest)
    return f"{court}/{full_year}/{filename_part}"


def build_download_url(row, lang, fmt, override_year=None):
    logic_doc_id = clean_str(row.get("logicDocId"))
    routing = first_nonempty(row.get("idProcedure"), row.get("procedureId"))
    if not logic_doc_id or not routing or not lang or not fmt:
        return ""
    blob_path = routing_to_blob_path(routing, override_year=override_year)
    if not blob_path:
        return ""
    logic_num = logic_doc_id.replace("id_", "")
    fmt = str(fmt).upper()
    lang = str(lang).upper()
    if fmt == "HTML":
        return f"https://infocuriaws.curia.europa.eu/blob/download-file-html/{blob_path}/{logic_num}-{lang}-1.html"
    if fmt == "PDF":
        return f"https://infocuriaws.curia.europa.eu/blob/download-file/{blob_path}/{logic_num}-{lang}-1.pdf"
    return ""


def safe_filename(value, fallback="document"):
    value = "" if value is None else str(value)
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("_")
    return value or fallback


def filename_from_url(url, fallback):
    try:
        parsed = urlparse(str(url))
        name = Path(parsed.path).name
        return safe_filename(name, fallback=fallback)
    except Exception:
        return safe_filename(fallback)


def primary_target_dir_for_format(fmt):
    fmt = str(fmt or "").upper()
    if fmt == "HTML":
        return COURT_INFOCURIA_HTML_DIR
    if fmt == "PDF":
        return COURT_INFOCURIA_PDFS_DIR
    return COURT_INFOCURIA_RAW_DIR / "other"


def candidate_existing_dirs_for_format(fmt):
    fmt = str(fmt or "").upper()
    if fmt == "HTML":
        return COURT_INFOCURIA_HTML_EXISTING_DIRS
    if fmt == "PDF":
        return COURT_INFOCURIA_PDF_EXISTING_DIRS
    return [COURT_INFOCURIA_RAW_DIR / "other"]


def find_existing_download_path(fmt, filename):
    for folder in candidate_existing_dirs_for_format(fmt):
        candidate = Path(folder) / filename
        if candidate.exists():
            return candidate.resolve()
    return None


def build_candidate_download_records(row):
    records = []
    years = candidate_years_for_row(row) or [None]
    for lang, fmt in language_format_pairs_for_row(row):
        for year in years:
            url = build_download_url(row, lang, fmt, override_year=year)
            if not url:
                continue
            fallback_base = safe_filename("_".join([
                clean_str(row.get("procedure_key")),
                first_nonempty(row.get("logicDocId"), row.get("docId")),
                clean_str(row.get("normalized_docTypeCode")),
                clean_str(lang),
                clean_str(year),
            ]), fallback="infocuria_document")
            ext = "html" if fmt == "HTML" else "pdf" if fmt == "PDF" else "bin"
            filename = filename_from_url(url, fallback=f"{fallback_base}.{ext}")
            target_dir = primary_target_dir_for_format(fmt)
            target_dir.mkdir(parents=True, exist_ok=True)
            intended_local_path = target_dir / filename
            existing_path = find_existing_download_path(fmt, filename)
            records.append({
                "url": url,
                "language": lang,
                "format": fmt,
                "year": year,
                "filename": filename,
                "local_path": str(existing_path if existing_path is not None else intended_local_path),
                "intended_local_path": str(intended_local_path),
                "existing_local_path": str(existing_path) if existing_path is not None else "",
                "file_exists_before_download": bool(existing_path is not None or intended_local_path.exists()),
            })
    out = []
    seen = set()
    for r in records:
        if r["url"] not in seen:
            seen.add(r["url"])
            out.append(r)
    return out


def build_download_manifest(relevant_docs):
    """Build one download target row per relevant document.

    v7 guarantee: every row in df_relevant_docs, including ORD_COMM and
    combined codes such as ORD_COMM|REQ_COMM, receives a download-manifest row.
    Candidate URL lists are generated only in memory and are not exported.
    """
    rows = []
    for _, row in relevant_docs.iterrows():
        candidate_records = build_candidate_download_records(row)
        first = candidate_records[0] if candidate_records else {}
        already_exists = bool(first.get("file_exists_before_download", False))
        rows.append({
            "internal_key": row.get("internal_key"),
            "procedure_key": row.get("procedure_key"),
            "procedure_key_base": row.get("procedure_key_base"),
            "case_number_raw": row.get("case_number_raw"),
            "case_number_clean": row.get("case_number_clean"),
            "affId": row.get("affId"),
            "procedureId": row.get("procedureId"),
            "publishedId": row.get("publishedId"),
            "docId": row.get("docId"),
            "logicDocId": row.get("logicDocId"),
            "jpLogicDocId": row.get("jpLogicDocId"),
            "idProcedure": row.get("idProcedure"),
            "idPublished": row.get("idPublished"),
            "docType": row.get("docType"),
            "docTypeCode": row.get("docTypeCode"),
            "normalized_docTypeCode": row.get("normalized_docTypeCode"),
            "docDate": row.get("docDate"),
            "available_languages": row.get("available_languages"),
            "docFormats": row.get("docFormats"),
            "candidate_years": json.dumps(candidate_years_for_row(row), ensure_ascii=False),
            "download_url": first.get("url", ""),
            "selected_language": first.get("language", ""),
            "selected_format": first.get("format", ""),
            "selected_year": first.get("year", ""),
            "target_filename": first.get("filename", ""),
            "intended_local_path": first.get("intended_local_path", ""),
            "existing_local_path": first.get("existing_local_path", ""),
            "final_local_path": first.get("local_path", "") if already_exists else pd.NA,
            "file_present_on_disk": already_exists,
            "file_exists_before_download": already_exists,
            "download_success": already_exists,
            "download_status": "already_exists" if already_exists else "pending",
            "attempted_urls": json.dumps([], ensure_ascii=False),
            "attempted_http_statuses": json.dumps([], ensure_ascii=False),
            "attempt_count": 0,
            "working_url": first.get("url", "") if already_exists else pd.NA,
            "working_language": first.get("language", "") if already_exists else pd.NA,
            "working_format": first.get("format", "") if already_exists else pd.NA,
            "working_year": first.get("year", "") if already_exists else pd.NA,
            "http_status": pd.NA,
            "error": pd.NA,
            "bytes": pd.NA,
        })
    df = pd.DataFrame(rows)
    return load_previous_download_status_if_available(df)


def load_previous_download_status_if_available(df):
    if df.empty or not DOWNLOAD_MANIFEST_JSONL.exists():
        return df
    try:
        prev = read_manifest(DOWNLOAD_MANIFEST_JSONL)
    except Exception as e:
        print(f"Could not load previous download manifest for status merge: {e}")
        return df
    key_cols = [c for c in ["logicDocId", "idProcedure"] if c in df.columns and c in prev.columns]
    if not key_cols:
        key_cols = [c for c in ["docId", "procedureId"] if c in df.columns and c in prev.columns]
    status_cols = [c for c in [
        "download_success", "download_status", "http_status", "error", "bytes", "working_url",
        "working_language", "working_format", "working_year", "final_local_path",
        "attempted_urls", "attempted_http_statuses", "attempt_count"
    ] if c in prev.columns]
    if not key_cols or not status_cols:
        return df
    prev_small = prev[key_cols + status_cols].drop_duplicates(subset=key_cols, keep="last")
    out = df.merge(prev_small, on=key_cols, how="left", suffixes=("", "__previous"))
    for col in status_cols:
        prev_col = f"{col}__previous"
        if prev_col in out.columns:
            has_prev = out[prev_col].notna() & out[prev_col].astype(str).ne("")
            out.loc[has_prev, col] = out.loc[has_prev, prev_col]
            out = out.drop(columns=[prev_col])
    return out


df_download_manifest = build_download_manifest(df_relevant_docs)
df_download_manifest = df_download_manifest.drop(columns=[c for c in DROP_HUGE_DOWNLOAD_COLUMNS + ["document_dedup_key"] if c in df_download_manifest.columns], errors="ignore")


# v7 QA: every relevant document must be targeted for download.
def _document_identity_frame(df):
    key_cols = [c for c in ["logicDocId", "docId", "procedureId", "idProcedure", "affId"] if c in df.columns]
    return df[key_cols].astype(str).replace({"nan": "", "<NA>": ""}) if key_cols else pd.DataFrame(index=df.index)

if not df_relevant_docs.empty:
    rel_key_cols = [c for c in ["logicDocId", "docId", "procedureId", "idProcedure", "affId"] if c in df_relevant_docs.columns and c in df_download_manifest.columns]
    if rel_key_cols:
        rel_keys = df_relevant_docs[rel_key_cols].astype(str).replace({"nan": "", "<NA>": ""}).drop_duplicates()
        dl_keys = df_download_manifest[rel_key_cols].astype(str).replace({"nan": "", "<NA>": ""}).drop_duplicates()
        missing_targets = rel_keys.merge(dl_keys, on=rel_key_cols, how="left", indicator=True)
        missing_targets = missing_targets[missing_targets["_merge"].eq("left_only")].drop(columns=["_merge"])
        print("Relevant document rows targeted for download:", f"{len(df_download_manifest):,} / {len(df_relevant_docs):,}")
        print("Relevant document keys missing from download manifest:", f"{len(missing_targets):,}")
        if len(missing_targets):
            display(missing_targets.head(20))
    if "normalized_docTypeCode" in df_relevant_docs.columns:
        print("Relevant document types targeted for download, top 50:")
        display(df_relevant_docs["normalized_docTypeCode"].fillna("UNKNOWN").value_counts().head(50).to_frame("relevant_download_targets"))

print("Download manifest rows:", len(df_download_manifest))
if not df_download_manifest.empty:
    print("Rows with first candidate URL:", df_download_manifest["download_url"].fillna("").ne("").sum())
    print("Rows already existing:", df_download_manifest["file_exists_before_download"].fillna(False).astype(bool).sum())
    display(df_download_manifest[[
        c for c in ["internal_key", "procedure_key", "publishedId", "logicDocId", "normalized_docTypeCode", "docDate", "selected_language", "selected_format", "selected_year", "download_url", "intended_local_path", "existing_local_path", "download_status"]
        if c in df_download_manifest.columns
    ]].head(20))


/tmp/ipykernel_50701/4204036464.py:325: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[True True True ... True True True]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  out.loc[has_prev, col] = out.loc[has_prev, prev_col]


Relevant document rows targeted for download: 77,085 / 77,085
Relevant document keys missing from download manifest: 0
Relevant document types targeted for download, top 50:


,relevant_download_targets
normalized_docTypeCode,
ARRET,19061
ARRET_SOM,16792
ORD_NP,13219
ARRET_NP,7291
ARRET_DR,6784
ORD_COMM,5077
ORD_DR,4144
ORD,2653
ORD_SOM,2045


Download manifest rows: 77085
Rows with first candidate URL: 77085
Rows already existing: 72342


,internal_key,procedure_key,publishedId,logicDocId,normalized_docTypeCode,docDate,selected_language,selected_format,selected_year,download_url,intended_local_path,existing_local_path,download_status
0,2000/0001/C//,2000/0001/C//,C-1/00,id_85906,ARRET_SOM,2001-12-13,EN,PDF,2000,https://infocuriaws.curia.europa.eu/blob/download-file/C/2000/C_0001_00_00000000RD_01_P_01/85906-EN-1.pdf,/home/edik/projects/eccjeu/data/raw/court_infocuria/pdfs/85906-EN-1.pdf,/home/edik/projects/eccjeu/data/raw/court_infocuria/pdfs/85906-EN-1.pdf,already_exists
1,2000/0001/C//,2000/0001/C//,C-1/00,id_46950,ARRET,2001-12-13,EN,HTML,2000,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2000/C_0001_00_00000000RD_01_P_01/46950-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/46950-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/46950-EN-1.html,already_exists
2,2000/0001/C/SA/,2000/0001/C/SA/,C-1/00 SA,id_46421,ORD,2001-05-29,EN,HTML,2000,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2000/C_0001_00_00000000SA_01_P_01/46421-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/46421-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/46421-EN-1.html,already_exists
3,2000/0001/C/SA/,2000/0001/C/SA/,C-1/00 SA,id_85947,ORD_SOM,2001-05-29,EN,PDF,2000,https://infocuriaws.curia.europa.eu/blob/download-file/C/2000/C_0001_00_00000000SA_01_P_01/85947-EN-1.pdf,/home/edik/projects/eccjeu/data/raw/court_infocuria/pdfs/85947-EN-1.pdf,/home/edik/projects/eccjeu/data/raw/court_infocuria/pdfs/85947-EN-1.pdf,already_exists
4,2001/0001/C/P/,2001/0001/C/P/,C-1/01 P,id_85572,ORD_SOM,2001-09-20,EN,PDF,2001,https://infocuriaws.curia.europa.eu/blob/download-file/C/2001/C_0001_01_00000000PV_01_P_01/85572-EN-1.pdf,/home/edik/projects/eccjeu/data/raw/court_infocuria/pdfs/85572-EN-1.pdf,/home/edik/projects/eccjeu/data/raw/court_infocuria/pdfs/85572-EN-1.pdf,already_exists
5,2001/0001/C/P/,2001/0001/C/P/,C-1/01 P,id_46652,ORD,2001-09-20,EN,HTML,2001,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2001/C_0001_01_00000000PV_01_P_01/46652-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/46652-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/46652-EN-1.html,already_exists
6,2002/0001/C//,2002/0001/C//,C-1/02,id_49061,ARRET,2004-04-01,EN,HTML,2002,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2002/C_0001_02_00000000RP_01_P_01/49061-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/49061-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/49061-EN-1.html,already_exists
7,2002/0001/C//,2002/0001/C//,C-1/02,id_84842,ORD_COMM,2003-08-22,EN,HTML,2002,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2002/C_0001_02_00000000RP_01_P_01/84842-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/84842-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/84842-EN-1.html,already_exists
8,2002/0001/C//,2002/0001/C//,C-1/02,id_54478,ORD_COMM,2003-09-06,FR,HTML,2002,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2002/C_0001_02_00000000RP_01_P_01/54478-FR-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/54478-FR-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/54478-FR-1.html,already_exists
9,2002/0001/C//,2002/0001/C//,C-1/02,id_55247,ARRET_SOM,2004-04-01,EN,HTML,2002,https://infocuriaws.curia.europa.eu/blob/download-file-html/C/2002/C_0001_02_00000000RP_01_P_01/55247-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/55247-EN-1.html,/home/edik/projects/eccjeu/data/raw/court_infocuria/html/55247-EN-1.html,already_exists


In [24]:
# ---------------------------------------------------------------------
# 6) Optional downloader for relevant documents only
# ---------------------------------------------------------------------

SUCCESS_STATUSES = {"downloaded", "already_exists"}
FAILED_OR_RETRYABLE_STATUSES = {
    "pending", "failed_http", "failed_exception", "failed_missing_url", "failed_missing_target_path", "failed_empty_content",
}


def candidate_records_from_row(row):
    """Generate candidate download records in memory.

    v4 no longer stores huge candidate_download_records/all_candidate_download_urls
    columns in the manifest. The candidates are rebuilt from metadata when needed.
    """
    records = build_candidate_download_records(row)
    if records:
        return records
    if clean_str(row.get("download_url")):
        return [{
            "url": row.get("download_url"),
            "language": row.get("selected_language"),
            "format": row.get("selected_format"),
            "year": row.get("selected_year"),
            "local_path": first_nonempty(row.get("final_local_path"), row.get("existing_local_path"), row.get("intended_local_path")),
        }]
    return []


def infer_tabs_document_url(file_url, row=None):
    try:
        parsed = urlparse(str(file_url))
        parts = parsed.path.strip("/").split("/")
        if len(parts) < 4:
            return "https://infocuria.curia.europa.eu/"
        filename = parts[-1]
        logic_fmt = filename.replace(".", "-")
        doc_type = first_nonempty((row or {}).get("normalized_docTypeCode"), (row or {}).get("docTypeCode"))
        routing = first_nonempty((row or {}).get("idProcedure"), (row or {}).get("procedureId"))
        if routing and doc_type:
            routing_pretty = routing.replace("/", "-")
            return f"https://infocuria.curia.europa.eu/tabs/document/{routing_pretty}/{doc_type}/{logic_fmt}"
    except Exception:
        pass
    return "https://infocuria.curia.europa.eu/"


def get_download_session():
    session = requests.Session()
    session.headers.update(HEADERS)
    return session


def polite_sleep(base=None):
    base = DOWNLOAD_SLEEP_SECONDS if base is None else base
    delay = float(base) + random.uniform(0, float(DOWNLOAD_SLEEP_JITTER_SECONDS))
    if delay > 0:
        time.sleep(delay)


def warmup_session_for_url(session, file_url, row):
    if not SESSION_WARMUP:
        return None
    referrer = infer_tabs_document_url(file_url, row=row)
    try:
        session.get(referrer, timeout=REQUEST_TIMEOUT_SECONDS, headers={**HEADERS, "Referer": "https://infocuria.curia.europa.eu/"}, allow_redirects=True)
        time.sleep(float(SESSION_WARMUP_SLEEP_SECONDS))
        return referrer
    except Exception:
        return "https://infocuria.curia.europa.eu/"


def download_one_file(row, overwrite=False, session=None):
    records = candidate_records_from_row(row)
    if not records:
        return {
            "download_success": False,
            "download_status": "failed_missing_url",
            "http_status": pd.NA,
            "error": "Missing candidate_download_records/download_url",
            "bytes": pd.NA,
            "working_url": pd.NA,
            "working_language": pd.NA,
            "working_format": pd.NA,
            "working_year": pd.NA,
            "final_local_path": pd.NA,
            "attempted_urls": json.dumps([], ensure_ascii=False),
            "attempted_http_statuses": json.dumps([], ensure_ascii=False),
            "attempt_count": 0,
        }

    session = session or get_download_session()
    attempted_urls = []
    attempted_statuses = []
    last_error = ""
    last_status = pd.NA
    last_length = 0

    for record in records:
        file_url = clean_str(record.get("url"))
        target_path = Path(clean_str(record.get("local_path"))) if clean_str(record.get("local_path")) else None
        selected_format = clean_str(record.get("format")).upper()

        if not file_url or target_path is None:
            continue

        if target_path.exists() and not overwrite:
            return {
                "download_success": True,
                "download_status": "already_exists",
                "http_status": pd.NA,
                "error": pd.NA,
                "bytes": target_path.stat().st_size,
                "working_url": file_url,
                "working_language": record.get("language"),
                "working_format": selected_format,
                "working_year": record.get("year"),
                "final_local_path": str(target_path),
                "attempted_urls": json.dumps(attempted_urls, ensure_ascii=False),
                "attempted_http_statuses": json.dumps(attempted_statuses, ensure_ascii=False),
                "attempt_count": len(attempted_urls),
            }

        referrer = warmup_session_for_url(session, file_url, row)

        for retry_no in range(1, int(RETRIES_PER_URL) + 1):
            attempted_urls.append(file_url)
            try:
                response = session.get(
                    file_url,
                    timeout=REQUEST_TIMEOUT_SECONDS,
                    headers={**HEADERS, "Referer": referrer or "https://infocuria.curia.europa.eu/"},
                    allow_redirects=True,
                )
                status_code = response.status_code
                content = response.content or b""
                last_status = status_code
                last_length = len(content)
                attempted_statuses.append(status_code)

                if status_code == 200 and content:
                    target_path.parent.mkdir(parents=True, exist_ok=True)
                    if selected_format == "HTML" or file_url.lower().endswith(".html"):
                        target_path.write_text(response.text, encoding="utf-8")
                    else:
                        target_path.write_bytes(content)
                    return {
                        "download_success": True,
                        "download_status": "downloaded",
                        "http_status": status_code,
                        "error": pd.NA,
                        "bytes": target_path.stat().st_size,
                        "working_url": file_url,
                        "working_language": record.get("language"),
                        "working_format": selected_format,
                        "working_year": record.get("year"),
                        "final_local_path": str(target_path),
                        "attempted_urls": json.dumps(attempted_urls, ensure_ascii=False),
                        "attempted_http_statuses": json.dumps(attempted_statuses, ensure_ascii=False),
                        "attempt_count": len(attempted_urls),
                    }

                last_error = f"HTTP {status_code}; length={len(content)}"
                if status_code in {403, 429, 500, 502, 503, 504} and retry_no < int(RETRIES_PER_URL):
                    time.sleep(float(RETRY_BACKOFF_SECONDS) * retry_no + random.uniform(0, 1.5))
                    continue
                # 404: try the next URL candidate immediately.
                break
            except Exception as e:
                last_error = repr(e)
                attempted_statuses.append("EXCEPTION")
                if retry_no < int(RETRIES_PER_URL):
                    time.sleep(float(RETRY_BACKOFF_SECONDS) * retry_no + random.uniform(0, 1.5))
                    continue
                break
        polite_sleep()

    return {
        "download_success": False,
        "download_status": "failed_http" if attempted_urls else "failed_missing_url",
        "http_status": last_status,
        "error": f"{last_error}; last_length={last_length}",
        "bytes": last_length,
        "working_url": pd.NA,
        "working_language": pd.NA,
        "working_format": pd.NA,
        "working_year": pd.NA,
        "final_local_path": pd.NA,
        "attempted_urls": json.dumps(attempted_urls, ensure_ascii=False),
        "attempted_http_statuses": json.dumps(attempted_statuses, ensure_ascii=False),
        "attempt_count": len(attempted_urls),
    }


def run_downloads(df_download, limit=None, overwrite=False, save_every=100):
    df = df_download.copy()
    if df.empty:
        print("Download manifest is empty.")
        return df

    # If a final/specified local path exists, mark it as already present.
    # Use NA-safe helpers: pandas pd.NA cannot be used with `or`/`if`.
    existing_now = df.apply(
        lambda r: path_exists_safe(r.get("final_local_path"), r.get("existing_local_path"), r.get("intended_local_path")),
        axis=1,
    )
    if not overwrite:
        df.loc[existing_now, "download_success"] = True
        df.loc[existing_now, "download_status"] = "already_exists"

    status = df["download_status"].fillna("pending").astype(str)
    success_bool = df.get("download_success", pd.Series(False, index=df.index)).astype(str).str.lower().isin(["true", "1", "yes"])
    has_candidates = df["download_url"].fillna("").astype(str).str.strip().ne("")

    needs_attempt = (
        has_candidates
        & (overwrite | ~existing_now)
        & (
            overwrite
            | status.isin(FAILED_OR_RETRYABLE_STATUSES)
            | status.str.startswith("failed")
            | (~success_bool & ~status.isin(SUCCESS_STATUSES))
        )
        & ~status.isin(SUCCESS_STATUSES)
    )

    pending_indices = df.index[needs_attempt].tolist()
    if limit is not None:
        pending_indices = pending_indices[:int(limit)]

    print(f"Download attempts planned: {len(pending_indices):,}")
    session = get_download_session()
    for n, idx in enumerate(tqdm(pending_indices, desc="Downloading relevant InfoCuria documents"), start=1):
        result = download_one_file(df.loc[idx].to_dict(), overwrite=overwrite, session=session)
        for k, v in result.items():
            df.at[idx, k] = v
        if save_every and n % int(save_every) == 0:
            save_jsonl(df.drop(columns=[c for c in DROP_HUGE_DOWNLOAD_COLUMNS + ["document_dedup_key"] if c in df.columns], errors="ignore"), DOWNLOAD_MANIFEST_JSONL)

    df = df.drop(columns=[c for c in DROP_HUGE_DOWNLOAD_COLUMNS + ["document_dedup_key"] if c in df.columns], errors="ignore")
    save_jsonl(df, DOWNLOAD_MANIFEST_JSONL)
    return df


if RUN_DOWNLOADS:
    df_download_manifest = run_downloads(
        df_download_manifest,
        limit=DOWNLOAD_LIMIT,
        overwrite=OVERWRITE_EXISTING_FILES,
        save_every=DOWNLOAD_SAVE_EVERY,
    )
else:
    print("RUN_DOWNLOADS=False. No documents downloaded.")
    print("The download manifest contains selected/attempted URLs and previous/current status where available.")


Download attempts planned: 4,247


Saved JSONL 77,085 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_download_manifest.jsonl
Saved JSONL 77,085 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_download_manifest.jsonl
Saved JSONL 77,085 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_download_manifest.jsonl
Saved JSONL 77,085 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_download_manifest.jsonl
Saved JSONL 77,085 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_download_manifest.jsonl
Saved JSONL 77,085 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_download_manifest.jsonl
Saved JSONL 77,085 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_download_manifest.jsonl
Saved JSONL 77,085 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_download_manifes

# Optional notebook-only QA preview

The requested persistent outputs are saved in the next section. This section only creates an in-memory QA preview so that the notebook remains useful for manual inspection without producing extra CSV files.


In [25]:
# ---------------------------------------------------------------------
# 7) Optional notebook-only QA preview
# ---------------------------------------------------------------------

qa_preview_cols = [
    "procedure_key", "procedure_key_base", "case_number_raw", "case_number_clean",
    "metadata_join_status", "available_document_types", "relevant_document_types",
    "total_document_count", "relevant_document_count", "irrelevant_document_count",
    "case_relevance_flag", "curia_join_status",
]
qa_preview_cols = [c for c in qa_preview_cols if c in df_case_metadata_manifest.columns]

print("Case QA preview rows:", len(df_case_metadata_manifest))
if "metadata_join_status" in df_case_metadata_manifest.columns:
    display(df_case_metadata_manifest["metadata_join_status"].fillna("missing").value_counts(dropna=False).to_frame("count"))
if "case_relevance_flag" in df_case_metadata_manifest.columns:
    display(df_case_metadata_manifest["case_relevance_flag"].fillna(False).astype(bool).value_counts(dropna=False).to_frame("count"))

if qa_preview_cols:
    display(df_case_metadata_manifest[qa_preview_cols].head(30))


Case QA preview rows: 56828


,count
metadata_join_status,
metadata_joined,56828


,count
case_relevance_flag,
True,40117
False,16711


,procedure_key,case_number_raw,case_number_clean,metadata_join_status,available_document_types,relevant_document_types,total_document_count,relevant_document_count,irrelevant_document_count,case_relevance_flag,curia_join_status
0,2000/0001/C//,C-1/00,C-1/00,metadata_joined,ARRET|ARRET_SOM|ARR_COMM|AVI_COMM|CONCL,ARRET|ARRET_SOM,5,2,3,True,matched
1,2000/0001/C/SA/,C-1/00 SA,C-1/00 SA,metadata_joined,ORD|ORD_SOM,ORD|ORD_SOM,2,2,0,True,matched
2,2001/0001/C/P/,C-1/01 P,C-1/01 P,metadata_joined,ORD|ORD_SOM,ORD|ORD_SOM,2,2,0,True,matched
3,2002/0001/C//,C-1/02,C-1/02,metadata_joined,ARRET|ARRET_SOM|ARR_COMM|CONCL|DDP_COMM|ORD_COMM,ARRET|ARRET_SOM|ORD_COMM,7,4,3,True,matched
4,2002/0001/C/SA/,C-1/02 SA,C-1/02 SA,metadata_joined,ORD|ORD_SOM,ORD|ORD_SOM,2,2,0,True,matched
5,2003/0001/C//,C-1/03,C-1/03,metadata_joined,ARRET|ARRET_SOM|ARR_COMM|CONCL|DDP_COMM|RAD_COMM|REQ_COMM,ARRET|ARRET_SOM,7,2,5,True,matched
6,2003/0001/C/SA/,C-1/03 SA,C-1/03 SA,metadata_joined,,,0,0,0,False,matched
7,2004/0001/C//,C-1/04,C-1/04,metadata_joined,ARRET|ARRET_SOM|AVI_COMM|CONCL|DDP_COMM|ORD_COMM|RAD_COMM|REQ_COMM,ARRET|ARRET_SOM|ORD_COMM,9,4,5,True,matched
8,2004/0001/C/SA/,C-1/04 SA,C-1/04 SA,metadata_joined,ORD|ORD_SOM,ORD|ORD_SOM,2,2,0,True,matched
9,2005/0001/C//,C-1/05,C-1/05,metadata_joined,ARRET|ARRET_SOM|CONCL|DDP_COMM,ARRET|ARRET_SOM,4,2,2,True,matched


In [26]:

# ---------------------------------------------------------------------
# 8) Joined-case lookup repair via InfoCuria procedures endpoint
# ---------------------------------------------------------------------
# v9 patch:
# The previous manual lookup rendered the Angular InfoCuria UI and scraped
# "Joined Case(s): ..." from page text. That was fragile because the UI can render
# duplicate <body> nodes, empty shells, translation-key placeholders, and truncated
# joined-case previews such as "C-7/56, C-4/57, ..., C-7/57".
#
# The robust source is the same API endpoint behind the side pane:
#   https://infocuriaws.curia.europa.eu/elastic-connector/affairId/procedures
#
# For every case with a usable affId and case identifier, we call the endpoint with its affId and extract
# joined cases ONLY from:
#   content.idPilot
#   content.joinAffairs
#
# These values have a stable format such as:
#   C/0007/56/00000000RF/01/P/01#C-7/56
# so we take the case number after '#'. We deliberately do not scan doctrine notes,
# document body text, highlights, or arbitrary JSON strings.

JOINED_CASE_PROCEDURES_URL = PROCEDURES_URL


def joinexist_mask(df: pd.DataFrame) -> pd.Series:
    """Return True for rows where any joinExist-like column is truthy."""
    join_cols = [
        c for c in df.columns
        if c.lower() == "joinexist" or c.lower().endswith(".joinexist") or "joinexist" in c.lower()
    ]
    if not join_cols:
        return pd.Series(False, index=df.index)
    mask = pd.Series(False, index=df.index)
    for c in join_cols:
        vals = df[c].fillna("").astype(str).str.strip().str.lower()
        mask = mask | vals.isin(["1", "true", "yes"])
    return mask


def normalize_joined_case_number(x):
    """Normalize a displayed InfoCuria case number, preserving suffixes such as P."""
    x = clean_str(x).upper()
    x = re.sub(r"<[^>]+>", " ", x)
    x = re.sub(r"&nbsp;|&#160;", " ", x, flags=re.I)
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"^([CTF])\s*-\s*", r"\1-", x)
    x = re.sub(r"\s*/\s*", "/", x)
    return x.strip()


JOINED_CASE_NUMBER_RE = re.compile(
    r"\b(?:C|T|F)-\s*\d+/\d+(?:\s*(?:P|PPU|R|DEP|P-DEP|REC|RX))?\b",
    flags=re.I,
)


def extract_case_after_hash(value):
    """
    Extract the displayed case number from an InfoCuria joined-affair reference.

    Expected shape:
        C/0007/56/00000000RF/01/P/01#C-7/56
    Returns:
        C-7/56
    """
    if value is None:
        return ""
    s = clean_str(value)
    if not s:
        return ""
    if "#" in s:
        return normalize_joined_case_number(s.split("#", 1)[1])
    # Conservative fallback for unexpected shapes.
    m = JOINED_CASE_NUMBER_RE.search(s)
    return normalize_joined_case_number(m.group(0)) if m else ""


def parse_join_affairs_value(value):
    """Parse joinAffairs whether it is already a list or a JSON/stringified list."""
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        s = value.strip()
        if not s or s in {"[]", "nan", "None"}:
            return []
        try:
            parsed = json.loads(s)
            if isinstance(parsed, list):
                return parsed
            if parsed:
                return [parsed]
            return []
        except Exception:
            # Allow pipe-separated or comma-separated diagnostics as fallback.
            if "|" in s:
                return [x.strip() for x in s.split("|") if x.strip()]
            if "," in s:
                return [x.strip() for x in s.split(",") if x.strip()]
            return [s]
    return [value]


def extract_joined_cases_from_content(content: dict, query_case: str):
    """
    Production joined-case extraction rule.

    Use only:
      - content.idPilot
      - content.joinAffairs

    Do not inspect doctrine notes, full document text, highlights, or arbitrary JSON.
    """
    query_norm = normalize_joined_case_number(query_case)
    raw_sources = []

    id_pilot = content.get("idPilot")
    if id_pilot:
        raw_sources.append(("idPilot", id_pilot))

    for v in parse_join_affairs_value(content.get("joinAffairs") or []):
        if v:
            raw_sources.append(("joinAffairs", v))

    joined_cases = []
    for source, value in raw_sources:
        c = extract_case_after_hash(value)
        if c and c != query_norm:
            joined_cases.append(c)

    # Deduplicate while preserving API order.
    seen = set()
    ordered = []
    for c in joined_cases:
        if c not in seen:
            seen.add(c)
            ordered.append(c)

    return ordered, raw_sources


def procedures_payload_for_joined_cases(aff_id: str, published_id: str, language: str = LANGUAGE):
    return {
        "affId": clean_str(aff_id),
        "searchTerm": f'"{clean_str(published_id)}"',
        "tabName": "affair",
        "language": language,
    }


def choose_procedure_content_for_affid(response_json: dict, aff_id: str, published_id: str = ""):
    """Choose the exact searchHits[*].content for the requested affId/publishedId."""
    hits = response_json.get("searchHits") or []
    if not hits:
        return {}, "no_hits"

    aff_id = clean_str(aff_id)
    published_norm = normalize_joined_case_number(published_id)

    # Prefer exact affId + exact publishedId/publishedAffId.
    for hit in hits:
        c = hit.get("content") or {}
        if clean_str(c.get("affId")) != aff_id:
            continue
        if published_norm and normalize_joined_case_number(c.get("publishedId")) == published_norm:
            return c, "exact_affid_publishedId"
        if published_norm and normalize_joined_case_number(c.get("publishedAffId")) == published_norm:
            return c, "exact_affid_publishedAffId"

    # Then exact affId only.
    for hit in hits:
        c = hit.get("content") or {}
        if clean_str(c.get("affId")) == aff_id:
            return c, "exact_affid"

    # Last fallback: first hit.
    return (hits[0].get("content") or {}), "first_hit_fallback"


def local_joined_cases_from_row(row: pd.Series, published_id: str):
    """Extract joined cases from already-flattened local row columns if available."""
    content = {
        "idPilot": row.get("idPilot", ""),
        "joinAffairs": row.get("joinAffairs", []),
    }
    return extract_joined_cases_from_content(content, published_id)


def display_case_for_join_collection(row: pd.Series, candidate_cols=None):
    """Pick the best displayed case identifier for joined-case grouping."""
    if candidate_cols is None:
        candidate_cols = [
            "publishedId", "case_number_raw", "idPublished", "publishedAffId",
            "case_number_clean", "procNumber", "case_number",
        ]
    for c in candidate_cols:
        if c in row.index:
            v = normalize_joined_case_number(row.get(c))
            if JOINED_CASE_NUMBER_RE.search(v):
                return v
            # Some columns may already be exactly normalized but fail due to suffix quirks.
            if re.match(r"^(?:C|T|F)-\d+/\d+", v):
                return v
    return ""


def split_pipe_cases(value):
    """Split a pipe/list-like case cell into normalized case numbers."""
    s = clean_str(value)
    if not s:
        return []
    values = []
    # JSON/list-style cells are tolerated.
    if s.startswith("["):
        try:
            obj = json.loads(s)
            if isinstance(obj, list):
                values = obj
            else:
                values = [obj]
        except Exception:
            values = re.split(r"[|,;]", s)
    else:
        values = re.split(r"[|,;]", s)
    out = []
    for v in values:
        c = normalize_joined_case_number(v)
        if c and re.match(r"^(?:C|T|F)-\d+/\d+", c):
            out.append(c)
    # dedupe preserving order
    seen = set()
    ordered = []
    for c in out:
        if c not in seen:
            seen.add(c)
            ordered.append(c)
    return ordered


def add_joined_case_collection(df_cases_manifest: pd.DataFrame) -> pd.DataFrame:
    """
    Add propagated joined-case groups after manual_joined_case_search.

    Example:
      C-1/00 row says C-2/00|C-3/00
      then rows C-1/00, C-2/00 and C-3/00 all receive:
      C-1/00|C-2/00|C-3/00

    This does not replace the direct `manual_joined_case_search`; it adds a new
    aggregate column. Existing values in `manual_joined_case_collection` are
    merged with newly inferred values rather than discarded.
    """
    out = df_cases_manifest.copy()
    collection_col = "manual_joined_case_collection"
    if collection_col not in out.columns:
        out[collection_col] = ""

    row_cases = out.apply(display_case_for_join_collection, axis=1)
    out["_join_collection_case_number"] = row_cases

    parent = {}
    order_seen = []

    def add_node(x):
        x = normalize_joined_case_number(x)
        if not x:
            return ""
        if x not in parent:
            parent[x] = x
            order_seen.append(x)
        return x

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        a = add_node(a)
        b = add_node(b)
        if not a or not b:
            return
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    # Build links: row's own displayed case + all directly extracted joined cases.
    for idx, row in out.iterrows():
        own_case = row_cases.loc[idx]
        direct_joined = split_pipe_cases(row.get("manual_joined_case_search", ""))
        existing_collection = split_pipe_cases(row.get(collection_col, ""))
        group = [own_case] + direct_joined + existing_collection
        group = [g for g in group if g]
        if not group:
            continue
        add_node(group[0])
        for g in group[1:]:
            union(group[0], g)

    # Collect connected components preserving discovery order.
    components = {}
    for case in order_seen:
        root = find(case)
        components.setdefault(root, [])
        if case not in components[root]:
            components[root].append(case)

    case_to_collection = {}
    for cases in components.values():
        joined = "|".join(cases)
        for c in cases:
            case_to_collection[c] = joined

    # Assign propagated collection to every row with a known case.
    for idx, own_case in row_cases.items():
        existing = split_pipe_cases(out.at[idx, collection_col])
        propagated = split_pipe_cases(case_to_collection.get(own_case, ""))
        merged = []
        for c in existing + propagated:
            if c and c not in merged:
                merged.append(c)
        out.at[idx, collection_col] = "|".join(merged)

    populated = out[collection_col].fillna("").astype(str).str.strip().ne("").sum()
    print(f"Rows with manual_joined_case_collection populated: {populated:,}")
    out.drop(columns=["_join_collection_case_number"], inplace=True, errors="ignore")
    return out



def joined_case_request_key(aff_id: str, published_id: str) -> str:
    """Stable key for the joined-case API cache."""
    return f"{clean_str(aff_id)}||{normalize_joined_case_number(published_id)}"


def joined_case_cache_record_key(record: dict) -> str:
    return clean_str(record.get("joined_case_request_key")) or joined_case_request_key(
        record.get("affId", ""),
        record.get("publishedId", ""),
    )


def load_joined_case_cache_latest(path: Path) -> dict:
    """Load latest JSONL record per affId/publishedId key, keeping the last attempt."""
    latest = {}
    if not path.exists():
        return latest
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                # Do not spam warnings during long runs; just ignore malformed cache lines.
                continue
            key = joined_case_cache_record_key(rec)
            if key:
                latest[key] = rec
    return latest


def append_joined_case_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", newline="\n") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def compact_joined_case_jsonl_keep_last(path: Path):
    """Compact joined-case JSONL to one latest record per request key."""
    if not path.exists():
        return
    latest = load_joined_case_cache_latest(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8", newline="\n") as f:
        for rec in latest.values():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    tmp.replace(path)
    print(f"Compacted joined-case JSONL to {len(latest):,} latest records -> {path}")


def joined_case_cache_record_is_reusable(record: dict) -> bool:
    """
    True means update_failed mode can reuse this cache record without calling the API again.

    We reuse HTTP 200/success records, including rows where the API worked but found no joined cases.
    We retry explicit failure statuses and retryable HTTP statuses such as 403/429/5xx.
    """
    if not record:
        return False
    status = clean_str(record.get("status")).lower()
    http_status = record.get("http_status")
    try:
        http_status_int = int(http_status) if clean_str(http_status) else None
    except Exception:
        http_status_int = None

    if status == "success" and http_status_int == 200:
        return True
    if http_status_int in JOINED_CASE_RETRY_HTTP_STATUSES:
        return False
    if status in {"http_error", "exception", "parse_error", "local_fallback_error", "missing_identifiers"}:
        return False
    return False


def response_summary_for_joined_case_cache(response_json: dict) -> dict:
    """Light diagnostics only; avoids storing the entire API payload for every row."""
    hits = response_json.get("searchHits") or [] if isinstance(response_json, dict) else []
    summary = {
        "search_hits_count": len(hits),
        "hit_affIds": [],
        "hit_publishedIds": [],
    }
    if JOINED_CASE_RECORD_RESPONSE_SUMMARY:
        for hit in hits[:10]:
            c = hit.get("content") or {}
            summary["hit_affIds"].append(clean_str(c.get("affId")))
            summary["hit_publishedIds"].append(first_nonempty(c.get("publishedId"), c.get("publishedAffId")))
    return summary


def build_joined_case_cache_record_from_local(row, idx, aff_id, published_id, reason="local_only"):
    """Create a non-API record from local flattened fields, used for disabled/csv/cache fallback cases."""
    local_cases, local_raw_sources = local_joined_cases_from_row(row, published_id)
    return {
        "joined_case_request_key": joined_case_request_key(aff_id, published_id),
        "row_index": int(idx) if isinstance(idx, (int, np.integer)) else str(idx),
        "affId": aff_id,
        "publishedId": published_id,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "status": reason,
        "http_status": None,
        "content_choice": "local_flattened_fields",
        "manual_joined_case_search": "|".join(local_cases),
        "manual_joined_case_search_source": "local_flattened_fields" if local_cases else "no_local_joined_cases",
        "manual_joined_case_search_raw_sources": "|".join(f"{src}={val}" for src, val in local_raw_sources),
        "api_joined_cases": [],
        "local_joined_cases": local_cases,
        "error_type": "",
        "error": "",
    }


def apply_joined_case_cache_record_to_output(out: pd.DataFrame, idx, row, record: dict, published_id: str):
    """Apply one cache/API/local record to the case manifest row."""
    if not record:
        local_cases, local_raw_sources = local_joined_cases_from_row(row, published_id)
        out.at[idx, "manual_joined_case_search"] = "|".join(local_cases)
        out.at[idx, "manual_joined_case_search_source"] = "local_flattened_fields_no_cache" if local_cases else "no_cache_no_local_joined_cases"
        out.at[idx, "manual_joined_case_search_raw_sources"] = "|".join(f"{src}={val}" for src, val in local_raw_sources)
        out.at[idx, "manual_joined_case_search_status"] = "no_cache"
        return

    out.at[idx, "manual_joined_case_search"] = clean_str(record.get("manual_joined_case_search"))
    out.at[idx, "manual_joined_case_search_source"] = clean_str(record.get("manual_joined_case_search_source"))
    out.at[idx, "manual_joined_case_search_raw_sources"] = clean_str(record.get("manual_joined_case_search_raw_sources"))

    status = clean_str(record.get("status"))
    http_status = clean_str(record.get("http_status"))
    if http_status:
        out.at[idx, "manual_joined_case_search_status"] = f"{status}:http_{http_status}"
    else:
        out.at[idx, "manual_joined_case_search_status"] = status


def request_joined_case_record(session, row, idx, aff_id: str, published_id: str, request_counter: int) -> dict:
    """
    Call InfoCuria once and return a JSONL-safe result record.

    No warning is logged for 403/429/etc. These are normal, restartable outcomes.
    """
    local_cases, local_raw_sources = local_joined_cases_from_row(row, published_id)
    payload = procedures_payload_for_joined_cases(aff_id, published_id)
    base_record = {
        "joined_case_request_key": joined_case_request_key(aff_id, published_id),
        "row_index": int(idx) if isinstance(idx, (int, np.integer)) else str(idx),
        "request_counter": int(request_counter),
        "affId": aff_id,
        "publishedId": published_id,
        "payload": payload,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "local_joined_cases": local_cases,
        "manual_joined_case_search_raw_sources": "|".join(f"{src}={val}" for src, val in local_raw_sources),
        "api_joined_cases": [],
        "content_choice": "",
        "response_summary": {},
    }

    try:
        resp = session.post(
            JOINED_CASE_PROCEDURES_URL,
            json=payload,
            timeout=METADATA_REQUEST_TIMEOUT_SECONDS,
        )
        base_record["http_status"] = int(resp.status_code)

        if resp.status_code >= 400:
            final_cases = local_cases
            base_record.update({
                "status": "http_error",
                "manual_joined_case_search": "|".join(final_cases),
                "manual_joined_case_search_source": (
                    "local_flattened_fields_after_api_error" if final_cases else "api_error_no_local_joined_cases"
                ),
                "error_type": "HTTPError",
                "error": f"HTTP {resp.status_code}: {resp.text[:300]}",
            })
            return base_record

        try:
            response_json = resp.json()
        except Exception as e:
            final_cases = local_cases
            base_record.update({
                "status": "parse_error",
                "manual_joined_case_search": "|".join(final_cases),
                "manual_joined_case_search_source": (
                    "local_flattened_fields_after_parse_error" if final_cases else "parse_error_no_local_joined_cases"
                ),
                "error_type": type(e).__name__,
                "error": repr(e),
            })
            return base_record

        content, content_choice = choose_procedure_content_for_affid(response_json, aff_id, published_id)
        api_cases, api_raw_sources = extract_joined_cases_from_content(content, published_id)
        final_cases = api_cases or local_cases
        final_raw_sources = api_raw_sources or local_raw_sources
        source = f"procedures_endpoint:{content_choice}" if api_cases else "local_flattened_fields"

        base_record.update({
            "status": "success",
            "content_choice": content_choice,
            "response_summary": response_summary_for_joined_case_cache(response_json),
            "api_joined_cases": api_cases,
            "manual_joined_case_search": "|".join(final_cases),
            "manual_joined_case_search_source": source if final_cases else f"no_joined_cases_found:{content_choice}",
            "manual_joined_case_search_raw_sources": "|".join(f"{src}={val}" for src, val in final_raw_sources),
            "error_type": "",
            "error": "",
        })
        return base_record

    except Exception as e:
        final_cases = local_cases
        base_record.update({
            "status": "exception",
            "http_status": None,
            "manual_joined_case_search": "|".join(final_cases),
            "manual_joined_case_search_source": (
                "local_flattened_fields_after_exception" if final_cases else "exception_no_local_joined_cases"
            ),
            "error_type": type(e).__name__,
            "error": repr(e)[:500],
        })
        return base_record


def add_manual_joined_case_search(df_cases_manifest: pd.DataFrame) -> pd.DataFrame:
    out = df_cases_manifest.copy()

    for col in [
        "manual_joined_case_search",
        "manual_joined_case_search_status",
        "manual_joined_case_search_source",
        "manual_joined_case_search_raw_sources",
        "manual_joined_case_collection",
    ]:
        if col not in out.columns:
            out[col] = ""

    published_col = next(
        (c for c in ["publishedId", "case_number_raw", "idPublished", "case_number_clean"] if c in out.columns),
        None,
    )
    if published_col is None:
        print("Joined-case API lookup skipped: no published case identifier column found.")
        return add_joined_case_collection(out)

    aff_col = next((c for c in ["affId", "idAffaire"] if c in out.columns), None)
    if aff_col is None:
        print("Joined-case API lookup skipped: no affId column found.")
        return add_joined_case_collection(out)

    # v12/v14: run through every case with usable identifiers, not only joinExist == 1.
    # This is intentional because asymmetric joined-case metadata can be carried back
    # to all members of a connected joined-case component.
    todo = out.loc[
        out[published_col].fillna("").astype(str).str.strip().ne("")
        & out[aff_col].fillna("").astype(str).str.strip().ne("")
    ].copy()

    if MANUAL_JOINED_CASE_SEARCH_LIMIT is not None:
        todo = todo.head(int(MANUAL_JOINED_CASE_SEARCH_LIMIT)).copy()

    print(f"Joined-case repair candidate rows, all cases with identifiers: {len(todo):,}")
    print("JOINED_CASE_JSONL_MODE:", JOINED_CASE_JSONL_MODE)
    print("Joined-case JSONL:", JOINED_CASE_JSONL)

    if todo.empty:
        return add_joined_case_collection(out)

    if not RUN_MANUAL_JOINED_CASE_SEARCH:
        print("Joined-case API lookup disabled; building collection from existing/manual/local fields only.")
        for idx, row in todo.iterrows():
            aff_id = clean_str(row.get(aff_col))
            published_id = clean_str(row.get(published_col))
            record = build_joined_case_cache_record_from_local(row, idx, aff_id, published_id, reason="local_only_api_disabled")
            apply_joined_case_cache_record_to_output(out, idx, row, record, published_id)
        return add_joined_case_collection(out)

    if JOINED_CASE_JSONL_MODE == "rebuild_jsonl" and JOINED_CASE_JSONL.exists():
        JOINED_CASE_JSONL.unlink()
        print("Deleted existing joined-case JSONL for rebuild:", JOINED_CASE_JSONL)

    cache_latest = load_joined_case_cache_latest(JOINED_CASE_JSONL)
    print(f"Loaded joined-case cache records: {len(cache_latest):,}")

    session = requests.Session()
    session.headers.update({
        "accept": "application/json",
        "accept-language": HEADERS.get("Accept-Language", "en-US,en;q=0.9"),
        "content-type": "application/json; charset=UTF-8",
        "origin": "https://infocuria.curia.europa.eu",
        "referer": "https://infocuria.curia.europa.eu/",
        "user-agent": HEADERS.get("User-Agent", "Mozilla/5.0"),
    })

    stats = Counter()
    api_request_count = 0

    if JOINED_CASE_JSONL_MODE == "csv_from_jsonl":
        for idx, row in tqdm(todo.iterrows(), total=len(todo), desc="Joined-case repair from JSONL only"):
            aff_id = clean_str(row.get(aff_col))
            published_id = clean_str(row.get(published_col))
            key = joined_case_request_key(aff_id, published_id)
            record = cache_latest.get(key)
            if record:
                stats[f"cache_{clean_str(record.get('status')) or 'unknown'}"] += 1
            else:
                stats["missing_cache_local_fallback"] += 1
            apply_joined_case_cache_record_to_output(out, idx, row, record, published_id)

        print("Joined-case JSONL-only rebuild summary:")
        print(pd.Series(dict(stats)).sort_index())
        found = out["manual_joined_case_search"].fillna("").astype(str).str.strip().ne("").sum()
        print(f"Cases with manual_joined_case_search populated: {found:,}")
        return add_joined_case_collection(out)

    rows_to_request = []
    rows_to_reuse = []
    for idx, row in todo.iterrows():
        aff_id = clean_str(row.get(aff_col))
        published_id = clean_str(row.get(published_col))
        key = joined_case_request_key(aff_id, published_id)
        cached = cache_latest.get(key)
        if JOINED_CASE_JSONL_MODE == "update_failed" and joined_case_cache_record_is_reusable(cached):
            rows_to_reuse.append((idx, row, cached))
        else:
            rows_to_request.append((idx, row))

    print(f"Joined-case rows reused from cache: {len(rows_to_reuse):,}")
    print(f"Joined-case rows to request now: {len(rows_to_request):,}")

    for idx, row, cached in tqdm(rows_to_reuse, total=len(rows_to_reuse), desc="Joined-case cache reuse"):
        published_id = clean_str(row.get(published_col))
        apply_joined_case_cache_record_to_output(out, idx, row, cached, published_id)
        stats[f"reused_{clean_str(cached.get('status')) or 'unknown'}"] += 1

    pbar = tqdm(rows_to_request, total=len(rows_to_request), desc="Joined-case API/cache update")
    for idx, row in pbar:
        aff_id = clean_str(row.get(aff_col))
        published_id = clean_str(row.get(published_col))

        api_request_count += 1
        record = request_joined_case_record(session, row, idx, aff_id, published_id, api_request_count)
        append_joined_case_jsonl(JOINED_CASE_JSONL, record)
        cache_latest[joined_case_cache_record_key(record)] = record
        apply_joined_case_cache_record_to_output(out, idx, row, record, published_id)

        status = clean_str(record.get("status")) or "unknown"
        http_status = clean_str(record.get("http_status"))
        stats[status] += 1
        if http_status:
            stats[f"http_{http_status}"] += 1
        pbar.set_postfix({
            "ok": stats.get("success", 0),
            "403": stats.get("http_403", 0),
            "429": stats.get("http_429", 0),
            "err": stats.get("exception", 0) + stats.get("parse_error", 0),
        })

        if MANUAL_JOINED_CASE_SLEEP_SECONDS:
            time.sleep(float(MANUAL_JOINED_CASE_SLEEP_SECONDS))

        if (
            JOINED_CASE_BATCH_PAUSE_EVERY
            and api_request_count % int(JOINED_CASE_BATCH_PAUSE_EVERY) == 0
            and api_request_count < len(rows_to_request)
        ):
            pause = float(JOINED_CASE_BATCH_PAUSE_SECONDS)
            print(f"\nJoined-case cooldown after {api_request_count:,} API requests: sleeping {pause:.0f}s")
            time.sleep(pause)

    if JOINED_CASE_COMPACT_JSONL_AFTER_RUN:
        compact_joined_case_jsonl_keep_last(JOINED_CASE_JSONL)

    print("Joined-case repair summary for this run:")
    if stats:
        print(pd.Series(dict(stats)).sort_index())
    else:
        print("No joined-case API requests were needed.")

    found = out["manual_joined_case_search"].fillna("").astype(str).str.strip().ne("").sum()
    print(f"Cases with manual_joined_case_search populated: {found:,}")
    out = add_joined_case_collection(out)
    return out


# Execute lookup before final manifests are saved.
df_case_metadata_manifest = add_manual_joined_case_search(df_case_metadata_manifest)


Joined-case repair candidate rows, all cases with identifiers: 56,828
JOINED_CASE_JSONL_MODE: update_failed
Joined-case JSONL: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_joined_case_api_cache.jsonl
Loaded joined-case cache records: 51,655
Joined-case rows reused from cache: 55,026
Joined-case rows to request now: 1,802


Joined-case cache reuse:   0%|          | 0/55026 [00:00<?, ?it/s]

Joined-case API/cache update:   0%|          | 0/1802 [00:00<?, ?it/s]


Joined-case cooldown after 1,000 API requests: sleeping 60s
Compacted joined-case JSONL to 53,310 latest records -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_joined_case_api_cache.jsonl
Joined-case repair summary for this run:
http_200           1802
reused_success    55026
success            1802
dtype: int64
Cases with manual_joined_case_search populated: 8,269
Rows with manual_joined_case_collection populated: 56,828


In [27]:

# ---------------------------------------------------------------------
# 9) Save requested manifests
# ---------------------------------------------------------------------


# ---------------------------------------------------------------------
# v15: merge document download status into document metadata and roll it up by procedureId
# ---------------------------------------------------------------------

DOWNLOAD_SUCCESS_STATUSES = {"downloaded", "already_exists"}

def _truthy_download_success(s):
    return s.astype(str).str.lower().isin(["true", "1", "yes"])


def _download_path_exists_row(row):
    return path_exists_safe(
        row.get("final_local_path"),
        row.get("existing_local_path"),
        row.get("intended_local_path"),
    )


def attach_download_status_to_documents(df_docs: pd.DataFrame, df_downloads: pd.DataFrame) -> pd.DataFrame:
    docs = df_docs.copy()
    if docs.empty:
        return docs

    default_cols = {
        "download_target_flag": False,
        "download_success": False,
        "download_status": "not_targeted_for_download",
        "file_present_on_disk": False,
        "download_url": "",
        "working_url": "",
        "final_local_path": "",
        "intended_local_path": "",
        "existing_local_path": "",
        "attempted_urls": "[]",
        "attempt_count": 0,
    }
    for col, val in default_cols.items():
        if col not in docs.columns:
            docs[col] = val

    if df_downloads is None or df_downloads.empty:
        return docs

    dl = df_downloads.copy()
    if "file_present_on_disk" not in dl.columns:
        dl["file_present_on_disk"] = dl.apply(_download_path_exists_row, axis=1)
    else:
        dl["file_present_on_disk"] = dl["file_present_on_disk"].astype(str).str.lower().isin(["true", "1", "yes"]) | dl.apply(_download_path_exists_row, axis=1)

    if "download_success" not in dl.columns:
        dl["download_success"] = False
    dl["download_success"] = _truthy_download_success(dl["download_success"]) | dl["file_present_on_disk"] | dl.get("download_status", pd.Series("", index=dl.index)).astype(str).isin(DOWNLOAD_SUCCESS_STATUSES)

    key_cols = [c for c in ["logicDocId", "docId", "procedureId", "idProcedure", "affId"] if c in docs.columns and c in dl.columns]
    if not key_cols:
        print("Warning: could not attach download status to document metadata; no shared document key columns.")
        return docs

    dl_cols = key_cols + [c for c in [
        "download_success", "download_status", "file_present_on_disk", "download_url", "working_url",
        "final_local_path", "intended_local_path", "existing_local_path", "attempted_urls", "attempt_count",
    ] if c in dl.columns]
    dl_small = dl[dl_cols].drop_duplicates(subset=key_cols, keep="last")
    merged = docs.merge(dl_small, on=key_cols, how="left", suffixes=("", "__download"))

    for col, default_val in default_cols.items():
        dl_col = f"{col}__download"
        if dl_col in merged.columns:
            has = merged[dl_col].notna() & merged[dl_col].astype(str).ne("")
            merged.loc[has, col] = merged.loc[has, dl_col]
            merged = merged.drop(columns=[dl_col])

    merged["download_target_flag"] = merged["download_status"].fillna("not_targeted_for_download").astype(str).ne("not_targeted_for_download")
    merged["download_success"] = _truthy_download_success(merged["download_success"])
    merged["file_present_on_disk"] = merged["file_present_on_disk"].astype(str).str.lower().isin(["true", "1", "yes"]) | merged.apply(_download_path_exists_row, axis=1)
    return merged


def _join_ids(values, max_items=5000):
    return unique_join(values, sep="|", max_items=max_items)


def add_case_download_rollup_by_procedure(
    df_cases_manifest: pd.DataFrame,
    df_docs_with_downloads: pd.DataFrame,
) -> pd.DataFrame:
    """
    Roll document/download status up to the case manifest.

    procedureId is the canonical grouping and merge key. affId is used
    only as a fallback when procedureId is unavailable.
    """
    out = df_cases_manifest.copy()

    if out.empty or df_docs_with_downloads.empty:
        return out

    if (
        "procedureId" in out.columns
        and "procedureId" in df_docs_with_downloads.columns
        and out["procedureId"].fillna("").astype(str).str.strip().ne("").any()
        and df_docs_with_downloads["procedureId"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .any()
    ):
        group_col = "procedureId"

    elif (
        "affId" in out.columns
        and "affId" in df_docs_with_downloads.columns
    ):
        group_col = "affId"
        print(
            "Warning: procedureId unavailable for download roll-up; "
            "falling back to affId."
        )

    else:
        print(
            "Warning: could not roll up downloads; no shared "
            "procedureId or affId column."
        )
        return out

    docs = df_docs_with_downloads.copy()

    docs["relevance_flag"] = (
        bool_series(docs["relevance_flag"])
        if "relevance_flag" in docs.columns
        else False
    )

    docs["download_success"] = (
        _truthy_download_success(docs["download_success"])
        if "download_success" in docs.columns
        else False
    )

    docs["download_target_flag"] = (
        docs["download_target_flag"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
        if "download_target_flag" in docs.columns
        else False
    )

    docs["file_present_on_disk"] = (
        docs["file_present_on_disk"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
        if "file_present_on_disk" in docs.columns
        else False
    )

    docs["downloaded_or_present"] = (
        docs["download_success"]
        | docs["file_present_on_disk"]
    )

    document_id_cols = [
        c
        for c in ["logicDocId", "docId", "idPublished"]
        if c in docs.columns
    ]

    if document_id_cols:
        docs["document_id_for_status"] = (
            docs[document_id_cols]
            .astype(str)
            .replace({"nan": "", "<NA>": ""})
            .agg(lambda values: first_nonempty(*values), axis=1)
        )
    else:
        docs["document_id_for_status"] = ""

    rel_docs = docs[docs["relevance_flag"]].copy()
    rel_not_downloaded = rel_docs[
        ~rel_docs["downloaded_or_present"]
    ].copy()
    rel_downloaded = rel_docs[
        rel_docs["downloaded_or_present"]
    ].copy()

    agg = (
        docs
        .groupby(group_col, dropna=False)
        .agg(
            all_document_count=("document_id_for_status", "count"),
            all_download_target_document_count=(
                "download_target_flag",
                "sum",
            ),
            all_downloaded_or_present_document_count=(
                "downloaded_or_present",
                "sum",
            ),
            all_document_download_statuses=(
                "download_status",
                unique_join,
            ),
        )
        .reset_index()
    )

    rel_count_col = "relevant_document_count_by_procedure"

    rel_agg = (
        rel_docs
        .groupby(group_col, dropna=False)
        .agg(
            relevant_document_count_by_procedure=(
                "document_id_for_status",
                "count",
            ),
            relevant_download_target_document_count=(
                "download_target_flag",
                "sum",
            ),
            relevant_downloaded_or_present_document_count=(
                "downloaded_or_present",
                "sum",
            ),
            relevant_document_download_statuses=(
                "download_status",
                unique_join,
            ),
            relevant_document_ids=(
                "document_id_for_status",
                _join_ids,
            ),
        )
        .reset_index()
    )

    rel_down_ids = (
        rel_downloaded
        .groupby(group_col, dropna=False)["document_id_for_status"]
        .apply(_join_ids)
        .reset_index(name="relevant_downloaded_document_ids")
    )

    rel_not_down_ids = (
        rel_not_downloaded
        .groupby(group_col, dropna=False)["document_id_for_status"]
        .apply(_join_ids)
        .reset_index(name="relevant_not_downloaded_document_ids")
    )

    roll = agg.merge(rel_agg, on=group_col, how="left")
    roll = roll.merge(rel_down_ids, on=group_col, how="left")
    roll = roll.merge(rel_not_down_ids, on=group_col, how="left")

    numeric_cols = [
        rel_count_col,
        "relevant_download_target_document_count",
        "relevant_downloaded_or_present_document_count",
    ]

    for column in numeric_cols:
        if column not in roll.columns:
            roll[column] = 0

        roll[column] = (
            pd.to_numeric(roll[column], errors="coerce")
            .fillna(0)
            .astype(int)
        )

    text_cols = [
        "relevant_document_download_statuses",
        "relevant_document_ids",
        "relevant_downloaded_document_ids",
        "relevant_not_downloaded_document_ids",
    ]

    for column in text_cols:
        if column not in roll.columns:
            roll[column] = ""

        roll[column] = roll[column].fillna("")

    roll["case_has_any_document"] = (
        roll["all_document_count"].gt(0)
    )

    roll["case_has_relevant_document"] = (
        roll[rel_count_col].gt(0)
    )

    roll["case_has_downloaded_relevant_document"] = (
        roll["relevant_downloaded_or_present_document_count"].gt(0)
    )

    roll["case_all_relevant_documents_downloaded"] = (
        roll["case_has_relevant_document"]
        & roll["relevant_downloaded_or_present_document_count"].eq(
            roll[rel_count_col]
        )
    )

    roll["case_any_document_downloaded_or_present"] = (
        roll["all_downloaded_or_present_document_count"].gt(0)
    )

    roll["case_all_targeted_documents_downloaded"] = (
        roll["all_download_target_document_count"].gt(0)
        & roll["all_downloaded_or_present_document_count"].eq(
            roll["all_download_target_document_count"]
        )
    )

    replace_cols = [
        c
        for c in roll.columns
        if c != group_col
    ]

    out = out.drop(
        columns=[
            c
            for c in replace_cols
            if c in out.columns
        ],
        errors="ignore",
    )

    out = out.merge(
        roll,
        on=group_col,
        how="left",
    )

    out["case_relevance_flag"] = (
        out["case_has_relevant_document"]
        .fillna(False)
        .astype(bool)
    )

    out["relevant_document_count"] = (
        pd.to_numeric(
            out[rel_count_col],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
    )

    return out


# Attach download status before writing the final document/case manifests.
df_doc_meta = attach_download_status_to_documents(df_doc_meta, df_download_manifest)
df_case_metadata_manifest = add_case_download_rollup_by_procedure(df_case_metadata_manifest, df_doc_meta)

# Final requested outputs.
df_document_metadata_manifest = df_doc_meta.copy()

# Clean final exported manifests.
for _name in ["df_case_metadata_manifest", "df_document_metadata_manifest", "df_document_context_manifest", "df_download_manifest", "df_curia_failed_join"]:
    _df = globals().get(_name)
    if isinstance(_df, pd.DataFrame):
        _df = _df.drop(columns=[c for c in DROP_NON_ENGLISH_METADATA_COLUMNS if c in _df.columns], errors="ignore")
        if _name in {"df_document_metadata_manifest", "df_download_manifest"}:
            _df = _df.drop(columns=[c for c in ["document_dedup_key"] + DROP_HUGE_DOWNLOAD_COLUMNS if c in _df.columns], errors="ignore")
        globals()[_name] = _df

save_outputs(df_case_metadata_manifest, CASE_METADATA_STEM, CASE_METADATA_XLSX, CASE_METADATA_JSONL, CASE_METADATA_PARQUET)
save_outputs(df_document_metadata_manifest, DOCUMENT_METADATA_STEM, DOCUMENT_METADATA_XLSX, DOCUMENT_METADATA_JSONL, DOCUMENT_METADATA_PARQUET)
save_outputs(df_document_context_manifest, DOCUMENT_CONTEXT_STEM, DOCUMENT_CONTEXT_XLSX, DOCUMENT_CONTEXT_JSONL, DOCUMENT_CONTEXT_PARQUET)
save_outputs(df_download_manifest, DOWNLOAD_MANIFEST_STEM, DOWNLOAD_MANIFEST_XLSX, DOWNLOAD_MANIFEST_JSONL, DOWNLOAD_MANIFEST_PARQUET)
save_outputs(df_curia_failed_join, LEGACY_JOIN_FAILURES_STEM, LEGACY_JOIN_FAILURES_XLSX, LEGACY_JOIN_FAILURES_JSONL, LEGACY_JOIN_FAILURES_PARQUET)


/tmp/ipykernel_50701/4259691232.py:75: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[True True True ... True True True]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  merged.loc[has, col] = merged.loc[has, dl_col]
/tmp/ipykernel_50701/4259691232.py:75: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[True True True ... True True True]' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  merged.loc[has, col] = merged.loc[has, dl_col]
/tmp/ipykernel_50701/4259691232.py:345: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)

Saved JSONL 56,828 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_case_metadata_manifest.jsonl
Saved parquet 56,828 rows -> /home/edik/projects/eccjeu/output/court_infocuria/parquet/infocuria_case_metadata_manifest.parquet
Saved Excel 56,828 rows -> /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_case_metadata_manifest.xlsx
Saved JSONL 152,915 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_metadata_manifest.jsonl
Saved parquet 152,915 rows -> /home/edik/projects/eccjeu/output/court_infocuria/parquet/infocuria_document_metadata_manifest.parquet


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)

/home/edik/projects/.venv/lib/python3.12/site-packages/xlsxwriter/worksheet.py:1321: UserWarning: Ignoring URL 'https://infocuriaws.curia.europa.eu/blob/download-file-html/T/2021/T_0433_19_0000000RAL_01_I_01/303228-EN-1.html' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
/home/edik/projects/.venv/lib/python3.12/site-packages/xlsxwriter/worksheet.py:1321: UserWarning: Ignoring URL 'https://infocuriaws.curia.europa.eu/blob/download-file-html/T/2021/T_0437_19_0000000RAL_01_I_01/303231-EN-1.html' since it exceeds Excel's limit of 65,530 URLs per worksheet.
  warn(
/home/edik/projects/.venv/lib/python3.12/site-packages/xlsxwriter/worksheet.py:1321: User

Saved Excel 152,915 rows -> /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_document_metadata_manifest.xlsx
Saved JSONL 152,915 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_context_manifest.jsonl
Saved parquet 152,915 rows -> /home/edik/projects/eccjeu/output/court_infocuria/parquet/infocuria_document_context_manifest.parquet
Saved Excel 152,915 rows -> /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_document_context_manifest.xlsx
Saved JSONL 77,085 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_download_manifest.jsonl
Saved parquet 77,085 rows -> /home/edik/projects/eccjeu/output/court_infocuria/parquet/infocuria_document_download_manifest.parquet
Saved Excel 77,085 rows -> /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_document_download_manifest.xlsx
Saved JSONL 135 rows -> /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_legacy_join_failur

In [28]:

# ---------------------------------------------------------------------
# 10) Final validation checks
# ---------------------------------------------------------------------

print("==== INPUTS ====")
print("Case skeleton:", INFOCURIA_CASE_MANIFEST)
print("Raw procedural metadata JSONL:", RAW_METADATA_JSONL)
print("Raw JSONL size MB:", round(file_size_mb(RAW_METADATA_JSONL), 2))
print("Raw JSONL lines:", f"{count_jsonl_lines(RAW_METADATA_JSONL):,}")

print("\n==== CASE METADATA MANIFEST ====")
print("Rows:", f"{len(df_case_metadata_manifest):,}")
if "metadata_join_status" in df_case_metadata_manifest.columns:
    display(df_case_metadata_manifest["metadata_join_status"].fillna("missing").value_counts(dropna=False).to_frame("count"))
if "case_relevance_flag" in df_case_metadata_manifest.columns:
    print("Cases with at least one relevant document:", f"{df_case_metadata_manifest['case_relevance_flag'].astype(bool).sum():,}")
if "case_all_relevant_documents_downloaded" in df_case_metadata_manifest.columns:
    print("Relevant cases with all relevant documents downloaded/present:", f"{df_case_metadata_manifest['case_all_relevant_documents_downloaded'].fillna(False).astype(bool).sum():,}")
if "case_has_downloaded_relevant_document" in df_case_metadata_manifest.columns:
    print("Cases with at least one downloaded/present relevant document:", f"{df_case_metadata_manifest['case_has_downloaded_relevant_document'].fillna(False).astype(bool).sum():,}")

print("\n==== DOCUMENT METADATA MANIFEST ====")
print("Rows:", f"{len(df_document_metadata_manifest):,}")
if not df_document_metadata_manifest.empty and "normalized_docTypeCode" in df_document_metadata_manifest.columns:
    display(df_document_metadata_manifest["normalized_docTypeCode"].fillna("UNKNOWN").value_counts(dropna=False).to_frame("count").head(100))
if not df_document_metadata_manifest.empty and "relevance_category" in df_document_metadata_manifest.columns:
    display(df_document_metadata_manifest["relevance_category"].fillna("UNKNOWN").value_counts(dropna=False).to_frame("count"))
if not df_document_metadata_manifest.empty and "download_status" in df_document_metadata_manifest.columns:
    display(df_document_metadata_manifest["download_status"].fillna("missing").value_counts(dropna=False).to_frame("document_download_status_count"))

print("\n==== DOCUMENT CONTEXT MANIFEST ====")
print("Rows:", f"{len(df_document_context_manifest):,}")
if "context_text_en" in df_document_context_manifest.columns:
    print("Rows with English context:", f"{df_document_context_manifest['context_text_en'].fillna('').ne('').sum():,}")

print("\n==== DOWNLOAD MANIFEST ====")
print("Rows:", f"{len(df_download_manifest):,}")
if not df_download_manifest.empty:
    print("Rows with selected URL:", f"{df_download_manifest['download_url'].fillna('').ne('').sum():,}")
    if "download_status" in df_download_manifest.columns:
        display(df_download_manifest["download_status"].fillna("pending").value_counts(dropna=False).to_frame("count"))
    big_cols_present = [c for c in DROP_HUGE_DOWNLOAD_COLUMNS if c in df_download_manifest.columns]
    print("Huge candidate columns present:", big_cols_present)

print("\n==== LEGACY JOIN FAILURES ====")
print("Rows:", f"{len(df_curia_failed_join):,}")

print("\nFiles written:")
for p in [
    CASE_METADATA_XLSX, DOCUMENT_METADATA_XLSX, DOCUMENT_CONTEXT_XLSX, DOWNLOAD_MANIFEST_XLSX, LEGACY_JOIN_FAILURES_XLSX,
    CASE_METADATA_JSONL, DOCUMENT_METADATA_JSONL, DOCUMENT_CONTEXT_JSONL, DOWNLOAD_MANIFEST_JSONL, LEGACY_JOIN_FAILURES_JSONL,
    CASE_METADATA_PARQUET, DOCUMENT_METADATA_PARQUET, DOCUMENT_CONTEXT_PARQUET, DOWNLOAD_MANIFEST_PARQUET, LEGACY_JOIN_FAILURES_PARQUET,
]:
    print(p)


==== INPUTS ====
Case skeleton: /home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_case_scrape_manifest.xlsx
Raw procedural metadata JSONL: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_procedural_metadata_by_procedure_raw.jsonl
Raw JSONL size MB: 44807.45
Raw JSONL lines: 56,828

==== CASE METADATA MANIFEST ====
Rows: 56,828


,count
metadata_join_status,
metadata_joined,56828


Cases with at least one relevant document: 40,117
Relevant cases with all relevant documents downloaded/present: 40,117
Cases with at least one downloaded/present relevant document: 40,117

==== DOCUMENT METADATA MANIFEST ====
Rows: 152,915


/tmp/ipykernel_50701/1297412798.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  print("Relevant cases with all relevant documents downloaded/present:", f"{df_case_metadata_manifest['case_all_relevant_documents_downloaded'].fillna(False).astype(bool).sum():,}")
/tmp/ipykernel_50701/1297412798.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  print("Cases with at least one downloaded/present relevant document:", f"{df_case_metadata_manifest['case_has_downloaded_relevant_document'].fillna(False).astype(bool).sum():,}")


,count
normalized_docTypeCode,
ARR_COMM,19229
ARRET,19061
REQ_COMM,17123
ARRET_SOM,16792
ORD_NP,13219
CONCL,12100
DDP_COMM,9801
ARRET_NP,7291
ARRET_DR,6784


,count
relevance_category,
relevant,77085
irrelevant,75830


,document_download_status_count
download_status,
not_targeted_for_download,75830
already_exists,72849
downloaded,4236



==== DOCUMENT CONTEXT MANIFEST ====
Rows: 152,915
Rows with English context: 131,759

==== DOWNLOAD MANIFEST ====
Rows: 77,085
Rows with selected URL: 77,085


,count
download_status,
already_exists,72849
downloaded,4236


Huge candidate columns present: []

==== LEGACY JOIN FAILURES ====
Rows: 135

Files written:
/home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_case_metadata_manifest.xlsx
/home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_document_metadata_manifest.xlsx
/home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_document_context_manifest.xlsx
/home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_document_download_manifest.xlsx
/home/edik/projects/eccjeu/output/court_infocuria/excel/infocuria_legacy_join_failures.xlsx
/home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_case_metadata_manifest.jsonl
/home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_metadata_manifest.jsonl
/home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_context_manifest.jsonl
/home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_download_manifest.jsonl
/home/edik/projects/eccjeu/output/cou

In [29]:

# ---------------------------------------------------------------------
# 10) Optional download-manifest size / existence QA
# ---------------------------------------------------------------------

manifest_path = DOWNLOAD_MANIFEST_JSONL
print("Download manifest path:", manifest_path)

df = read_manifest(manifest_path)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

mem = df.memory_usage(deep=True).sort_values(ascending=False)
print("\nLargest columns by memory:")
print((mem / 1024**2).head(20).round(2).astype(str) + " MB")

text_cols = df.select_dtypes(include="object").columns
if len(text_cols):
    lengths = (
        df[text_cols]
        .fillna("")
        .astype(str)
        .apply(lambda s: s.str.len().sum())
        .sort_values(ascending=False)
    )
    print("\nLargest text columns by total characters:")
    print((lengths / 1024**2).head(20).round(2).astype(str) + " MB chars")

path_cols = ["final_local_path", "existing_local_path", "intended_local_path"]
available_path_cols = [c for c in path_cols if c in df.columns]

def first_existing_path(row):
    for c in available_path_cols:
        val = row.get(c)
        if pd.notna(val) and str(val).strip():
            return str(val)
    return ""

resolved_path = df.apply(first_existing_path, axis=1) if available_path_cols else pd.Series("", index=df.index)
exists = resolved_path.apply(lambda p: Path(p).exists() if p else False)

print("\nDownload status:")
if "download_status" in df.columns:
    print(df["download_status"].value_counts(dropna=False))

print("\nResolved file existence:")
print(f"Files present on disk: {exists.sum():,} / {len(df):,}")
print(f"Missing files: {(~exists).sum():,}")

print("\nHuge candidate columns present:", [c for c in DROP_HUGE_DOWNLOAD_COLUMNS if c in df.columns])


Download manifest path: /home/edik/projects/eccjeu/output/court_infocuria/jsonl/infocuria_document_download_manifest.jsonl
Rows: 77,085
Columns: 42

Largest columns by memory:
download_url           11.73 MB
working_url            11.72 MB
available_languages     9.37 MB
intended_local_path     8.93 MB
final_local_path        8.93 MB
existing_local_path      8.6 MB
candidate_years         7.56 MB
procedureId             5.66 MB
idProcedure             5.66 MB
affId                   5.29 MB
docType                 5.23 MB
attempted_urls          5.09 MB
target_filename         4.74 MB
download_status         4.62 MB
internal_key            4.58 MB
procedure_key           4.58 MB
procedure_key_base      4.48 MB
docDate                 4.34 MB
logicDocId              4.24 MB
docFormats              4.22 MB
dtype: object

Largest text columns by total characters:
download_url           8.12 MB chars
working_url            8.12 MB chars
available_languages    5.76 MB chars
intended_local_p